
# 🧬 AlphaFold Fusion v2.3

**Integrated Protein Structure Prediction & Analysis Platform**

---
  <div style="display:flex;justify-content:center;gap:12px;flex-wrap:wrap">
    <span style="background:rgba(255,255,255,.2);padding:8px 16px;
    border-radius:20px;font-size:.9rem">⚡ ColabFold</span>
    <span style="background:rgba(255,255,255,.2);padding:8px 16px;
    border-radius:20px;font-size:.9rem">🌊 Disorder</span>
    <span style="background:rgba(255,255,255,.2);padding:8px 16px;
    border-radius:20px;font-size:.9rem">🧬 Conservation</span>
    <span style="background:rgba(255,255,255,.2);padding:8px 16px;
    border-radius:20px;font-size:.9rem">🔗 Interface</span>
    <span style="background:rgba(255,255,255,.2);padding:8px 16px;
    border-radius:20px;font-size:.9rem">✓ DisProt AUC=0.80</span>
  </div>
</div>


## 🚀 Quick Start (for reviewers)

1. **Runtime → Change runtime type → GPU (T4) → Save**
2. **Runtime → Run all** (installation ~10 min on first run)
3. Wait for the **PUBLIC (Cloudflare) link** in the last cell
4. Click it → the interface opens in your browser

> ⚠️ If the page doesn't load immediately, wait ~20s and refresh
> (the tunnel is initialising). Keep the last cell running while testing.

---

## 🧪 Example sequences to test

| Type | Input | Time |
|------|-------|------|
| **Monomer** | Ubiquitin (paste sequence below) | ~5 min |
| **Multimer** | Barnase + Barstar | ~15 min |
| **Load from AFDB** | Accession `P00698` (lysozyme);  `Q9UKV8` (Argonaute 2) | instant |

**Ubiquitin (monomer test):MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG

## 📊 What this platform does

- **Predicts** structures (ColabFold, AlphaFold2)
- **Analyses**: pLDDT confidence, PAE, intrinsic disorder,
  evolutionary conservation, inter-chain contacts, ensemble RMSF
- **Annotates**: UniProt/InterPro domains
- **Validates**: disorder against DisProt (experimental ground truth)
- **Exports**: FAIR-compliant JSON/TSV reports

⚠️ **Data privacy**: sequences are sent to third-party services
(ColabFold MMseqs2, EBI). Do not submit confidential sequences.

## 📁 1. Setup & Package Build

In [ ]:
import os
os.makedirs("alphafold_fusion/pages", exist_ok=True)
os.makedirs("examples", exist_ok=True)
os.makedirs("tests", exist_ok=True)
print("✅ Directory tree successfully created")

In [ ]:
%%writefile alphafold_fusion/__init__.py
"""AlphaFold Fusion — Integrated Protein Structure Analysis Platform."""
__version__ = "2.3.0"
__author__ = "AlphaFold Fusion Contributors"

In [ ]:
%%writefile alphafold_fusion/config.py
"""Global configuration: filesystem paths, colour schemes, API endpoints, CSS."""

from __future__ import annotations
import logging
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("alphafold_fusion")

_BASE = Path("/content") if Path("/content").exists() else Path.cwd()
RESULTS_DIR: Path = _BASE / "alphafold_results"
CACHE_DIR: Path = _BASE / "alphafold_cache"

PLDDT_SCHEME: dict[str, dict] = {
    "Very high (>90)": {"min": 90, "max": 100, "color": "#2166F3"},
    "High (70-90)":    {"min": 70, "max": 90,  "color": "#8FD3FF"},
    "Low (50-70)":     {"min": 50, "max": 70,  "color": "#FFD35A"},
    "Very low (<50)":  {"min": 0,  "max": 50,  "color": "#FF8C42"},
}

C_VHIGH = "#2166F3"
C_HIGH  = "#8FD3FF"
C_LOW   = "#FFD35A"
C_VLOW  = "#FF8C42"
C_SPEC_LT70 = "#FF7F0E"
C_SPEC_GE70 = "#4CA6FF"
C_SPEC_GE90 = "#1F57F7"

DOMAIN_COLORS: list[str] = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
]

UNIPROT_API  = "https://rest.uniprot.org/uniprotkb/{acc}.json"
INTERPRO_APIS: list[str] = [
    "https://www.ebi.ac.uk/interpro/api/entry/interpro/protein/uniprot/{acc}?page_size=200",
    "https://www.ebi.ac.uk/interpro/api/entry/all/protein/uniprot/{acc}?page_size=200",
    "https://www.ebi.ac.uk/interpro/api/protein/uniprot/{acc}?page_size=200",
]
AFDB_API    = "https://alphafold.ebi.ac.uk/api/prediction/{acc}"
PY3DMOL_CDN = "https://cdn.jsdelivr.net/npm/3dmol@2.0.4/build/3Dmol-min.js"

AA3: frozenset[str] = frozenset({
    "ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS",
    "ILE", "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP",
    "TYR", "VAL",
})

MSA_MODES: dict[str, str] = {
    "full": "mmseqs2_uniref_env", "fast": "mmseqs2_uniref",
    "minimal": "single_sequence",
}
MSA_LABELS: dict[str, str] = {
    "Complete (UniRef+Environ.)": "full",
    "Fast (UniRef)": "fast",
    "Minimal (single sequence)": "minimal",
}
RECYCLE_PRESETS: dict[str, dict[str, int]] = {
    "Fast":           {"monomer": 3,  "complex": 6},
    "Classic":        {"monomer": 6,  "complex": 12},
    "High precision": {"monomer": 12, "complex": 20},
}

CSS = """<style>
.main-header{font-size:3rem;background:linear-gradient(135deg,#4361ee,#3a0ca3);
-webkit-background-clip:text;-webkit-text-fill-color:transparent;
text-align:center;margin-bottom:1rem;font-weight:800}
.sub-header{font-size:1.6rem;color:#4361ee;margin:1rem 0 .6rem;
border-bottom:3px solid #4cc9f0}
.card{padding:1rem;border-radius:1rem;background:var(--aff-card-bg,#fff);
color:var(--aff-card-fg,#212529);
box-shadow:0 10px 25px rgba(0,0,0,.08);margin:1rem 0}
.info-card{background:var(--aff-info-bg,#f0f2f6);
color:var(--aff-card-fg,#212529);padding:1rem;
border-radius:1rem;border-left:5px solid #4361ee;margin:1rem 0}
.stButton>button{background:linear-gradient(135deg,#4361ee,#3a0ca3);
color:#fff;border:none;padding:.6rem 1.2rem;border-radius:50px;
font-weight:600;width:100%}

/* Dark mode support */
@media (prefers-color-scheme: dark){
  :root{
    --aff-card-bg:#1e1e2e;
    --aff-card-fg:#e0e0e0;
    --aff-info-bg:#252535;
  }
  .sub-header{color:#8fb3ff;border-bottom-color:#4cc9f0}
  .card{box-shadow:0 10px 25px rgba(0,0,0,.4)}
}
</style>"""
# EBI services require a real contact email (usage policy).
# Change this to YOUR email before deployment.
EBI_CONTACT_EMAIL = "your-email@example.com"

In [ ]:
%%writefile alphafold_fusion/sequence.py
"""Sequence parsing, cleaning, and filesystem helpers."""

from __future__ import annotations
import hashlib, re, secrets, time
from pathlib import Path
from typing import Optional


def clean_sequence(seq: str) -> str:
    seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY:]", "", (seq or "").upper())
    return re.sub(r":+", ":", seq).strip(":")


def is_complex(seq: str) -> bool:
    return ":" in (seq or "")


def total_length(seq: str) -> int:
    return sum(len(p) for p in clean_sequence(seq).split(":") if p)


def safe_basename(name: str, seq: str = "", maxlen: int = 80) -> str:
    base = re.sub(r"[^A-Za-z0-9._-]+", "", name or "seq").strip("._-") or "seq"
    h = hashlib.sha1(f"{name}|{seq}".encode()).hexdigest()[:10]
    return f"{base[:max(8, maxlen - len(h) - 1)]}_{h}"


def parse_fasta(text: str) -> list[tuple[str, str]]:
    entries: list[tuple[str, str]] = []
    head: Optional[str] = None
    buf: list[str] = []
    for ln in (text or "").splitlines():
        ln = ln.strip()
        if not ln:
            if head and buf:
                entries.append((head, "".join(buf)))
                head = None; buf = []
            continue
        if ln.startswith(">"):
            if head and buf:
                entries.append((head, "".join(buf)))
            head = ln[1:].strip(); buf = []
        else:
            if head is not None:
                buf.append(ln)
            else:
                head = ln
    if head and buf:
        entries.append((head, "".join(buf)))
    return entries


def new_run_dir(results_dir: Path) -> Path:
    rid = time.strftime("%Y%m%d-%H%M%S") + "-" + secrets.token_hex(3)
    root = results_dir / f"run_{rid}"
    root.mkdir(parents=True, exist_ok=True)
    return root

In [ ]:
%%writefile alphafold_fusion/api.py
"""Remote API clients for UniProt, InterPro, and AlphaFold DB."""

from __future__ import annotations
import json, re, time, urllib.error, urllib.request
from typing import Any, Optional
import numpy as np, pandas as pd, streamlit as st
from .config import AFDB_API, INTERPRO_APIS, UNIPROT_API, log, EBI_CONTACT_EMAIL

# UniProt accession format (official pattern):
# [OPQ][0-9][A-Z0-9]{3}[0-9] | [A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}
_ACC_RE = re.compile(
    r"^(?:[OPQ][0-9][A-Z0-9]{3}[0-9]"
    r"|[A-NR-Z][0-9](?:[A-Z][A-Z0-9]{2}[0-9]){1,2})"
    r"(?:-\d+)?$"
)


def is_uniprot_acc(s: str) -> bool:
    return bool(s and _ACC_RE.match(s.strip().upper()))


def extract_uniprot_acc(s: str) -> Optional[str]:
    if not s:
        return None
    for tok in re.split(r"[|\s,;/]+", str(s).strip()):
        if is_uniprot_acc(tok.upper()):
            return tok.upper()
    m = re.search(r"UniRef\d+_([A-Z0-9]{6,10}(?:-\d+)?)", s, re.I)
    if m and is_uniprot_acc(m.group(1)):
        return m.group(1).upper()
    return None


def guess_acc(seq_name: str,
              afdb_used: Optional[dict[str, str]] = None) -> Optional[str]:
    if afdb_used:
        acc = afdb_used.get(seq_name)
        if acc:
            return acc
    if is_uniprot_acc(seq_name):
        return seq_name
    m = re.search(r"\b([A-Z0-9]{6,10})\b", seq_name, re.I)
    if m and is_uniprot_acc(m.group(1)):
        return m.group(1)
    return None


# ══════════════════════════════════════════════════════════════
# Generic JSON fetcher (for UniProt, AFDB)
# ══════════════════════════════════════════════════════════════
def _fetch_json(url: str, timeout: int = 15,
                retries: int = 3) -> Optional[dict]:
    for attempt in range(max(1, retries)):
        try:
            req = urllib.request.Request(
                url, headers={"Accept": "application/json"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return json.loads(r.read().decode("utf-8", "ignore"))
        except urllib.error.HTTPError as exc:
            if exc.code == 404:
                return None
            if exc.code == 429 or exc.code >= 500:
                time.sleep(1.0 * (attempt + 1))
                continue
            return None
        except Exception:
            time.sleep(0.5 * (attempt + 1))
    return None


# ══════════════════════════════════════════════════════════════
# UniProt
# ══════════════════════════════════════════════════════════════
@st.cache_data(show_spinner=False, ttl=86400)
def fetch_uniprot(acc: str) -> Optional[dict]:
    acc = (acc or "").strip()
    if not acc:
        return None
    return _fetch_json(UNIPROT_API.format(acc=acc), timeout=20)


def uniprot_domains(j: Optional[dict]) -> list[dict]:
    keep = {"Domain", "Repeat", "Region", "Coiled coil", "Zinc finger"}
    segs: list[dict] = []
    for f in (j or {}).get("features", []):
        if f.get("type") not in keep:
            continue
        loc = f.get("location") or {}
        try:
            b = int((loc.get("start") or {}).get("value"))
            e = int((loc.get("end") or {}).get("value"))
        except (TypeError, ValueError):
            continue
        segs.append({
            "start": b, "end": e,
            "label": f.get("description") or f.get("type", "Domain"),
            "type": f.get("type"),
        })
    return sorted(segs, key=lambda s: (s["start"], s["end"]))


# ══════════════════════════════════════════════════════════════
# InterPro — COPIÉ DU CODE FONCTIONNEL (all-in-one)
# ══════════════════════════════════════════════════════════════
@st.cache_data(show_spinner=False, ttl=86400)
def fetch_interpro(acc: str, timeout: int = 20,
                   retries: int = 3, backoff: float = 1.0) -> Optional[dict]:
    """Fetch InterPro entries for a UniProt accession.

    Uses the EXACT same logic as the working all-in-one version:
    - Try each endpoint in order
    - Per endpoint: retry up to `retries` times
    - On 404: break inner loop, try next endpoint
    - On 408/5xx: sleep and retry same endpoint
    - On success with results: return immediately
    - On success with empty results: try next endpoint
    """
    try_acc = (acc or "").strip()
    if not try_acc:
        return None
    base_acc = try_acc.split("-")[0] if "-" in try_acc else try_acc

    endpoints = [
        f"https://www.ebi.ac.uk/interpro/api/entry/interpro/protein/uniprot/{base_acc}?page_size=200",
        f"https://www.ebi.ac.uk/interpro/api/entry/all/protein/uniprot/{base_acc}?page_size=200",
        f"https://www.ebi.ac.uk/interpro/api/protein/uniprot/{base_acc}?page_size=200",
    ]
    headers = {"Accept": "application/json"}

    for url in endpoints:
        for attempt in range(max(1, retries)):
            try:
                req = urllib.request.Request(url, headers=headers)
                with urllib.request.urlopen(req, timeout=timeout) as r:
                    raw = r.read().decode("utf-8", "ignore")
                    j = json.loads(raw)

                # Check for results
                results = j.get("results") or []
                if results:
                    log.info("fetch_interpro: %d results from %s",
                             len(results), url[:80])
                    return {"results": results, "_source_url": url}
                if isinstance(j, list) and j:
                    log.info("fetch_interpro: %d results (list) from %s",
                             len(j), url[:80])
                    return {"results": j, "_source_url": url}

                # Empty results from this endpoint → try next endpoint
                break

            except urllib.error.HTTPError as e:
                if e.code == 404:
                    # Not found on this endpoint → try next
                    break
                elif e.code == 408 or e.code == 429 or e.code >= 500:
                    # Timeout / rate-limit / server error → retry
                    log.warning("fetch_interpro: HTTP %d, retry %d/%d",
                                e.code, attempt + 1, retries)
                    time.sleep(backoff * (attempt + 1))
                    continue
                else:
                    # Other HTTP error → try next endpoint
                    break
            except Exception as exc:
                log.warning("fetch_interpro: %s, retry %d/%d",
                            exc, attempt + 1, retries)
                time.sleep(backoff * (attempt + 1))
                continue

    log.warning("fetch_interpro: all endpoints failed for %s", base_acc)
    return None


def interpro_domains(j: Optional[dict]) -> list[dict]:
    """Extract domain segments from InterPro API response.

    Uses the EXACT same logic as the working all-in-one version:
    - Check metadata, then entry, then root-level fields
    - Also check short_name, acc, type_name (not just name/accession/type)
    - Extract from: proteins[].entry_protein_locations,
                     root locations, root entry_protein_locations
    - Deduplicate by (start, end, label)
    """
    segs: list[dict] = []
    if not j:
        return segs

    results = j.get("results") or []
    if not results and isinstance(j, list):
        results = j

    keep_types = {
        "domain", "repeat", "homologous_superfamily",
        "homologous superfamily", "family", "conserved_site",
        "active_site", "binding_site", "ptm", "coiled_coil",
        "coiled-coil",
    }

    for r in results:
        # ── Extract metadata (same priority as working code) ──
        metadata = r.get("metadata") or {}
        entry = r.get("entry") or {}

        if metadata:
            acc_id = metadata.get("accession") or ""
            name = (metadata.get("name")
                    or metadata.get("short_name") or "")
            etype = (metadata.get("type") or "").strip().lower()
        elif entry:
            acc_id = (entry.get("accession")
                      or entry.get("acc") or "")
            name = (entry.get("name")
                    or entry.get("short_name") or "")
            etype = (entry.get("type")
                     or entry.get("type_name") or "").strip().lower()
        else:
            acc_id = (r.get("accession")
                      or r.get("acc") or "")
            name = (r.get("name")
                    or r.get("short_name") or "")
            etype = (r.get("type") or "").strip().lower()

        # ── Type filter ──
        if etype and etype not in keep_types:
            continue

        label = (f"{acc_id} {name}".strip()
                 if (acc_id or name) else "InterPro entry")

        # ── Source 1: proteins[].entry_protein_locations ──
        proteins = r.get("proteins") or []
        for prot in proteins:
            locations = (prot.get("entry_protein_locations")
                         or prot.get("locations") or [])
            for loc in locations:
                fragments = loc.get("fragments") or []
                for fr in fragments:
                    try:
                        beg = int(fr.get("start"))
                        end = int(fr.get("end"))
                        if beg <= end:
                            segs.append({
                                "start": beg, "end": end,
                                "label": label,
                                "type": (etype.title()
                                         if etype else "Domain"),
                            })
                    except (TypeError, ValueError):
                        continue

        # ── Source 2: root-level locations ──
        locations_direct = r.get("locations") or []
        for loc in locations_direct:
            fragments = loc.get("fragments") or []
            for fr in fragments:
                try:
                    beg = int(fr.get("start"))
                    end = int(fr.get("end"))
                    if beg <= end:
                        segs.append({
                            "start": beg, "end": end,
                            "label": label,
                            "type": (etype.title()
                                     if etype else "Domain"),
                        })
                except (TypeError, ValueError):
                    continue

        # ── Source 3: root-level entry_protein_locations ──
        epl = r.get("entry_protein_locations") or []
        for loc in epl:
            fragments = loc.get("fragments") or []
            for fr in fragments:
                try:
                    beg = int(fr.get("start"))
                    end = int(fr.get("end"))
                    if beg <= end:
                        segs.append({
                            "start": beg, "end": end,
                            "label": label,
                            "type": (etype.title()
                                     if etype else "Domain"),
                        })
                except (TypeError, ValueError):
                    continue

    # ── Deduplicate ──
    uniq: list[dict] = []
    seen: set[tuple] = set()
    for s in segs:
        key = (s["start"], s["end"], s["label"])
        if key in seen:
            continue
        seen.add(key)
        uniq.append(s)

    log.info("interpro_domains: %d segments (deduped) from %d results",
             len(uniq), len(results))
    return sorted(uniq, key=lambda x: (x["start"], x["end"]))


# ══════════════════════════════════════════════════════════════
# AlphaFold DB
# ══════════════════════════════════════════════════════════════
def fetch_afdb(name: str, timeout: int = 10) -> Optional[dict]:
    if not name:
        return None
    cands = [t for t in re.split(r"[|\s,;/]+", str(name))
             if is_uniprot_acc(t)]
    if not cands:
        m = re.search(r"\b([A-Z0-9]{6,10}(?:-\d+)?)\b", name)
        if m:
            cands.append(m.group(1))
    for acc in cands:
        try_list = [acc]
        if "-" in acc:
            try_list.append(acc.split("-")[0])
        for a in try_list:
            j = _fetch_json(AFDB_API.format(acc=a), timeout=timeout)
            if isinstance(j, list) and j:
                o = j[0]
                return {
                    "acc": o.get("uniprotAccession", a),
                    "pdb_url": o.get("pdbUrl"),
                    "cif_url": o.get("cifUrl"),
                    "bcif_url": o.get("bcifUrl"),
                }
    return None


def enrich_df_urls(df: pd.DataFrame,
                   col: str = "Accession") -> pd.DataFrame:
    if df is None or df.empty or col not in df.columns:
        return df
    out = df.copy()
    u_urls: list[Optional[str]] = []
    a_urls: list[Optional[str]] = []
    for v in out[col].astype(str):
        acc = extract_uniprot_acc(v)
        if acc:
            u_urls.append(f"https://www.uniprot.org/uniprotkb/{acc}")
            a_urls.append(
                f"https://alphafold.ebi.ac.uk/entry/{acc.split('-')[0]}")
        else:
            u_urls.append(None)
            a_urls.append(None)
    out["UniProt_URL"] = u_urls
    out["AFDB_URL"] = a_urls
    return out
def blast_identify(seq: str, timeout: int = 1800) -> Optional[dict]:
    """Identify a protein by BLAST against UniProtKB (EBI service).

    Returns the top hit with accession, identity %, description.
    Reference: Madeira et al. (2022) NAR 50:W276-W279.
    """
    import urllib.parse
    seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", (seq or "").upper())
    if len(seq) < 10:
        return None
    base = "https://www.ebi.ac.uk/Tools/services/rest/ncbiblast"
    try:
        params = {
            "email": EBI_CONTACT_EMAIL,
            "program": "blastp",
            "stype": "protein",
            "database": "uniprotkb",
            "sequence": seq,
        }
        data = urllib.parse.urlencode(params).encode()
        req = urllib.request.Request(f"{base}/run", data=data)
        with urllib.request.urlopen(req, timeout=30) as r:
            job_id = r.read().decode().strip()
        log.info("blast_identify: job %s", job_id)

        # Poll (up to `timeout` seconds, 5 s intervals)
        finished = False
        for _ in range(int(timeout / 5)):
            time.sleep(5)
            sreq = urllib.request.Request(f"{base}/status/{job_id}")
            with urllib.request.urlopen(sreq, timeout=15) as r:
                status = r.read().decode().strip()
            if status == "FINISHED":
                finished = True
                break
            if status in ("ERROR", "FAILURE", "NOT_FOUND"):
                log.warning("blast_identify: %s", status)
                return None
        if not finished:
            log.warning("blast_identify: timeout after %ds", timeout)
            return None

        rreq = urllib.request.Request(f"{base}/result/{job_id}/json")
        with urllib.request.urlopen(rreq, timeout=30) as r:
            j = json.loads(r.read().decode("utf-8", "ignore"))

        hits = j.get("hits", [])
        if not hits:
            return None
        top = hits[0]
        hsp = (top.get("hit_hsps") or [{}])[0]
        acc = top.get("hit_acc", "") or top.get("hit_id", "")
        ident = float(hsp.get("hsp_identity", 0))
        cov = None
        try:
            cov = round(100.0 * int(hsp.get("hsp_align_len", 0))
                        / max(1, len(seq)), 1)
        except Exception:
            pass
        return {
            "accession": acc,
            "description": top.get("hit_desc", ""),
            "gene": top.get("hit_uni_gn", ""),
            "organism": top.get("hit_uni_os", ""),
            "identity_pct": round(ident, 1),
            "coverage_pct": cov,
            "length": top.get("hit_len"),
            "is_exact": ident >= 99.0,
        }
    except Exception as exc:
        log.warning("blast_identify: %s", exc)
        return None
def blast_hits(seq: str, timeout: int = 1800,
               max_hits: int = 20) -> Optional[list[dict]]:
    """BLAST against UniProtKB, return top hits with organism info.

    Lets the user pick the correct species/entry rather than forcing
    the single top hit (which may be an ortholog).
    Reference: Madeira et al. (2022) NAR 50:W276.
    """
    import urllib.parse
    seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", (seq or "").upper())
    if len(seq) < 10:
        return None
    base = "https://www.ebi.ac.uk/Tools/services/rest/ncbiblast"
    try:
        params = {
            "email": EBI_CONTACT_EMAIL, "program": "blastp",
            "stype": "protein", "database": "uniprotkb",
            "sequence": seq,
        }
        data = urllib.parse.urlencode(params).encode()
        req = urllib.request.Request(f"{base}/run", data=data)
        with urllib.request.urlopen(req, timeout=30) as r:
            job_id = r.read().decode().strip()

        finished = False
        for _ in range(int(timeout / 5)):
            time.sleep(5)
            sreq = urllib.request.Request(f"{base}/status/{job_id}")
            with urllib.request.urlopen(sreq, timeout=15) as r:
                status = r.read().decode().strip()
            if status == "FINISHED":
                finished = True
                break
            if status in ("ERROR", "FAILURE", "NOT_FOUND"):
                return None
        if not finished:
            return None

        rreq = urllib.request.Request(f"{base}/result/{job_id}/json")
        with urllib.request.urlopen(rreq, timeout=30) as r:
            j = json.loads(r.read().decode("utf-8", "ignore"))

        out = []
        for top in j.get("hits", [])[:max_hits]:
            hsp = (top.get("hit_hsps") or [{}])[0]
            raw_acc = top.get("hit_acc", "") or top.get("hit_id", "")
            acc = extract_uniprot_acc(raw_acc) or raw_acc
            ident = float(hsp.get("hsp_identity", 0))
            out.append({
                "accession": acc,
                "gene": top.get("hit_uni_gn", ""),
                "organism": top.get("hit_uni_os", ""),
                "description": top.get("hit_uni_de", ""),
                "identity_pct": round(ident, 1),
                "length": top.get("hit_len"),
            })
        return out if out else None
    except Exception as exc:
        log.warning("blast_hits: %s", exc)
        return None

In [ ]:
import urllib.request, urllib.parse, time

seq = "MEEKKKKKQEEKKKKEG"  # court, juste pour tester la soumission
base = "https://www.ebi.ac.uk/Tools/services/rest/ncbiblast"
params = {
    "email": "test@example.org", "program": "blastp",
    "stype": "protein", "database": "uniprotkb", "sequence": seq,
}
data = urllib.parse.urlencode(params).encode()
try:
    req = urllib.request.Request(f"{base}/run", data=data)
    with urllib.request.urlopen(req, timeout=30) as r:
        job = r.read().decode().strip()
    print("✅ BLAST job submitted:", job)
except Exception as e:
    print("❌ Error:", e)

In [ ]:
%%writefile alphafold_fusion/structure.py
"""Structure file I/O: CIF/PDB conversion, chain detection, format helpers."""

from __future__ import annotations
import os, re, tempfile
from pathlib import Path
from typing import Optional
import numpy as np, gemmi
from Bio.PDB import PDBIO
from Bio.PDB.MMCIFParser import MMCIFParser
from .config import AA3


def is_amino_acid(res) -> bool:
    try:
        return (res.name or "").strip().upper() in AA3
    except Exception:
        return False


def cif_to_pdb(cif_path: Path, pdb_out: Path) -> bool:
    try:
        pdb_out.write_text(
            gemmi.read_structure(str(cif_path)).make_minimal_pdb())
        return pdb_out.stat().st_size > 0
    except Exception:
        pass
    try:
        io = PDBIO()
        io.set_structure(
            MMCIFParser(QUIET=True).get_structure("S", str(cif_path)))
        io.save(str(pdb_out))
        return pdb_out.stat().st_size > 0
    except Exception:
        return False


def polymer_chains(path) -> list[str]:
    try:
        st_obj = gemmi.read_structure(str(path))
        ch = [(c.name, sum(1 for r in c if is_amino_acid(r)))
              for c in st_obj[0]]
        return [n for n, cnt in sorted(ch, key=lambda x: -x[1]) if cnt > 0]
    except Exception:
        return []


def avg_plddt(path, fmt: str) -> Optional[float]:
    """Mean pLDDT over CA atoms only (one value per residue).

    pLDDT is a per-residue confidence score stored in the B-factor
    column. Averaging over ALL atoms would bias the mean toward large
    side-chain residues and diverge from the CA-based per-residue
    profile used elsewhere (plddt_by_chain). We therefore restrict to
    CA atoms so that avg_plddt equals the mean of the residue-level
    profile.
    """
    vals: list[float] = []
    if fmt == "pdb":
        try:
            with open(path) as f:
                for ln in f:
                    if not ln.startswith("ATOM") or len(ln) < 66:
                        continue
                    # CA atoms only (name field is columns 13-16)
                    if ln[12:16].strip() != "CA":
                        continue
                    try:
                        b = float(ln[60:66])
                        if 0 <= b <= 100:
                            vals.append(b)
                    except ValueError:
                        pass
        except Exception:
            pass
    else:
        try:
            st_obj = gemmi.read_structure(str(path))
            if len(st_obj) > 0:
                for ch in st_obj[0]:
                    for res in ch:
                        if not is_amino_acid(res):
                            continue
                        for at in res:
                            if at.name == "CA":
                                b = float(at.b_iso)
                                if 0 <= b <= 100:
                                    vals.append(b)
                                break
        except Exception:
            pass
    return float(np.mean(vals)) if vals else None


def detect_fmt(txt: str) -> str:
    h = (txt or "")[:2000]
    if "atom_site." in h or h.lstrip().startswith("data"):
        return "cif"
    return "pdb"


def first_model_only(fmt: str, txt: str) -> str:
    if fmt == "pdb" and re.search(r"^MODEL", txt, re.M):
        m = re.search(
            r"^MODEL[^\n]*\n(.*?)(?:\nENDMDL|\Z)", txt, re.S | re.M)
        if m:
            return "\n".join(
                ln for ln in m.group(1).splitlines()
                if ln.startswith(("ATOM", "HETATM", "TER"))
            ) + "\n"
    return txt

In [ ]:
%%writefile alphafold_fusion/plddt.py
"""pLDDT confidence score extraction, binning, and Plotly visualisation."""

from __future__ import annotations
import os, tempfile
from collections import OrderedDict, defaultdict
from pathlib import Path
from typing import Optional
import gemmi, numpy as np
import plotly.express as px, plotly.graph_objects as go
import streamlit as st
from .config import C_HIGH, C_LOW, C_VHIGH, C_VLOW, PLDDT_SCHEME
from .structure import detect_fmt, is_amino_acid


def plddt_by_chain(txt: str, fmt: str = "pdb",
                   chain: Optional[str] = None) -> dict[str, list[float]]:
    fmt = fmt or detect_fmt(txt)
    result: dict[str, list[float]] = {}
    if fmt == "pdb":
        maps: dict[str, OrderedDict] = defaultdict(OrderedDict)
        for ln in (txt or "").splitlines():
            if not ln.startswith("ATOM") or len(ln) < 66:
                continue
            if ln[12:16].strip() != "CA":
                continue
            ch = ln[21:22].strip() or "A"
            if chain and ch != chain:
                continue
            key = (ln[22:26].strip(), ln[26:27])
            try:
                b = float(ln[60:66])
                if 0 <= b <= 100 and key not in maps[ch]:
                    maps[ch][key] = b
            except ValueError:
                pass
        for ch, od in maps.items():
            if od:
                result[ch] = list(od.values())
        return result
    tmp_path = None
    try:
        tmp = tempfile.NamedTemporaryFile("w", suffix=".cif", delete=False)
        tmp.write(txt or ""); tmp.flush(); tmp.close()
        tmp_path = tmp.name
        st_obj = gemmi.read_structure(tmp_path)
        if len(st_obj) > 0:
            for ch_obj in st_obj[0]:
                if not any(is_amino_acid(r) for r in ch_obj):
                    continue
                if chain and ch_obj.name != chain:
                    continue
                vals: list[float] = []
                for res in ch_obj:
                    if not is_amino_acid(res):
                        continue
                    ca_b = None
                    for at in res:
                        if at.name == "CA":
                            ca_b = float(at.b_iso)
                            break
                    if ca_b is not None:
                        if 0 <= ca_b <= 100:
                            vals.append(ca_b); continue
                    bs = [float(a.b_iso) for a in res
                          if 0 <= float(getattr(a, "b_iso", -1)) <= 100]
                    if bs:
                        vals.append(sum(bs) / len(bs))
                if vals:
                    result[ch_obj.name] = vals
    except Exception:
        pass
    finally:
        if tmp_path:
            try: os.remove(tmp_path)
            except OSError: pass
    return result


def classify_plddt(txt: str, fmt: str,
                   only_chain: Optional[str] = None) -> dict:
    out: dict[str, dict[str, list[int]]] = {
        "lt50": {}, "m50": {}, "m70": {}, "ge90": {}, "lt70": {},
    }
    def _add(ch: str, resi: int, v: float) -> None:
        if v < 50:
            out["lt50"].setdefault(ch, []).append(resi)
            out["lt70"].setdefault(ch, []).append(resi)
        elif v < 70:
            out["m50"].setdefault(ch, []).append(resi)
            out["lt70"].setdefault(ch, []).append(resi)
        elif v < 90:
            out["m70"].setdefault(ch, []).append(resi)
        else:
            out["ge90"].setdefault(ch, []).append(resi)
    if fmt == "pdb":
        for ln in (txt or "").splitlines():
            if not ln.startswith("ATOM") or len(ln) < 66:
                continue
            if ln[12:16].strip() != "CA":
                continue
            ch = ln[21:22].strip() or "A"
            if only_chain and ch != only_chain:
                continue
            try: _add(ch, int(ln[22:26]), float(ln[60:66]))
            except ValueError: pass
        return out
    tmp_path = None
    try:
        tmp = tempfile.NamedTemporaryFile("w", suffix=".cif", delete=False)
        tmp.write(txt or ""); tmp.close(); tmp_path = tmp.name
        st_obj = gemmi.read_structure(tmp_path)
        if len(st_obj) > 0:
            for ch_obj in st_obj[0]:
                if not any(is_amino_acid(r) for r in ch_obj):
                    continue
                if only_chain and ch_obj.name != only_chain:
                    continue
                for res in ch_obj:
                    if not is_amino_acid(res):
                        continue
                    v = None
                    for at in res:
                        if at.name == "CA":
                            v = float(at.b_iso)
                            break
                    if v is None:
                        bs = [float(a.b_iso) for a in res
                              if 0 <= float(getattr(a, "b_iso", -1)) <= 100]
                        if bs: v = sum(bs) / len(bs)
                    try: resi = int(res.seqid.num)
                    except Exception: continue
                    if v is not None and 0 <= v <= 100:
                        _add(ch_obj.name, resi, v)
    except Exception:
        pass
    finally:
        if tmp_path:
            try: os.remove(tmp_path)
            except OSError: pass
    return out


def plddt_bins(vals: list[float]) -> dict[str, int]:
    b = {n: 0 for n in PLDDT_SCHEME}
    for v in vals:
        if v < 50: b["Very low (<50)"] += 1
        elif v < 70: b["Low (50-70)"] += 1
        elif v < 90: b["High (70-90)"] += 1
        else: b["Very high (>90)"] += 1
    return b


def plddt_legend() -> str:
    items = "".join(
        f'<div style="display:flex;align-items:center;margin:6px 0">'
        f'<span style="width:14px;height:14px;background:{c["color"]};'
        f'border-radius:2px;margin-right:8px;display:inline-block"></span>'
        f'{n}</div>'
        for n, c in PLDDT_SCHEME.items()
    )
    return (
        '<div style="border-radius:10px;border:1px solid #dce1ea;'
        'max-width:360px"><div style="background:#e9f0ff;padding:10px 12px;'
        'font-weight:700;color:#163dff">Model Confidence</div>'
        f'<div style="padding:10px 12px;background:#fff">{items}</div></div>'
    )


def plddt_figures(txt: str, fmt: str, title: str = "pLDDT",
                  chain: Optional[str] = None):
    pc = plddt_by_chain(txt, fmt, chain)
    vals = [v for arr in pc.values() for v in arr if 0 <= v <= 100]
    counts = plddt_bins(vals)
    names = list(PLDDT_SCHEME.keys())
    colors = [PLDDT_SCHEME[n]["color"] for n in names]
    pie = px.pie(values=[counts[n] for n in names], names=names,
                 title=f"pLDDT — {title}")
    pie.update_traces(textposition="inside", textinfo="percent+label",
                      marker=dict(colors=colors))
    pie.update_layout(margin=dict(l=10, r=10, t=40, b=10))
    mean = float(np.mean(vals)) if vals else 0.0
    gauge = go.Figure(go.Indicator(
        mode="gauge+number", value=mean,
        number={"suffix": " pLDDT", "valueformat": ".1f"},
        gauge={
            "axis": {"range": [0, 100]}, "bar": {"color": C_VHIGH},
            "steps": [
                {"range": [0, 50], "color": C_VLOW},
                {"range": [50, 70], "color": C_LOW},
                {"range": [70, 90], "color": C_HIGH},
                {"range": [90, 100], "color": C_VHIGH},
            ],
            "threshold": {"line": {"color": "black", "width": 2},
                          "thickness": 0.75, "value": mean},
        },
        title={"text": "Average pLDDT"},
    ))
    gauge.update_layout(margin=dict(l=10, r=10, t=40, b=10))
    return pie, gauge


def show_plddt_panels(txt: str, fmt: str, label: str,
                      chain: Optional[str], prefix: str) -> None:
    if not txt:
        st.info("Empty structure data."); return
    if st.checkbox("Show pLDDT legend", True, key=f"{prefix}_leg"):
        st.markdown(plddt_legend(), unsafe_allow_html=True)
    title = Path(label).name if label else ""
    p, g = plddt_figures(txt, fmt, title, chain)
    c1, c2 = st.columns(2)
    with c1: st.plotly_chart(p, use_container_width=True)
    with c2: st.plotly_chart(g, use_container_width=True)


def plddt_profile(path: str, fmt: str,
                  chain: Optional[str] = None
                  ) -> tuple[Optional[str], Optional[list[float]]]:
    try: txt = open(path).read()
    except Exception: return None, None
    pc = plddt_by_chain(txt, fmt, chain)
    if not pc: pc = plddt_by_chain(txt, fmt)
    if not pc: return None, None
    cid, vals = max(pc.items(), key=lambda kv: len(kv[1]))
    prof = [v for v in vals if 0 <= v <= 100]
    return (cid, prof) if prof else (None, None)

In [ ]:
%%writefile alphafold_fusion/pae.py
"""PAE (Predicted Aligned Error) heatmap generation and pickle loading."""

from __future__ import annotations
import json, pickle, re
from pathlib import Path
from typing import Optional, Union
import numpy as np
import plotly.express as px


def pae_heatmap(source: Union[str, Path, np.ndarray],
                title: str = "PAE Heatmap"):
    try:
        if isinstance(source, (str, Path)):
            with open(source) as f:
                j = json.load(f)
            mat = j.get("pae") or j.get("predicted_aligned_error")
        else:
            mat = source
        if mat is None:
            return None
        arr = np.array(mat)
        fig = px.imshow(
            arr, color_continuous_scale="Viridis", origin="lower",
            labels={"color": "PAE (Å)"}, title=title,
            zmin=0, zmax=max(30, float(np.nanmax(arr))),
            aspect="auto",
        )
        fig.update_layout(margin=dict(l=10, r=10, t=40, b=10))
        return fig
    except Exception:
        return None


def pae_from_pkl(job_dir: Path,
                 model_id: str) -> Optional[np.ndarray]:
    if not job_dir.exists() or not model_id:
        return None
    # model_id may be stored as "model_1" or "model1"; ColabFold pkl
    # filenames use "result_model_1_...". Normalise to the underscore
    # form so the glob matches regardless of the stored variant.
    mid_glob = re.sub(r"model_?(\d+)", r"model_\1", model_id)
    files = (list(job_dir.rglob(f"result_{mid_glob}*.pkl"))
             or list(job_dir.rglob(f"result_{model_id}*.pkl")))
    if not files:
        return None
    try:
        with open(
            sorted(files, key=lambda p: p.stat().st_mtime)[-1], "rb"
        ) as f:
            # SECURITY: pickle.load executes arbitrary code. Only load
            # .pkl files produced by our own ColabFold runs, never from
            # untrusted sources.
            d = pickle.load(f)
        pae = d.get("predicted_aligned_error") or d.get("pae")
        if pae is None and "pae_output" in d:
            pae = (d["pae_output"] or {}).get("pae")
        return np.array(pae) if pae is not None else None
    except Exception:
        return None

In [ ]:
%%writefile alphafold_fusion/render.py
"""3-D molecular visualisation with py3Dmol and domain overlays."""

from __future__ import annotations
import hashlib, re
from typing import Any, Optional
import py3Dmol
from .config import (
    C_HIGH, C_LOW, C_SPEC_GE70, C_SPEC_GE90, C_SPEC_LT70,
    C_VHIGH, C_VLOW, DOMAIN_COLORS, PY3DMOL_CDN,
)
from .plddt import classify_plddt
from .structure import detect_fmt


def viewer_key(prefix: str, **kw) -> str:
    payload = "|".join(str(v) for v in kw.values())
    return f"{prefix}::{hashlib.md5(payload.encode()).hexdigest()}"


def patch_cdn(html: str) -> str:
    return re.sub(
        r'src="https?://[^"]*3Dmol[^"]*\.js"',
        f'src="{PY3DMOL_CDN}"', html or "", count=1,
    )


def render_3d(txt: str, fmt: str, style: str, scheme: str,
              mono: bool = False, chain: Optional[str] = None,
              dark: bool = False) -> str:
    v = py3Dmol.view(width=1000, height=650)
    v.addModel(txt, "cif" if fmt == "cif" else "pdb")
    v.setBackgroundColor("black" if dark else "white")
    sel_g: dict[str, Any] = {"chain": chain} if (mono and chain) else {}
    rep = (style or "Cartoon").lower()
    if rep not in ("cartoon", "stick", "sphere", "line", "surface"):
        rep = "cartoon"

    def _sty(sel: dict, color: str) -> None:
        m = {
            "cartoon": {"cartoon": {"color": color}},
            "stick":   {"stick": {"color": color, "radius": 0.3}},
            "sphere":  {"sphere": {"color": color, "radius": 1.0}},
            "line":    {"line": {"color": color}},
            "surface": {"surface": {"opacity": 0.85, "color": color}},
        }
        v.setStyle(sel, m[rep])

    _f = fmt if fmt in ("pdb", "cif") else detect_fmt(txt)
    cls = classify_plddt(txt, _f, chain if mono else None)
    has = any(cls.get(k) for k in ("lt50", "m50", "m70", "ge90"))

    eff = scheme
    if scheme in ("AlphaFold (4-color)", "Special (blue/orange)") and not has:
        eff = "pLDDT (B-factor)"

    if eff == "AlphaFold (4-color)":
        for k, c in [("lt50", C_VLOW), ("m50", C_LOW),
                      ("m70", C_HIGH), ("ge90", C_VHIGH)]:
            for ch_name, rl in (cls.get(k) or {}).items():
                if rl:
                    s = dict(sel_g); s["resi"] = rl
                    if not mono: s["chain"] = ch_name
                    _sty(s, c)
    elif eff == "Special (blue/orange)":
        for k, c in [("lt70", C_SPEC_LT70), ("m70", C_SPEC_GE70),
                      ("ge90", C_SPEC_GE90)]:
            for ch_name, rl in (cls.get(k) or {}).items():
                if rl:
                    s = dict(sel_g); s["resi"] = rl
                    if not mono: s["chain"] = ch_name
                    _sty(s, c)
    else:
        cs_map = {
            "pLDDT (B-factor)": {"prop": "b", "gradient": "roygb",
                                  "min": 0, "max": 100},
            "Spectrum": "spectrum",
            "Chain": "chain",
        }
        cs = cs_map.get(eff, {"prop": "b", "gradient": "roygb",
                               "min": 0, "max": 100})
        sty_dict = {"colorscheme": cs}
        if rep == "surface":
            v.setStyle(sel_g, {"surface": {"opacity": 0.85}})
        elif rep == "stick":
            v.setStyle(sel_g, {"stick": {**sty_dict, "radius": 0.3}})
        elif rep == "sphere":
            v.setStyle(sel_g, {"sphere": {**sty_dict, "radius": 1.0}})
        else:
            v.setStyle(sel_g, {rep: sty_dict})

    v.zoomTo(); v.render()
    return v._make_html()


def render_domains_3d(txt: str, fmt: str, segs: list[dict],
                      style: str = "Cartoon",
                      chain: Optional[str] = None) -> str:
    v = py3Dmol.view(width=1000, height=650)
    v.addModel(txt, "cif" if fmt == "cif" else "pdb")
    v.setStyle({}, {"cartoon": {"color": "#DDD"}})
    for i, s in enumerate(segs):
        c = DOMAIN_COLORS[i % len(DOMAIN_COLORS)]
        rng = list(range(int(s["start"]), int(s["end"]) + 1))
        sel: dict[str, Any] = {"resi": rng}
        if chain: sel["chain"] = chain
        style_map = {
            "Cartoon": {"cartoon": {"color": c}},
            "Stick":   {"stick": {"color": c, "radius": 0.3}},
            "Sphere":  {"sphere": {"color": c, "radius": 1.0}},
            "Line":    {"line": {"color": c}},
        }
        v.setStyle(sel, style_map.get(style, {"cartoon": {"color": c}}))
    v.setBackgroundColor("white"); v.zoomTo()
    return v._make_html()


def domain_legend(segs: list[dict]) -> str:
    items = "".join(
        f'<div style="display:flex;align-items:center;margin:4px 0">'
        f'<span style="width:14px;height:14px;background:'
        f'{DOMAIN_COLORS[i % len(DOMAIN_COLORS)]};border-radius:2px;'
        f'margin-right:8px;display:inline-block"></span>'
        f'{s.get("label", "?")} ({s["start"]}–{s["end"]})</div>'
        for i, s in enumerate(segs)
    )
    return (
        '<div style="border:1px solid #e5e7eb;border-radius:10px;'
        f'padding:10px;background:#fff">{items}</div>'
    )
def confidence_style(value, kind="plddt"):
    """Return (pct, quality, color) for a metric value.

    kind: 'plddt' (0-100), 'score' (0-1: pTM/ipTM/F1...),
          'mcc' (-1..1), 'fraction' (0-1).
    """
    if value is None:
        return None, None, "#64748B"
    v = float(value)
    if kind == "plddt":
        pct = max(0, min(100, v))
        if v >= 90:  return pct, "Very High", "#059669"
        if v >= 70:  return pct, "High",      "#2563EB"
        if v >= 50:  return pct, "Medium",    "#D97706"
        return pct, "Low", "#DC2626"
    if kind == "score":            # pTM, ipTM, F1, precision, recall
        pct = max(0, min(100, v * 100))
        if v >= 0.80: return pct, "High",   "#059669"
        if v >= 0.60: return pct, "Good",   "#2563EB"
        if v >= 0.40: return pct, "Fair",   "#D97706"
        return pct, "Low", "#DC2626"
    if kind == "mcc":              # -1..1
        pct = max(0, min(100, (v + 1) / 2 * 100))
        if v >= 0.50: return pct, "Strong",   "#059669"
        if v >= 0.30: return pct, "Moderate", "#2563EB"
        if v >= 0.10: return pct, "Weak",     "#D97706"
        return pct, "Poor", "#DC2626"
    # fraction (0-1) — neutral
    pct = max(0, min(100, v * 100))
    return pct, None, "#7C3AED"


def metric_card(label, value, unit="", pct=None,
                quality=None, color="#2563EB"):
    """Scientific metric card with optional confidence bar.

    Use directly, or pair with confidence_style() for auto colors:
        pct, q, c = confidence_style(0.84, "score")
        metric_card("pTM", "0.84", pct=pct, quality=q, color=c)
    """
    bar = ""
    if pct is not None:
        bar = (f'<div style="background:#E2E8F0;border-radius:6px;'
               f'height:8px;margin-top:8px;overflow:hidden">'
               f'<div style="background:{color};width:{pct}%;'
               f'height:100%;border-radius:6px"></div></div>')
    q = (f'<span style="float:right;font-size:.75rem;color:{color};'
         f'font-weight:600">{quality}</span>' if quality else "")
    return (f'<div style="background:#fff;border:1px solid #E2E8F0;'
            f'border-radius:12px;padding:16px 18px;'
            f'box-shadow:0 1px 3px rgba(0,0,0,.06)">'
            f'<div style="font-size:.8rem;color:#64748B;'
            f'font-weight:500">{label}{q}</div>'
            f'<div style="font-size:1.8rem;font-weight:700;'
            f'color:#1E293B;margin-top:4px">{value}'
            f'<span style="font-size:1rem;color:#94A3B8">{unit}'
            f'</span></div>{bar}</div>')
def show_local_png(path, caption: str = "", width_pct: int = 100) -> bool:
    """Display a local PNG via base64 data-URI.

    Streamlit's st.image() serves files through an internal /media/
    endpoint that is NOT relayed behind the Colab proxyPort tunnel,
    producing broken images. Embedding the PNG as a base64 data-URI
    puts the bytes directly in the page, bypassing that endpoint.
    Returns True on success, False if the file is missing/empty.
    """
    import base64
    from pathlib import Path
    import streamlit as st
    try:
        p = Path(path)
        if not p.exists() or p.stat().st_size == 0:
            return False
        b64 = base64.b64encode(p.read_bytes()).decode()
        html = (
            f'<img src="data:image/png;base64,{b64}" '
            f'style="width:{width_pct}%;height:auto;border-radius:8px" '
            f'alt="{caption}"/>'
        )
        if caption:
            html += (f'<div style="text-align:center;color:#666;'
                     f'font-size:0.85rem;margin-top:4px">{caption}</div>')
        st.markdown(html, unsafe_allow_html=True)
        return True
    except Exception:
        return False

In [ ]:
%%writefile alphafold_fusion/alignment.py
"""A3M multiple-sequence-alignment parsing and enrichment pipeline."""

from __future__ import annotations
import gzip, re
from pathlib import Path
from typing import Any, Optional
import numpy as np
from .config import log
from .sequence import parse_fasta


def find_a3m(job_dir: Path) -> list[Path]:
    """Find all .a3m / .a3m.gz files in job_dir (recursive)."""
    if not job_dir.exists():
        log.info("find_a3m: directory does not exist: %s", job_dir)
        return []
    files: list[Path] = []
    # ══════ Patterns élargis ══════
    for pat in ("*.a3m", "*.a3m.gz", "*.aln"):
        files.extend(job_dir.rglob(pat))

    # ColabFold crée parfois un sous-dossier <jobname>/ avec le .a3m dedans
    # rglob couvre déjà ça, mais on vérifie aussi le parent
    if not files and job_dir.parent.exists():
        for pat in ("*.a3m", "*.a3m.gz"):
            files.extend(job_dir.parent.rglob(pat))

    if not files:
        # Log diagnostique : qu'y a-t-il dans le répertoire ?
        try:
            all_files = sorted(job_dir.rglob("*"))
            extensions = set(f.suffix for f in all_files if f.is_file())
            log.info("find_a3m: no .a3m in %s — found %d files, "
                     "extensions: %s", job_dir.name, len(all_files),
                     extensions or "none")
        except Exception:
            pass
        return []

    def _score(p: Path) -> tuple[int, float]:
        n = p.name.lower()
        s = (100 if "uniref" in n else 0) + (50 if "bfd" in n else 0)
        try: return (s, p.stat().st_mtime)
        except Exception: return (s, 0)

    result = sorted(set(files), key=_score, reverse=True)
    log.info("find_a3m: found %d file(s) in %s: %s",
             len(result), job_dir.name,
             [f.name for f in result[:5]])
    return result


def _read_a3m(path: Path) -> str:
    try:
        if str(path).endswith(".gz"):
            with gzip.open(path, "rt", errors="ignore") as f:
                return f.read()
        with open(path, errors="ignore") as f:
            return f.read()
    except Exception as e:
        log.warning("_read_a3m: cannot read %s: %s", path.name, e)
        return ""


def _parse_a3m(path: Path) -> Optional[dict]:
    if not path or not path.exists():
        return None
    text = _read_a3m(path)
    if not text.strip():
        log.warning("_parse_a3m: empty file %s", path.name)
        return None

    entries: list[tuple[str, str]] = []
    head: Optional[str] = None
    buf: list[str] = []
    for ln in text.splitlines():
        if ln.startswith(">"):
            if head is not None:
                entries.append((head, "".join(buf)))
            head = ln[1:].strip(); buf = []
        elif ln.startswith("#"):
            continue  # skip comment lines
        else:
            buf.append(ln.strip())
    if head is not None:
        entries.append((head, "".join(buf)))

    if not entries:
        log.warning("_parse_a3m: no entries in %s", path.name)
        return None

    log.info("_parse_a3m: %s → %d entries (query + %d hits)",
             path.name, len(entries), len(entries) - 1)

    qh, qa = entries[0]
    hits: list[dict] = []
    for hdr, aln in entries[1:]:
        toks = hdr.split()
        meta: dict[str, Any] = {"score": None, "id_pct": None, "evalue": None}
        if len(toks) >= 4:
            try: meta["score"] = float(toks[1])
            except (TypeError, ValueError): pass
            try:
                v = float(toks[2].rstrip("%"))
                meta["id_pct"] = v if v > 1 else v * 100
            except (TypeError, ValueError): pass
            try: meta["evalue"] = float(toks[3].replace("E", "e"))
            except (TypeError, ValueError): pass
        hits.append({"acc": toks[0] if toks else hdr,
                     "aln": aln, "meta": meta})
    return {"qh": qh, "qa": qa, "hits": hits}


def align_a3m(q: str, t: str) -> tuple[str, str]:
    qi = ti = 0
    ql: list[str] = []
    tl: list[str] = []
    while qi < len(q) or ti < len(t):
        qc = q[qi] if qi < len(q) else None
        tc = t[ti] if ti < len(t) else None
        if qc is not None and qc.islower():
            ql.append(qc); tl.append("-"); qi += 1; continue
        if tc is not None and tc.islower():
            tl.append(tc); ql.append("-"); ti += 1; continue
        ql.append(qc or "-"); tl.append(tc or "-")
        if qc is not None: qi += 1
        if tc is not None: ti += 1
    return "".join(ql), "".join(tl)


def compute_identity(ql: str, tl: str,
                     qlen: int) -> tuple[Optional[float], Optional[float], int]:
    match = aligned = cov = 0
    for q, t in zip(ql, tl):
        if q != "-": cov += 1
        if t != "-":
            aligned += 1
            if q.upper() == t.upper(): match += 1
    ident = (100 * match / aligned) if aligned > 0 else None
    coverage = round(100 * cov / max(1, qlen), 1) if qlen else None
    return ident, coverage, aligned


def build_a3m_data(name: str, job_dir: Path) -> Optional[dict]:
    """Build alignment data from .a3m files in job_dir."""
    a3m_files = find_a3m(job_dir)
    if not a3m_files:
        log.info("build_a3m_data: no .a3m files for '%s' in %s",
                 name, job_dir)
        return None

    parsed = None
    for f in a3m_files[:6]:
        parsed = _parse_a3m(f)
        if parsed and parsed.get("hits"):
            break
        parsed = None  # reset if no hits

    if not parsed:
        log.info("build_a3m_data: .a3m found but no parseable hits "
                 "for '%s'", name)
        return None

    qseq = re.sub(r"[^A-Za-z]", "", parsed["qa"]).upper()
    qlen = len(qseq)
    enriched: list[dict] = []
    for h in parsed["hits"]:
        try: ql, tl = align_a3m(parsed["qa"], h["aln"])
        except Exception: continue
        ident, cov, core = compute_identity(ql, tl, qlen)
        enriched.append({
            "acc": h["acc"],
            "aln": h["aln"],
            "Identity_pct": h["meta"].get("id_pct") or ident,
            "Score": h["meta"].get("score"),
            "Evalue": h["meta"].get("evalue"),
            "Coverage_pct": cov,
            "Core": core,
        })

    if not enriched:
        log.info("build_a3m_data: parsed but 0 enriched hits for '%s'", name)
        return None

    def _sk(x: dict) -> tuple:
        def _v(k, d):
            val = x.get(k)
            if val is None: return d
            if isinstance(val, float) and not np.isfinite(val): return d
            return val
        return (_v("Identity_pct", -1), _v("Score", -1),
                -_v("Evalue", float("inf")), _v("Coverage_pct", -1))
    enriched.sort(key=_sk, reverse=True)

    log.info("build_a3m_data: '%s' → %d hits enriched", name, len(enriched))
    return {"name": f"{name} (AUTO)", "qseq": qseq,
            "qlen": qlen, "hits": enriched}


def parse_text_block(txt: str) -> Optional[dict]:
    entries = parse_fasta(txt)
    if len(entries) < 2:
        return None
    qh, qs = entries[0]
    qseq = re.sub(r"[^A-Za-z]", "", qs).upper()
    hits = [{"acc": re.split(r"\s+", h)[0], "aln": s, "meta": {}}
            for h, s in entries[1:]]
    return {"name": f"Manual {qh}", "qseq": qseq,
            "qlen": len(qseq), "hits": hits}

In [ ]:
%%writefile alphafold_fusion/runner.py
"""ColabFold execution, result harvesting, and analysis."""

from __future__ import annotations
import hashlib, json, os, re, shutil, subprocess, sys, time
from pathlib import Path
from typing import Any, Optional
import numpy as np
from .config import CACHE_DIR, MSA_MODES, RESULTS_DIR
from .structure import avg_plddt, cif_to_pdb


def has_gpu() -> bool:
    """Quick check: is a GPU physically present?"""
    return shutil.which("nvidia-smi") is not None

def jax_backend() -> str:
    """Return JAX's active backend ('gpu', 'cpu', or 'unknown').

    ColabFold silently falls back to CPU when JAX cannot see the GPU
    (e.g. after a NumPy/CUDA desync), yielding catastrophically low
    pLDDT with no visible error. We probe the actual JAX backend—not
    just nvidia-smi—so the UI can warn before a long CPU run.
    """
    try:
        r = subprocess.run(
            [sys.executable, "-c",
             "import jax; print(jax.default_backend())"],
            capture_output=True, text=True, timeout=30,
        )
        out = (r.stdout or "").strip().lower()
        if "gpu" in out:
            return "gpu"
        if "cpu" in out:
            return "cpu"
        return "unknown"
    except Exception:
        return "unknown"

def _parse_ranking(job_dir: Path) -> dict:
    files: list[Path] = []
    for pat in ("ranking_debug.json", "ranking.json"):
        files = list(job_dir.rglob(pat))
        if files: break
    if not files: return {}
    try:
        with open(files[0]) as f:
            d = json.load(f)
        out: dict[str, Any] = {}
        if isinstance(d.get("order"), list): out["order"] = d["order"]
        for k in ("ranking_confidence", "plddts", "iptms", "ptms"):
            if isinstance(d.get(k), dict): out[k] = d[k]
        return out
    except Exception: return {}


def _extract_model_id(p: Path) -> Optional[str]:
    n = p.name
    patterns = [
        # KEEP underscore: "model_1" (matches ColabFold naming)
        (r"model_?(\d+)", lambda m: f"model_{m.group(1)}"),
        (r"ranked_?(\d+)", lambda m: f"model_{int(m.group(1)) + 1}"),
    ]
    for pat, fn in patterns:
        match = re.search(pat, n, re.I)
        if match:
            try: return fn(match)
            except Exception: pass
    return None


def _rank_from_name(name: str) -> Optional[int]:
    for pat in (r"ranked_?(\d+)", r"rank_?(\d+)"):
        m = re.search(pat, name, re.I)
        if m:
            try:
                offset = 1 if "ranked" in pat else 0
                return int(m.group(1)) + offset
            except Exception: pass
    return None

def _model_num(s: str) -> Optional[str]:
    """Extract the numeric model index from any model id / json key."""
    m = re.search(r"model_?(\d+)", s or "", re.I)
    return m.group(1) if m else None


def _lookup(d: Optional[dict], mid: str):
    """Tolerant lookup by model number.

    ColabFold ranking keys look like 'model_1_multimer_v3_pred_0'
    while our mid is 'model_1'. Match on the numeric index to avoid
    the model_1 / model_10 false-positive trap.
    """
    if not d or not mid:
        return None
    if mid in d:
        return d[mid]
    num = _model_num(mid)
    if num is None:
        return None
    for k, v in d.items():
        if _model_num(k) == num:
            return v
    return None


def _is_template(p: Path) -> bool:
    """True if a file lives under a ColabFold 'templates_*/' subdir.

    ColabFold downloads candidate PDB templates into 'templates_NNN/'
    subdirectories when --templates is enabled. These are experimental
    reference structures, NOT predicted models: they carry no model_N
    id, no rank and no pTM/ipTM. Including them in harvest() pollutes
    the model table (empty ranks/scores, template PDB codes shown as
    'best model'). We therefore exclude them everywhere in harvesting.
    """
    return "template" in str(p).lower()


def _scores_from_files(job_dir: Path) -> dict:
    """Read per-model ptm/iptm from ColabFold *_scores_*.json files.

    ColabFold 1.6.x stores scores per model in separate JSON files
    named '*_scores_rank_NNN_..._model_M_...json' instead of a single
    ranking_debug.json. Returns {model_num: {ptm, iptm}}.
    """
    out: dict[str, dict] = {}
    for f in job_dir.rglob("*scores_rank*.json"):
        if _is_template(f):
            continue
        try:
            with open(f) as fh:
                d = json.load(fh)
        except Exception:
            continue
        num = _model_num(f.name)   # extrait le numéro de model_M
        if num is None:
            continue
        out[num] = {
            "ptm": d.get("ptm"),
            "iptm": d.get("iptm"),
        }
    return out


def harvest(job_dir: Path) -> tuple[list[dict], dict]:
    rnk = _parse_ranking(job_dir)
    scores = _scores_from_files(job_dir)
    # ══════ FIX: exclude template files (ColabFold templates_*/ subdirs).
    # rglob is recursive and would otherwise pick up downloaded PDB
    # templates as if they were predicted models. ══════
    pdbs = sorted(
        (p for p in job_dir.rglob("*.pdb") if not _is_template(p)),
        key=lambda p: p.stat().st_mtime)
    cifs = sorted(
        (p for p in (list(job_dir.rglob("*.cif"))
                     + list(job_dir.rglob("*.bcif")))
         if not _is_template(p)),
        key=lambda p: p.stat().st_mtime,
    )
    # ════════════════════════════════════════════════════════════════
    if not pdbs and cifs:
        for c in cifs:
            out = c.with_name(c.stem + "_conv.pdb")
            try:
                if cif_to_pdb(c, out): pdbs.append(out)
            except Exception: pass
        pdbs.sort(key=lambda p: p.stat().st_mtime)
    order = rnk.get("order", [])
    rows: list[dict] = []
    for fmt, path in [("pdb", p) for p in pdbs] + [("cif", c) for c in cifs]:
        mid = _extract_model_id(path)
        rpos = None
        if mid and order:
            # match rank order by model number (avoids model_1/model_10 trap)
            for oi, ok in enumerate(order):
                if _model_num(ok) == _model_num(mid):
                    rpos = oi + 1
                    break
        if rpos is None:
            rpos = _rank_from_name(path.name)
        rows.append({
            "model_id": mid or "-", "file": str(path), "fmt": fmt,
            "avg_plddt": avg_plddt(path, fmt), "rank": rpos,
            "ranking_conf": _lookup(rnk.get("ranking_confidence"), mid),
            "ptm": _lookup(rnk.get("ptms"), mid)
                   or (scores.get(_model_num(mid) or "", {}).get("ptm")),
            "iptm": _lookup(rnk.get("iptms"), mid)
                    or (scores.get(_model_num(mid) or "", {}).get("iptm")),
        })
    rows.sort(key=lambda x: (
        x["rank"] if x["rank"] is not None else 9999,
        -(x["avg_plddt"] or 0),
    ))
    return rows, rnk


def analyze(job_dir: Path) -> dict:
    rows, rnk = harvest(job_dir)
    res: dict[str, Any] = {
        "status": "error", "message": "No PDB/CIF", "models": rows,
        "ranking": rnk, "pae_json": None, "pae_png": None, "coverage_png": None,
    }
    if rows:
        res["status"] = "success"; res["message"] = ""
    for tag, key in [("*pae*.json", "pae_json"),
                     ("*predicted_aligned_error*.json", "pae_json"),
                     ("*pae*.png", "pae_png"),
                     ("*coverage*.png", "coverage_png")]:
        if res.get(key):
            continue
        # ══════ FIX: also skip template dirs when locating PAE/coverage ══════
        files = [f for f in job_dir.rglob(tag) if not _is_template(f)]
        if files:
            res[key] = str(sorted(files, key=lambda f: f.stat().st_mtime)[-1])
    return res


def quality_warnings(metrics: dict) -> list[str]:
    w: list[str] = []
    v = metrics.get("avg_plddt")
    if v is not None and v < 70:
        w.append(f"⚠️ Low mean pLDDT ({v:.1f}): proceed with caution.")
    v = metrics.get("iptm")
    if v is not None and v < 0.4:
        w.append(f"⚠️ Low ipTM ({v:.2f}): interfaces likely uncertain.")
    v = metrics.get("ptm")
    if v is not None and v < 0.6:
        w.append(f"⚠️ Low pTM ({v:.2f}): global topology may be uncertain.")
    return w
def count_msa_sequences(job_dir: Path) -> Optional[int]:
    """Count sequences in the deepest .a3m of a job (max over files).

    ColabFold may leave an empty single-sequence .a3m from a failed
    MSA search alongside the real one; we take the MAX depth found.
    Returns None if no .a3m present.
    """
    import gzip, glob
    files = list(job_dir.rglob("*.a3m")) + list(job_dir.rglob("*.a3m.gz"))
    if not files:
        return None
    best = 0
    for f in files:
        try:
            opener = gzip.open if str(f).endswith(".gz") else open
            with opener(f, "rt", errors="ignore") as fh:
                n = sum(1 for line in fh if line.startswith(">"))
            best = max(best, n)
        except Exception:
            continue
    return best


def msa_warning(n_seqs: Optional[int],
                avg_plddt: Optional[float]) -> Optional[str]:
    """Warn when a low pLDDT is caused by an empty MSA, not disorder.

    A single-sequence MSA (n_seqs <= 1) forces ColabFold into
    single-sequence mode, collapsing pLDDT even for well-folded
    proteins. This must NOT be interpreted as intrinsic disorder.
    """
    if n_seqs is not None and n_seqs <= 1 and \
       avg_plddt is not None and avg_plddt < 70:
        return ("⚠️ Empty MSA (1 sequence): prediction ran in "
                "single-sequence mode. The low pLDDT reflects the "
                "MISSING MSA, NOT intrinsic disorder. Re-run with a "
                "valid MSA (check MMseqs2 server / disable cache).")
    return None

def model_name_for_path(path: str, results: dict) -> Optional[str]:
    for n, r in (results or {}).items():
        for m in (r or {}).get("models", []):
            if m.get("file") == path: return n
    return None


def _estimate_timeout(seq_len: int, msa_mode: str, params: dict) -> int:
    base = {"mmseqs2_uniref_env": 3000,
            "mmseqs2_uniref": 1800}.get(msa_mode, 600)
    factor = (
        max(1, params.get("num_models", 1))
        * max(1.0, params.get("num_recycles", 3) / 3.0)
        * (1.5 if "multimer" in str(params.get("model_type", "")) else 1.0)
        * min(4.0, max(1.0, (seq_len or 300) / 300))
    )
    return max(600, min(21600, int(base * factor)))


def _run_subprocess(cmd: list[str], timeout_sec: int = 7200,
                    env: Optional[dict] = None) -> tuple[bool, str]:
    try:
        p = subprocess.run(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, env=env, timeout=timeout_sec, check=False,
        )
        return p.returncode == 0, p.stdout or ""
    except subprocess.TimeoutExpired as e:
        return False, (e.stdout or "") + "\n[TIMEOUT]"
    except Exception as e:
        return False, str(e)


def run_colabfold(fasta: str, outdir: str, params: dict,
                  seq_len: int = 0) -> tuple[bool, str]:
    exe = shutil.which("colabfold_batch")
    cmd = [exe or sys.executable]
    if not exe: cmd += ["-m", "colabfold.batch"]
    cmd += [fasta, outdir]
    msa = MSA_MODES.get(
        str(params.get("msa_strategy", "fast")), "mmseqs2_uniref")
    if params.get("model_type"):
        cmd += ["--model-type", params["model_type"]]
    cmd += ["--msa-mode", msa]
    if params.get("pair_mode"):
        cmd += ["--pair-mode", params["pair_mode"]]
    cmd += [
        "--num-models", str(params.get("num_models", 1)),
        "--num-recycle", str(params.get("num_recycles", 3)),
        "--rank", "auto",
        "--jobname", params.get("jobname_prefix", "job"),
        "--recompile-padding", "1",
    ]
    if params.get("use_templates"): cmd += ["--templates"]
    if params.get("use_amber"): cmd += ["--amber"]
    sc = params.get("stop_at_score")
    if sc is not None:
        try: cmd += ["--stop-at-score", f"{float(sc):.3f}"]
        except (TypeError, ValueError): pass
    if shutil.which("nvidia-smi"):
        cmd += ["--disable-unified-memory"]
    if not params.get("reuse_cache", True):
        cmd += ["--overwrite-existing-results"]
    env = {
        **os.environ,
        "TF_CPP_MIN_LOG_LEVEL": "3",
        "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
        "TF_FORCE_GPU_ALLOW_GROWTH": "true",
        "XLA_PYTHON_CLIENT_MEM_FRACTION": ".95",
    }
    return _run_subprocess(cmd, _estimate_timeout(seq_len, msa, params), env)


def fallback_cpu(fasta: str, outdir: str, prefix: str,
                 multimer: bool, seq_len: int) -> tuple[bool, str]:
    exe = shutil.which("colabfold_batch")
    cmd = [exe or sys.executable]
    if not exe: cmd += ["-m", "colabfold.batch"]
    cmd += [
        fasta, outdir,
        "--model-type",
        "alphafold2_multimer_v3" if multimer else "alphafold2_ptm",
        "--msa-mode", "single_sequence",
        "--num-models", "1", "--num-recycle", "2",
        # single_sequence ne supporte PAS paired
        "--pair-mode", "unpaired",
        "--rank", "auto", "--random-seed", "42",
        "--disable-unified-memory", "--recompile-padding", "1",
        "--jobname", prefix, "--max-msa", "64:64",
        "--disable-cluster-profile", "--stop-at-score", "70",
    ]
    env = {
        **os.environ,
        "JAX_PLUGINS": "disabled", "JAX_PLATFORM_NAME": "cpu",
        "TF_CPP_MIN_LOG_LEVEL": "3",
        "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
    }
    return _run_subprocess(
        cmd, _estimate_timeout(seq_len, "single_sequence", {}), env)

In [ ]:
%%writefile alphafold_fusion/disorder.py
"""Intrinsic disorder region (IDR) identification from pLDDT scores.

The pLDDT < 50 threshold corresponds to the "very low confidence"
band defined for AlphaFold2 (Jumper et al. 2021, Nature 596:583-589;
Tunyasuvunakool et al. 2021, Nature 596:590-596). The principle that
low pLDDT predicts intrinsic disorder was established by Akdel et al.
(2022) Nat Struct Mol Biol 29:1056-1067, who benchmarked AlphaFold2
pLDDT-based disorder prediction against IUPred2 using AUC-ROC.

In this tool, pLDDT < 50 is used as a disorder threshold and
validated independently against the DisProt database (Piovesan
et al. 2022, Nucleic Acids Res 50:D471-D477) using AUC-ROC and MCC.

The minimum IDR length of 5 residues is a heuristic short-region
filter (not defined by any specific reference).

GFF3 export uses Sequence Ontology term SO:0100002
(intrinsically_unstructured_polypeptide_region).
"""

from __future__ import annotations
import numpy as np
from typing import Optional
from .plddt import plddt_by_chain


def identify_idrs(
    plddt_vals: list[float],
    threshold: float = 50.0,
    min_length: int = 5,
) -> dict:
    """Segment disordered regions from per-residue pLDDT.

    Parameters
    ----------
    plddt_vals : per-residue pLDDT confidence scores
    threshold  : pLDDT < threshold classified as disordered.
                 Default 50.0 = AlphaFold2 "very low confidence" band
                 (Jumper et al. 2021, Nature 596:583; Tunyasuvunakool
                 et al. 2021, Nature 596:590). Low pLDDT predicts
                 disorder (Akdel et al. 2022, NSMB 29:1056).
    min_length : minimum consecutive residues. Default 5 is a
                 heuristic short-region filter (no specific reference).
    """
    n = len(plddt_vals)
    if n == 0:
        return {"regions": [], "fraction_disordered": 0.0,
                "per_residue": [], "n_residues": 0}

    binary = [1 if v < threshold else 0 for v in plddt_vals]

    regions: list[dict] = []
    start = None
    for i in range(n + 1):
        if i < n and binary[i] == 1:
            if start is None:
                start = i
        else:
            if start is not None:
                length = i - start
                if length >= min_length:
                    seg = plddt_vals[start:i]
                    regions.append({
                        "start": start + 1,
                        "end": i,
                        "length": length,
                        "mean_plddt": round(float(np.mean(seg)), 1),
                        "min_plddt": round(float(np.min(seg)), 1),
                    })
                start = None

    return {
        "regions": regions,
        "fraction_disordered": round(sum(binary) / n, 3),
        "per_residue": binary,
        "n_residues": n,
        "n_disordered_residues": sum(binary),
        "n_idr_regions": len(regions),
        "threshold_used": threshold,
        "min_length_used": min_length,
        "threshold_reference": "AF2 very low confidence band: Jumper et al. (2021) Nature 596:583; Tunyasuvunakool et al. (2021) Nature 596:590. Disorder principle: Akdel et al. (2022) NSMB 29:1056",
        "min_length_reference": "heuristic short-region filter (>=5 aa)",
    }


def disorder_from_structure(
    txt: str, fmt: str,
    chain: Optional[str] = None,
    threshold: float = 50.0,
    min_length: int = 5,
) -> dict[str, dict]:
    """IDR identification on all chains of a structure file."""
    pc = plddt_by_chain(txt, fmt, chain)
    return {
        ch: identify_idrs(vals, threshold, min_length)
        for ch, vals in pc.items()
    }


def to_gff3(
    disorder_result: dict,
    seqid: str = "query",
    chain: str = "A",
    source: str = "AlphaFoldFusion",
) -> str:
    """Export IDRs as GFF3 (Sequence Ontology SO:0100002)."""
    lines = ["##gff-version 3"]
    for i, r in enumerate(disorder_result.get("regions", []), 1):
        attrs = (f"ID=IDR_{chain}_{i};chain={chain};"
                 f"length={r['length']};"
                 f"mean_plddt={r['mean_plddt']};"
                 f"min_plddt={r['min_plddt']}")
        lines.append(
            f"{seqid}\t{source}\t"
            f"intrinsically_unstructured_polypeptide_region\t"
            f"{r['start']}\t{r['end']}\t"
            f"{r['mean_plddt']}\t.\t.\t{attrs}"
        )
    return "\n".join(lines)

In [ ]:
%%writefile alphafold_fusion/interface_analysis.py
"""Inter-chain Cα-Cα contact analysis for multimers.

Contact identification uses Cα-Cα distance with an 8 Angstrom
threshold (Duarte et al. 2012, BMC Bioinformatics 13:334). This is a
contact-map definition, NOT a SASA-buried interface (cf. PISA,
Krissinel & Henrick 2007, J Mol Biol 372:774-797).

Inter-chain PAE decomposition uses the TM-score kernel with
length-dependent d0 = 1.24(L-15)^{1/3} - 1.8
(Zhang & Skolnick 2004, Proteins 57:702-710).
"""

from __future__ import annotations
import os, tempfile
from collections import Counter
from typing import Optional
import numpy as np
import gemmi
from .structure import is_amino_acid


def _chain_data(txt: str, fmt: str) -> dict[str, dict]:
    """Extract per-chain Ca coordinates, B-factors, residue info."""
    tmp_path = None
    try:
        suffix = ".cif" if fmt == "cif" else ".pdb"
        tmp = tempfile.NamedTemporaryFile("w", suffix=suffix, delete=False)
        tmp.write(txt); tmp.flush(); tmp.close()
        tmp_path = tmp.name
        st_obj = gemmi.read_structure(tmp_path)
        if len(st_obj) == 0:
            return {}
        chains = {}
        for ch in st_obj[0]:
            if not any(is_amino_acid(r) for r in ch):
                continue
            coords, resnums, bfacs, resnames = [], [], [], []
            for res in ch:
                if not is_amino_acid(res):
                    continue
                ca = None
                for at in res:
                    if at.name == "CA":
                        ca = at
                        break
                if ca is not None:
                    coords.append([ca.pos.x, ca.pos.y, ca.pos.z])
                    try:
                        resnums.append(int(res.seqid.num))
                    except Exception:
                        resnums.append(len(resnums) + 1)
                    bfacs.append(float(ca.b_iso))
                    resnames.append(res.name)
            if coords:
                chains[ch.name] = {
                    "coords": np.array(coords), "resnums": resnums,
                    "bfacs": bfacs, "resnames": resnames}
        return chains
    except Exception:
        return {}
    finally:
        if tmp_path:
            try: os.remove(tmp_path)
            except OSError: pass


def compute_contacts(
    txt: str, fmt: str,
    chain_a: Optional[str] = None,
    chain_b: Optional[str] = None,
    cutoff: float = 8.0,
) -> dict:
    """Identify inter-chain contacts and compute interface metrics.

    Parameters
    ----------
    cutoff  : Ca-Ca distance threshold (Angstrom).
              Default 8.0 per Duarte et al. (2012) BMC Bioinformatics 13:334.

    Returns all contacts with distances and per-residue pLDDT.
    """
    chains = _chain_data(txt, fmt)
    if len(chains) < 2:
        return {"error": "Requires >= 2 protein chains", "n_contacts": 0}

    names = list(chains.keys())
    if chain_a is None: chain_a = names[0]
    if chain_b is None: chain_b = names[1] if len(names) > 1 else names[0]
    da, db = chains.get(chain_a), chains.get(chain_b)
    if da is None or db is None:
        return {"error": f"Chain {chain_a} or {chain_b} not found"}

    ca, cb = da["coords"], db["coords"]
    diff = ca[:, None, :] - cb[None, :, :]
    dists = np.sqrt(np.sum(diff ** 2, axis=-1))

    idx_a, idx_b = np.where(dists <= cutoff)
    contacts = []
    iface_a, iface_b = set(), set()

    for ia, ib in zip(idx_a, idx_b):
        contacts.append({
            "chain_a": chain_a, "res_a": da["resnums"][ia],
            "resname_a": da["resnames"][ia],
            "chain_b": chain_b, "res_b": db["resnums"][ib],
            "resname_b": db["resnames"][ib],
            "distance_A": round(float(dists[ia, ib]), 2),
            "plddt_a": round(da["bfacs"][ia], 1) if 0 <= da["bfacs"][ia] <= 100 else None,
            "plddt_b": round(db["bfacs"][ib], 1) if 0 <= db["bfacs"][ib] <= 100 else None,
        })
        iface_a.add(ia); iface_b.add(ib)

    iplddt_a = [da["bfacs"][i] for i in iface_a if 0 <= da["bfacs"][i] <= 100]
    iplddt_b = [db["bfacs"][i] for i in iface_b if 0 <= db["bfacs"][i] <= 100]
    all_ip = iplddt_a + iplddt_b

    cc_a, cc_b = Counter(idx_a), Counter(idx_b)
    interface_details_a = [
        {"resnum": da["resnums"][i], "resname": da["resnames"][i],
         "plddt": round(da["bfacs"][i], 1), "n_contacts": cc_a[i]}
        for i in sorted(iface_a)
    ]
    interface_details_b = [
        {"resnum": db["resnums"][i], "resname": db["resnames"][i],
         "plddt": round(db["bfacs"][i], 1), "n_contacts": cc_b[i]}
        for i in sorted(iface_b)
    ]

    return {
        "chain_a": chain_a, "chain_b": chain_b,
        "cutoff_A": cutoff,
        "cutoff_reference": "Duarte et al. (2012) BMC Bioinformatics 13:334",
        "n_contacts": len(contacts),
        "n_interface_a": len(iface_a),
        "n_interface_b": len(iface_b),
        "contacts": contacts,
        "interface_residues_a": sorted(da["resnums"][i] for i in iface_a),
        "interface_residues_b": sorted(db["resnums"][i] for i in iface_b),
        "interface_details_a": interface_details_a,
        "interface_details_b": interface_details_b,
        "mean_plddt_interface": round(float(np.mean(all_ip)), 1) if all_ip else None,
        "mean_plddt_interface_a": round(float(np.mean(iplddt_a)), 1) if iplddt_a else None,
        "mean_plddt_interface_b": round(float(np.mean(iplddt_b)), 1) if iplddt_b else None,
    }


def interchain_pae(
    pae: np.ndarray,
    chain_lengths: list[int],
) -> dict:
    """Decompose PAE into per-block TM-score kernel summaries.

    For each (i, j) chain block we report the mean of the TM-score
    kernel weight w = 1 / (1 + (PAE/d0)^2), with length-dependent
    d0(L) = 1.24(L-15)^{1/3} - 1.8 (Zhang & Skolnick 2004, Proteins
    57:702-710).

    IMPORTANT: `tm_kernel_score` is the MEAN of the TM kernel over the
    block. It is NOT the AlphaFold ipTM/pTM score, which is defined as
    a MAXIMUM over alignment rows of a mean TM-kernel term. We use the
    block mean as a lightweight, symmetric descriptor of inter-chain
    PAE quality; do not interpret it as AlphaFold's (i)pTM.
    """
    boundaries = []
    offset = 0
    for length in chain_lengths:
        boundaries.append((offset, offset + length))
        offset += length

    result = {}
    for i, (si, ei) in enumerate(boundaries):
        for j, (sj, ej) in enumerate(boundaries):
            block = pae[si:ei, sj:ej]
            L = ej - sj
            d0 = max(0.5, 1.24 * max(0, L - 15) ** (1.0 / 3) - 1.8)
            tm_w = 1.0 / (1.0 + (block / d0) ** 2)
            result[f"{i}_{j}"] = {
                "chain_i": i, "chain_j": j,
                "len_i": ei - si, "len_j": ej - sj,
                "mean_pae": round(float(np.mean(block)), 2),
                "median_pae": round(float(np.median(block)), 2),
                "tm_kernel_score": round(float(np.mean(tm_w)), 4),
                "d0_used": round(d0, 3),
                "is_interchain": i != j,
            }
    return result

In [ ]:
%%writefile alphafold_fusion/ensemble_variance.py
"""Multi-model ensemble structural variance analysis.

Superposition: Kabsch algorithm for optimal rigid-body alignment
(Kabsch 1976, Acta Cryst A 32:922-923).

AlphaFold/ColabFold generate multiple models per target. This module
superposes those models and reports the per-residue root-mean-square
fluctuation (RMSF) as a descriptor of inter-model structural
variability. Massive sampling of diverse AlphaFold models has been
shown to improve multimer prediction (Wallner 2023, Bioinformatics
39:btad573); here we summarise the variability of the returned
ensemble rather than generating additional samples.

All reported quantities are continuous distributions.
"""

from __future__ import annotations
import os, tempfile
from typing import Optional
import numpy as np
import gemmi
from .structure import is_amino_acid


def _ca_coords(txt: str, fmt: str,
               chain: Optional[str] = None) -> Optional[np.ndarray]:
    """Extract Ca coordinates as (N, 3) array."""
    tmp_path = None
    try:
        suffix = ".cif" if fmt == "cif" else ".pdb"
        tmp = tempfile.NamedTemporaryFile("w", suffix=suffix, delete=False)
        tmp.write(txt); tmp.flush(); tmp.close()
        tmp_path = tmp.name
        st_obj = gemmi.read_structure(tmp_path)
        if len(st_obj) == 0:
            return None
        coords = []
        for ch in st_obj[0]:
            if chain and ch.name != chain:
                continue
            if not any(is_amino_acid(r) for r in ch):
                continue
            for res in ch:
                if not is_amino_acid(res):
                    continue
                ca = None
                for at in res:
                    if at.name == "CA":
                        ca = at
                        break
                if ca is not None:
                    coords.append([ca.pos.x, ca.pos.y, ca.pos.z])
        return np.array(coords) if coords else None
    except Exception:
        return None
    finally:
        if tmp_path:
            try: os.remove(tmp_path)
            except OSError: pass


def kabsch_superpose(mobile: np.ndarray,
                     target: np.ndarray) -> np.ndarray:
    """Optimal rigid-body superposition.

    Reference: Kabsch (1976) Acta Cryst A 32:922-923.
    """
    assert mobile.shape == target.shape
    cm, ct = mobile.mean(axis=0), target.mean(axis=0)
    m, t = mobile - cm, target - ct
    H = m.T @ t
    U, S, Vt = np.linalg.svd(H)
    d = np.linalg.det(Vt.T @ U.T)
    sign = np.diag([1, 1, np.sign(d)])
    R = Vt.T @ sign @ U.T
    return (m @ R.T) + ct


def compute_ensemble_rmsf(
    model_files: list[dict],
    chain: Optional[str] = None,
) -> dict:
    """Per-residue RMSF across AF2 models.

    Returns continuous distributions only, no binary classifications.
    """
    all_coords, model_ids = [], []
    for m in model_files:
        try: txt = open(m["file"]).read()
        except Exception: continue
        c = _ca_coords(txt, m["fmt"], chain)
        if c is not None and len(c) > 0:
            all_coords.append(c)
            model_ids.append(m.get("model_id", "?"))

    if len(all_coords) < 2:
        return {"error": "Requires >= 2 models", "n_models": len(all_coords)}

    n = all_coords[0].shape[0]
    compat = [(c, mid) for c, mid in zip(all_coords, model_ids)
              if c.shape[0] == n]
    if len(compat) < 2:
        return {"error": "Incompatible chain lengths", "n_models": len(compat)}

    coords_list = [c for c, _ in compat]
    used_ids = [mid for _, mid in compat]

    ref = coords_list[0]
    superposed = [ref] + [kabsch_superpose(c, ref) for c in coords_list[1:]]
    stack = np.stack(superposed)

    center = stack.mean(axis=0)
    deviations = stack - center[None, :, :]
    sq_dev = np.sum(deviations ** 2, axis=-1)
    rmsf = np.sqrt(np.mean(sq_dev, axis=0))

    M = len(superposed)
    pairwise = []
    for i in range(M):
        for j in range(i + 1, M):
            diff = superposed[i] - superposed[j]
            rmsd = float(np.sqrt(np.mean(np.sum(diff ** 2, axis=-1))))
            pairwise.append({"model_i": used_ids[i], "model_j": used_ids[j],
                             "rmsd_A": round(rmsd, 3)})

    return {
        "rmsf": [round(float(v), 3) for v in rmsf],
        "mean_rmsf": round(float(np.mean(rmsf)), 3),
        "median_rmsf": round(float(np.median(rmsf)), 3),
        "std_rmsf": round(float(np.std(rmsf)), 3),
        "max_rmsf": round(float(np.max(rmsf)), 3),
        "n_residues": int(n),
        "n_models": len(superposed),
        "models_used": used_ids,
        "pairwise_rmsd": pairwise,
        "superposition_reference": "Kabsch (1976) Acta Cryst A 32:922",
        "rmsf_reference": "McCammon & Harvey (1987) Dynamics of Proteins and Nucleic Acids",
        "complementarity_reference": "Wallner (2023) Bioinformatics 39:btad573",
    }

In [ ]:
%%writefile alphafold_fusion/conservation.py
"""Per-residue evolutionary conservation from MSA (Shannon entropy).

H(i) = -sum_a f_a(i) log2 f_a(i)
Reference: Shannon (1948) Bell Syst Tech J 27:379-423.

Normalised conservation: C(i) = 1 - H(i) / H_max
where H_max = log2(20) for 20 standard amino acids.
Review: Valdar (2002) Proteins 48:227-241.

Sequence weighting corrects for the phylogenetic redundancy of
ColabFold/MMseqs2 MSAs using position-based weights.
Reference: Henikoff & Henikoff (1994) J Mol Biol 243:574-578.

Weights are scaled so that their mean equals 1 (sum = n_sequences),
i.e. on the same scale as integer sequence counts. An optional
Laplace pseudocount (beta, default 0 = pure Shannon/Valdar) can be
added; when beta > 0 it is consistent with the count-scaled weights
(Durbin et al. 1998, Biological Sequence Analysis, Cambridge Univ.
Press).
"""

from __future__ import annotations
import math
from collections import Counter
from typing import Optional
import numpy as np

_AA20 = set("ACDEFGHIKLMNPQRSTVWY")
_K = 20


def _henikoff_weights(columns: list[list[str]], n_seqs: int) -> np.ndarray:
    """Position-based sequence weights.

    Reference: Henikoff & Henikoff (1994) J Mol Biol 243:574-578.
    For each column, each of the k observed residue types contributes
    weight 1/(k * count(type)) to the sequences carrying it.

    Weights are scaled so that their SUM equals n_seqs (mean weight
    = 1), placing them on the same scale as integer sequence counts.
    This keeps any downstream pseudocount consistent with the
    weighted residue counts and avoids over-smoothing shallow MSAs.
    """
    weights = np.zeros(n_seqs, dtype=float)
    for col in columns:
        present = [c for c in col if c in _AA20]
        if len(present) < 2:
            continue
        counts = Counter(present)
        k = len(counts)
        for i, c in enumerate(col):
            if c in _AA20:
                weights[i] += 1.0 / (k * counts[c])
    total = weights.sum()
    if total <= 0:
        # Fallback: uniform weight of 1 per sequence (sum = n_seqs).
        return np.ones(n_seqs, dtype=float)
    # Scale to sum = n_seqs (mean weight = 1), not to sum = 1.
    return weights * (n_seqs / total)


def _weighted_entropy(column: list[str], seq_weights: np.ndarray,
                      beta: float) -> float:
    """Weighted Shannon entropy of one MSA column (bits).

    beta : total pseudocount mass spread over the 20 amino acids
           (alpha = beta / 20 per residue). beta = 0 gives pure
           weighted Shannon entropy (Shannon 1948; Valdar 2002).
    """
    wcounts = {a: 0.0 for a in _AA20}
    wtotal = 0.0
    for i, c in enumerate(column):
        cu = c.upper()
        if cu in _AA20:
            wcounts[cu] += seq_weights[i]
            wtotal += seq_weights[i]
    if wtotal <= 0:
        return 0.0
    alpha = beta / _K            # per-residue pseudocount
    denom = wtotal + beta        # = wtotal + _K * alpha
    h = 0.0
    for a in _AA20:
        f = (wcounts[a] + alpha) / denom
        if f > 0:
            h -= f * math.log2(f)
    return h


def conservation_profile(
    query_seq: str,
    aligned_hits: list[str],
    beta: float = 0.0,
) -> dict:
    """Per-position conservation from a parsed A3M MSA, with
    Henikoff & Henikoff (1994) sequence weighting.

    Parameters
    ----------
    query_seq    : ungapped query sequence (match-state reference)
    aligned_hits : A3M-aligned homolog rows
    beta         : total pseudocount mass. Default 0.0 = pure
                   Shannon/Valdar (no smoothing). Set beta > 0 for
                   Laplace-style smoothing (Durbin et al. 1998).

    Returns continuous per-residue scores only.
    """
    L = len(query_seq)
    all_seqs = [query_seq] + aligned_hits
    n_total = len(all_seqs)

    # Build per-position columns (upper-case = match state in A3M).
    columns: list[list[str]] = [[query_seq[p]] for p in range(L)]
    for hit in aligned_hits:
        qi = 0
        col_chars = ["-"] * L
        for c in hit:
            if qi >= L:
                break
            if c == "-" or c.isupper():
                col_chars[qi] = c
                qi += 1
        for p in range(L):
            columns[p].append(col_chars[p])

    # Henikoff weights (computed once over all columns), scaled to
    # sum = n_total (mean weight = 1).
    seq_weights = _henikoff_weights(columns, n_total)
    sw_sum = float(np.sum(seq_weights))
    sw_sq = float(np.sum(seq_weights ** 2))
    # Effective number of sequences: Neff = (sum w)^2 / sum(w^2).
    n_effective = (sw_sum ** 2 / sw_sq) if sw_sq > 0 else 0.0

    H_max = math.log2(_K)
    conservation, entropy, gap_fracs, depth = [], [], [], []

    for pos in range(L):
        col = columns[pos]
        h = _weighted_entropy(col, seq_weights, beta)
        entropy.append(round(h, 4))
        conservation.append(round(max(0.0, 1.0 - h / H_max), 4))
        n_gap = sum(1 for c in col if c in "-.")
        gap_fracs.append(round(n_gap / len(col), 4))
        depth.append(len(col) - n_gap)

    return {
        "conservation": conservation,
        "entropy": entropy,
        "gap_fraction": gap_fracs,
        "depth": depth,
        "n_sequences": n_total,
        "n_effective": round(n_effective, 2),
        "n_positions": L,
        "beta_pseudocount": beta,
        "mean_conservation": round(float(np.mean(conservation)), 4),
        "weighting": "Henikoff & Henikoff (1994) position-based",
        "entropy_reference": "Shannon (1948) Bell Syst Tech J 27:379",
        "scoring_reference": "Valdar (2002) Proteins 48:227",
        "weighting_reference": "Henikoff & Henikoff (1994) J Mol Biol 243:574",
        "pseudocount_reference": "Durbin et al. (1998) Biological Sequence Analysis",
    }

In [ ]:
%%writefile alphafold_fusion/disprot.py
"""DisProt API client and per-residue comparison with pLDDT-based disorder.

DisProt: Piovesan et al. (2022) Nucleic Acids Res 50:D471-D477.
pLDDT<50 disorder baseline: Akdel et al. (2022) Nat Struct Mol Biol 29:1056.
"""

from __future__ import annotations
import json, urllib.request
from typing import Optional
import numpy as np
from .config import log


def fetch_disprot(acc: str, timeout: int = 30) -> Optional[dict]:
    """Fetch a single DisProt entry by UniProt accession."""
    acc = (acc or "").strip().upper()
    if not acc:
        return None
    url = (f"https://disprot.org/api/search?release=current"
           f"&show_ambiguous=false&format=json&acc={acc}")
    import time as _time
    for attempt in range(3):
        try:
            req = urllib.request.Request(
                url, headers={"Accept": "application/json"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                j = json.loads(r.read().decode("utf-8", "ignore"))
            for e in j.get("data", []):
                if e.get("acc", "").upper() == acc:
                    return e
            data = j.get("data", [])
            return data[0] if data else None
        except Exception as exc:
            log.warning("fetch_disprot attempt %d/3: %s",
                        attempt + 1, exc)
            if attempt < 2:
                _time.sleep(1.5 * (attempt + 1))
    return None


def disprot_regions(entry: Optional[dict]) -> list[tuple[int, int]]:
    """Consensus disordered regions (structural state type 'D').

    DisProt organises consensus into sub-categories. Pure structural
    disorder is annotated as type 'D' under 'Structural state'
    (NOT under 'full', which may report transitions 'T').
    """
    if not entry:
        return []
    cons = entry.get("disprot_consensus") or {}
    regs = []
    # Primary source: "Structural state" with type "D" (pure disorder)
    for r in cons.get("Structural state", []):
        if r.get("type") == "D":
            try:
                regs.append((int(r["start"]), int(r["end"])))
            except (TypeError, ValueError, KeyError):
                pass
    return regs


def compare_plddt_vs_disprot(
    plddt_vals: list[float],
    disprot_regs: list[tuple[int, int]],
    threshold: float = 50.0,
) -> dict:
    """Per-residue comparison of pLDDT<threshold vs DisProt consensus.

    Only positions covered by both are scored. Returns MCC, F1,
    precision, recall, confusion matrix, and per-residue arrays.
    """
    n = len(plddt_vals)
    if n == 0:
        return {"error": "No pLDDT values"}

    truth = np.zeros(n, dtype=int)
    for s, e in disprot_regs:
        truth[max(0, s - 1):min(n, e)] = 1

    pred = np.array([1 if v < threshold else 0 for v in plddt_vals], dtype=int)

    tp = int(np.sum((truth == 1) & (pred == 1)))
    tn = int(np.sum((truth == 0) & (pred == 0)))
    fp = int(np.sum((truth == 0) & (pred == 1)))
    fn = int(np.sum((truth == 1) & (pred == 0)))
    denom = np.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
    mcc = (tp * tn - fp * fn) / denom if denom > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

    return {
        "n_residues": n,
        "threshold": threshold,
        "disprot_disordered": int(truth.sum()),
        "aff_disordered": int(pred.sum()),
        "MCC": round(float(mcc), 3),
        "F1": round(float(f1), 3),
        "precision": round(float(prec), 3),
        "recall": round(float(rec), 3),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "truth_per_residue": truth.tolist(),
        "pred_per_residue": pred.tolist(),
        "n_disprot_regions": len(disprot_regs),
    }

In [ ]:
%%writefile alphafold_fusion/report.py
"""Structured multi-analysis report in JSON and TSV.

FAIR data principles (Wilkinson et al. 2016, Scientific Data 3:160018).
"""

from __future__ import annotations
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Optional


def build_report(
    sequence_name: str,
    sequence: str,
    model_metrics: list[dict],
    plddt_profile: Optional[list[float]] = None,
    disorder: Optional[dict] = None,
    domains: Optional[dict] = None,
    interface: Optional[dict] = None,
    conservation: Optional[dict] = None,
    ensemble: Optional[dict] = None,
    metadata: Optional[dict] = None,
) -> dict:
    report: dict[str, Any] = {
        "_schema": "AlphaFoldFusion_Report_v2.3",
        "_schema_description": "FAIR-compliant (Wilkinson et al. 2016, Sci Data 3:160018)",
        "_timestamp": datetime.now(timezone.utc).isoformat(),
        "sequence_name": sequence_name,
        "sequence_length": len(sequence.replace(":", "")),
        "is_multimer": ":" in sequence,
        "n_chains": sequence.count(":") + 1,
        "models": model_metrics,
    }
    for key, val in [
        ("plddt_profile", plddt_profile), ("disorder", disorder),
        ("domain_decomposition", domains), ("interface", interface),
        ("conservation", conservation), ("ensemble_variance", ensemble),
    ]:
        if val is not None:
            report[key] = val
    if metadata:
        report["metadata"] = metadata
    return report


def save_json(report: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(report, f, indent=2, default=str)


def to_tsv(report: dict) -> str:
    header = ["residue", "pLDDT", "disordered",
              "conservation", "entropy_bits", "gap_fraction", "RMSF_A"]
    lines = ["\t".join(header)]

    plddt = report.get("plddt_profile") or []
    dis = (report.get("disorder") or {}).get("per_residue") or []
    cons = (report.get("conservation") or {}).get("conservation") or []
    ent = (report.get("conservation") or {}).get("entropy") or []
    gf = (report.get("conservation") or {}).get("gap_fraction") or []
    rmsf = (report.get("ensemble_variance") or {}).get("rmsf") or []

    n = max(len(plddt), len(dis), len(cons), len(ent), len(gf), len(rmsf), 1)
    for i in range(n):
        lines.append("\t".join([
            str(i + 1),
            f"{plddt[i]:.1f}" if i < len(plddt) else "",
            str(dis[i]) if i < len(dis) else "",
            f"{cons[i]:.4f}" if i < len(cons) else "",
            f"{ent[i]:.4f}" if i < len(ent) else "",
            f"{gf[i]:.4f}" if i < len(gf) else "",
            f"{rmsf[i]:.3f}" if i < len(rmsf) else "",
        ]))
    return "\n".join(lines)

In [ ]:
%%writefile alphafold_fusion/pages/__init__.py
"""Page registry."""

PAGES: list[str] = [
    "🏠 Home",
    "📊 Predictions",
    "📈 Results",
    "👁️ 3D Viewer",
    "📐 Analysis",
    "✅ Validation",
    "⚙️ Settings",
]

VIEW_MODES: list[str] = [
    "pLDDT (structures)",
    "Identity (alignments)",
]

In [ ]:
%%writefile alphafold_fusion/pages/home.py
"""Home page — about information."""

import streamlit as st


def _metric_card(label, value, unit="", pct=None,
                 quality=None, color="#2563EB"):
    """Scientific metric card with optional confidence bar."""
    bar = ""
    if pct is not None:
        bar = (f'<div style="background:#E2E8F0;border-radius:6px;'
               f'height:8px;margin-top:8px;overflow:hidden">'
               f'<div style="background:{color};width:{pct}%;'
               f'height:100%;border-radius:6px"></div></div>')
    q = (f'<span style="float:right;font-size:.75rem;color:{color};'
         f'font-weight:600">{quality}</span>' if quality else "")
    return (f'<div style="background:#fff;border:1px solid #E2E8F0;'
            f'border-radius:12px;padding:16px 18px;'
            f'box-shadow:0 1px 3px rgba(0,0,0,.06)">'
            f'<div style="font-size:.8rem;color:#64748B;'
            f'font-weight:500">{label}{q}</div>'
            f'<div style="font-size:1.8rem;font-weight:700;'
            f'color:#1E293B;margin-top:4px">{value}'
            f'<span style="font-size:1rem;color:#94A3B8">{unit}'
            f'</span></div>{bar}</div>')


def render() -> None:
    st.markdown(
        '<div class="card"><b>🎯 About</b><br/>'
        'AlphaFold-faithful predictions via ColabFold with rich '
        'visualization: pLDDT coloring, PAE heatmaps, UniProt/InterPro '
        'domain overlays, batch multi-protein analysis, and AFDB-first '
        'optimization. Organism-agnostic.</div>',
        unsafe_allow_html=True,
    )
    st.markdown(
        '<div class="info-card"><b>📋 Input</b><br/>'
        '<b>Monomer</b>: FASTA sequences (1 protein = 1 prediction).<br/>'
        '<b>Multimer</b>: Side A + Side B proteins forming a complex.'
        '</div>',
        unsafe_allow_html=True,
    )
    st.markdown(
        '<div class="info-card" style="border-left-color:#e63946">'
        '<b>⚠️ Security &amp; Data Privacy</b><br/>'
        'This tool has no strong authentication. Sequences you submit '
        'are sent to <b>third-party services</b> (ColabFold MMseqs2 '
        'servers, EBI APIs for UniProt/InterPro/BLAST). '
        '<b>Do not submit confidential or proprietary sequences.</b><br/>'
        'When running via a public tunnel, do not share the URL openly, '
        'and stop the session when finished.'
        '</div>',
        unsafe_allow_html=True,
    )

In [ ]:
%%writefile alphafold_fusion/pages/predictions.py
"""Predictions page — input, parameter configuration, job launch.

FIX v2.1.1:
- 'or' changed to 'and' for complex assembly (both sides required)
- AFDB-first blocked for complexes (AFDB has monomers only)
- Added FASTA preview showing exact content sent to ColabFold
- Validation: complex must contain ':' before launch
- Per-side parsing feedback
"""

from __future__ import annotations
import hashlib, re, shutil, urllib.request
from pathlib import Path
import streamlit as st
from alphafold_fusion.config import (
    CACHE_DIR, MSA_LABELS, RECYCLE_PRESETS, RESULTS_DIR,
)
from alphafold_fusion.sequence import (
    is_complex, new_run_dir, parse_fasta, safe_basename, total_length,
)
from alphafold_fusion.api import fetch_afdb
from alphafold_fusion.structure import cif_to_pdb
from alphafold_fusion.runner import (
    analyze, fallback_cpu, has_gpu, quality_warnings, run_colabfold,
)


def render() -> None:
    ss = st.session_state
    st.markdown(
        '<div class="sub-header">Prediction Setup</div>',
        unsafe_allow_html=True,
    )
    mode = st.radio("Mode", ["Monomer", "Multimer"], horizontal=True,
                    key="pmode")
    valid: list[tuple[str, str]] = []

    # ── Direct AFDB fetch by UniProt accession ──
    with st.expander("🔎 Load existing model from AlphaFold DB (by accession)"):
        acc_in = st.text_input(
            "UniProt accession (e.g. Q9UKV8):",
            key="afdb_acc_input").strip().upper()
        if st.button("Fetch from AFDB", key="afdb_fetch_btn"):
            from alphafold_fusion.api import fetch_afdb, is_uniprot_acc
            import urllib.request as _u
            if not is_uniprot_acc(acc_in):
                st.error(f"'{acc_in}' is not a valid UniProt accession.")
            else:
                with st.spinner(f"Querying AFDB for {acc_in}..."):
                    af = fetch_afdb(acc_in)
                if not af:
                    st.error(
                        f"No AlphaFold model found for {acc_in} in AFDB. "
                        f"The protein may be absent (too long / not modelled). "
                        f"You can still try ColabFold below.")
                else:
                    url = (af.get("cif_url") or af.get("pdb_url")
                           or af.get("bcif_url"))
                    run_root = new_run_dir(RESULTS_DIR)
                    ss["run_root"] = str(run_root)
                    od = run_root / f"AFDB_{acc_in}"
                    od.mkdir(parents=True, exist_ok=True)
                    loc = od / f"{acc_in}_AFDB{Path(url).suffix}"
                    try:
                        _u.urlretrieve(url, loc)
                        if loc.suffix.lower() in (".cif", ".bcif"):
                            cif_to_pdb(loc, loc.with_name(loc.stem + "_c.pdb"))
                        r = analyze(od)
                        if r["status"] == "success":
                            ss.setdefault("results", {})[acc_in] = r
                            ss.setdefault("job_dirs", {})[acc_in] = str(od)
                            ss.setdefault("_afdb_used", {})[acc_in] = af["acc"]
                            best = r["models"][0]
                            st.success(
                                f"✅ AFDB model loaded for {acc_in} "
                                f"(pLDDT ~ {best['avg_plddt'] or 0:.1f}, "
                                f"{r['models'][0].get('fmt','').upper()}). "
                                f"Go to Results / 3D Viewer / Analysis.")
                        else:
                            st.error("Downloaded file could not be parsed.")
                    except Exception as e:
                        st.error(f"Download failed: {e}")

    if mode == "Monomer":
        fa = st.text_area(
            "FASTA:", key="fasta_text", height=200,
            placeholder=">P00533\nMRPSGT...\n\n>MyProt\nMADYK...",
        )
        if fa.strip():
            for n, s in parse_fasta(fa):
                cs = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", s.upper())
                if cs: valid.append((n, cs))
                else: st.warning(f"'{n}': empty after cleaning")
            if valid: st.success(f"{len(valid)} valid sequence(s)")
# ── Identify protein by BLAST (for headers without accession) ──
            from alphafold_fusion.api import extract_uniprot_acc
            no_acc = [(n, s) for n, s in valid
                      if not extract_uniprot_acc(n)]
            if no_acc:
                st.info(f"{len(no_acc)} sequence(s) have no UniProt "
                        f"accession in the header.")
                if st.button("🔍 Identify protein(s) by sequence (BLAST)",
                             key="blast_id"):
                    from alphafold_fusion.api import blast_identify
                    for n, s in no_acc:
                        with st.spinner(f"BLAST search for '{n}' "
                                        f"(~30-60s)..."):
                            hit = blast_identify(s)
                        if hit:
                            acc = hit["accession"]
                            ident = hit["identity_pct"]
                            ss.setdefault("_afdb_used", {})[n] = acc
                            if hit["is_exact"]:
                                st.success(
                                    f"✅ '{n}' = **{acc}** "
                                    f"({hit['description'][:50]}) "
                                    f"— {ident}% identity (exact match).")
                            else:
                                st.warning(
                                    f"⚠️ '{n}': closest match **{acc}** "
                                    f"({hit['description'][:50]}) "
                                    f"— {ident}% identity (homolog, "
                                    f"not identical).")
                        else:
                            st.error(f"'{n}': no BLAST hit found.")
    else:
        # ═══════════════════════════════════════
        # MULTIMER : Side A + Side B
        # ═══════════════════════════════════════
        cname = st.text_input("Complex name:", "MyComplex", key="cxname")

        cA, cB = st.columns(2)
        with cA:
            st.markdown("#### Side A")
            fa_a = st.text_area(
                "Side A (FASTA):", key="fa_a", height=160,
                placeholder=">TNRC6A\nMKDAYPFEKL...",
            )
        with cB:
            st.markdown("#### Side B")
            fa_b = st.text_area(
                "Side B (FASTA):", key="fa_b", height=160,
                placeholder=">AGO2\nMYSGAGPAL...",
            )

        # ── Parse each side independently ──
        clean_a = []
        if fa_a.strip():
            for n, s in parse_fasta(fa_a.strip()):
                cs = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", s.upper())
                if cs:
                    clean_a.append((n, cs))
                    st.caption(f"Side A parsed: **{n}** ({len(cs)} aa)")
            if not clean_a and fa_a.strip():
                st.error("Side A: text found but no valid amino acids extracted.")

        clean_b = []
        if fa_b.strip():
            for n, s in parse_fasta(fa_b.strip()):
                cs = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", s.upper())
                if cs:
                    clean_b.append((n, cs))
                    st.caption(f"Side B parsed: **{n}** ({len(cs)} aa)")
            if not clean_b and fa_b.strip():
                st.error("Side B: text found but no valid amino acids extracted.")

        # ═══════════════════════════════════════
        # FIX 1 : 'and' instead of 'or'
        # Both sides MUST have sequences
        # ═══════════════════════════════════════
        if clean_a and clean_b:
            chains_a = [s for _, s in clean_a]
            chains_b = [s for _, s in clean_b]
            all_chains = chains_a + chains_b
            complex_seq = ":".join(all_chains)
            total = sum(len(c) for c in all_chains)

            # Verify colon is present
            if ":" not in complex_seq:
                st.error("Internal error: assembled sequence has no ':'")
            else:
                valid = [(cname.strip() or "Complex", complex_seq)]
                st.success(
                    f"**{len(clean_a)}** chain(s) Side A + "
                    f"**{len(clean_b)}** chain(s) Side B = "
                    f"**{len(all_chains)} chains** ({total} residues)")

                # ═══════════════════════════════════
                # FIX 4 : Show FASTA preview
                # ═══════════════════════════════════
                with st.expander("Preview: what ColabFold will receive"):
                    names_a = [n for n, _ in clean_a]
                    names_b = [n for n, _ in clean_b]
                    st.markdown(
                        f"- **Side A**: {', '.join(names_a)} "
                        f"({sum(len(s) for s in chains_a)} aa)\n"
                        f"- **Side B**: {', '.join(names_b)} "
                        f"({sum(len(s) for s in chains_b)} aa)\n"
                        f"- **Chain separator** `:` at positions: "
                        f"{[i for i, c in enumerate(complex_seq) if c == ':']}\n"
                        f"- **Model type**: alphafold2_multimer_v3\n"
                        f"- **Total**: {len(all_chains)} chains, {total} residues"
                    )
                    st.code(f">{safe_basename(cname or 'Complex', complex_seq)}\n"
                            f"{complex_seq[:60]}...:{complex_seq.split(':')[-1][:30]}..."
                            if len(complex_seq) > 90 else
                            f">{safe_basename(cname or 'Complex', complex_seq)}\n"
                            f"{complex_seq}")

                if total > 2000:
                    st.warning(f"Large complex ({total} residues). "
                               f"A100 GPU recommended.")

        elif fa_a.strip() or fa_b.strip():
            # One side has text but assembly failed
            if clean_a and not clean_b:
                st.error("**Side B is empty or invalid.** "
                         "Both sides are required for multimer prediction.")
            elif clean_b and not clean_a:
                st.error("**Side A is empty or invalid.** "
                         "Both sides are required for multimer prediction.")
            elif not clean_a and not clean_b:
                st.error("Neither side has valid sequences.")

    # ── Parameters ──
    t1, t2 = st.tabs(["Basic", "Advanced"])
    with t1:
        c1, c2, c3 = st.columns(3)
        with c1:
            quality = st.selectbox("Quality", list(RECYCLE_PRESETS), 0)
            msa_strat = st.selectbox("MSA", list(MSA_LABELS), 1)
        with c2:
            nmod = st.slider("Models", 1, 5, 3)
            nrec = st.slider("Min recycles", 1, 20, 6)
        with c3:
            use_tmpl = st.checkbox("PDB Templates", True)
            use_amber = st.checkbox("AMBER Relax", False)
    with t2:
        c1, c2 = st.columns(2)
        with c1:
            mt_ui = st.selectbox("Model type",
                ["auto", "alphafold2_multimer_v3", "alphafold2_ptm"], 0)
            pm_ui = st.selectbox("Pairing",
                ["auto", "paired", "unpaired", "unpaired_paired"], 0)
        with c2:
            stop = st.number_input(
                "Stop at pLDDT (0=off)", 0.0, 100.0, 0.0, 1.0)
            purge = st.checkbox("Purge old results", False)

    strict = st.checkbox("Strict monomer (no pairing)", False)
    no_homo = st.checkbox("No known homologs (skip templates)", False)
    afdb_fast = st.checkbox(
        "AFDB-first (skip ColabFold if available)", False)
    cache = st.checkbox("Reuse cache", True)

    if st.button("Launch", type="primary", use_container_width=True):
        if not valid:
            st.error("No valid sequences. Check your input above.")
            return
# ══════ GPU backend check (warn on silent CPU fallback) ══════
        from alphafold_fusion.runner import jax_backend, has_gpu
        _backend = jax_backend()
        if _backend == "cpu":
            if has_gpu():
                st.error(
                    "⚠️ **GPU present but NOT visible to JAX!**\n\n"
                    "Predictions will run on **CPU** (10-50× slower, "
                    "**severely degraded quality** — pLDDT typically "
                    "<50 even for well-folded proteins).\n\n"
                    "**Fix:** Runtime → Restart session, then re-run the "
                    "installation cell (JAX GPU).\n\n"
                    "You may continue, but CPU results are **not "
                    "reliable**.")
            else:
                st.warning(
                    "⚠️ **No GPU detected** — running on CPU (slow, "
                    "degraded). To enable GPU: Runtime → Change runtime "
                    "type → GPU (T4).")
        elif _backend == "gpu":
            st.caption("✅ GPU active (JAX backend: gpu)")
        # ═══════════════════════════════════════════════════════════
        # ═══════════════════════════════════════
        # FIX 2 : Block launch if multimer has no ':'
        # ═══════════════════════════════════════
        if mode == "Multimer":
            seq_check = valid[0][1] if valid else ""
            if not is_complex(seq_check):
                st.error(
                    "**Cannot launch**: assembled sequence has no chain "
                    "separator. Both Side A and Side B must have valid "
                    "sequences.\n\n"
                    f"Current sequence starts with: `{seq_check[:40]}...`")
                return
            n_ch = seq_check.count(":") + 1
            st.info(f"Launching multimer: **{n_ch} chains**, "
                    f"**{total_length(seq_check)} residues**")

        if purge:
            shutil.rmtree(RESULTS_DIR, ignore_errors=True)
            RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        run_root = new_run_dir(RESULTS_DIR)
        ss["run_root"] = str(run_root)
        ss.results = {}; ss.job_dirs = {}; ss.fasta_paths = {}
        prog = st.progress(0); info = st.empty()
        mkey = MSA_LABELS.get(msa_strat, "fast")
        _gpu = has_gpu()

        for i, (name, seq) in enumerate(valid):
            safe = safe_basename(name, seq)
            fa_path = str(Path("/tmp") / f"{safe}.fasta")

            # Write FASTA
            with open(fa_path, "w") as fout:
                fout.write(f">{safe}\n{seq}\n")

            cplx = is_complex(seq)
            slen = total_length(seq)

            # ═══════════════════════════════════════
            # FIX 3 : Verify FASTA content after write
            # ═══════════════════════════════════════
            with open(fa_path) as f:
                written = f.read()
            if cplx and ":" not in written:
                st.error(f"FASTA file for '{name}' has no ':' separator. "
                         f"Something went wrong during file writing.")
                continue

            st.caption(
                f"FASTA: complex={cplx} | "
                f"chains={seq.count(':') + 1 if cplx else 1} | "
                f"{slen} residues | "
                f"model={'multimer_v3' if cplx else 'ptm'}")

            # ═══════════════════════════════════════
            # FIX 5 : AFDB-first ONLY for monomers
            # AFDB has no multimer structures
            # ═══════════════════════════════════════
            if afdb_fast and cplx:
                st.info(f"AFDB-first skipped: '{name}' is a complex "
                        f"(AFDB has monomers only)")
            elif afdb_fast and not cplx:
                if try_afdb(name, seq, safe, run_root, fa_path, ss):
                    prog.progress((i + 1) / len(valid)); continue

            pr = RECYCLE_PRESETS.get(quality, RECYCLE_PRESETS["Classic"])
            nrec_eff = pr["complex"] if cplx else pr["monomer"]
            nrec_eff = max(nrec_eff, nrec)
            if cplx:
                mtype = "alphafold2_multimer_v3" if mt_ui == "auto" else mt_ui
                pmode = "paired" if pm_ui == "auto" else pm_ui
                if mkey == "minimal" and pmode == "paired":
                    pmode = "unpaired"
                # Warn if user forced monomer model on complex
                if mtype == "alphafold2_ptm":
                    st.warning(
                        f"'{mtype}' is a monomer model but input is a "
                        f"complex with {seq.count(':') + 1} chains. "
                        f"Switching to alphafold2_multimer_v3.")
                    mtype = "alphafold2_multimer_v3"
            else:
                mtype = "alphafold2_ptm" if mt_ui == "auto" else mt_ui
                pmode = ("unpaired" if strict
                         else ("unpaired_paired" if pm_ui == "auto" else pm_ui))

            params = {
                "num_models": nmod, "num_recycles": nrec_eff,
                "use_amber": use_amber,
                "use_templates": use_tmpl and not no_homo,
                "model_type": mtype, "pair_mode": pmode,
                "stop_at_score": stop if stop > 0 else None,
                "msa_strategy": mkey, "jobname_prefix": safe,
                "reuse_cache": cache,
            }

            info.text(
                f"Running: {name} | {mtype} | {mkey} | "
                f"{nrec_eff} recycles | {pmode}"
                + (f" | {seq.count(':') + 1} chains | {slen} aa"
                   if cplx else f" | {slen} aa"))

            sh = hashlib.sha1(seq.encode()).hexdigest()[:10]
            ck = "".join([safe, sh, mkey, mtype, pmode,
                          f"{nmod}m", f"{nrec_eff}r",
                          "t" if params["use_templates"] else "n"])
            od = CACHE_DIR / ck if cache else run_root / f"{safe}_{sh}"
            od.mkdir(parents=True, exist_ok=True)

            ok, run_log = run_colabfold(fa_path, str(od), params, slen)
            r = analyze(od)

            if r["status"] != "success":
                if not _gpu:
                    st.info("No GPU — trying CPU fallback...")
                _, run_log = fallback_cpu(fa_path, str(od), safe, cplx, slen)
                r = analyze(od)
                # AFDB fallback ONLY for monomers
                if r["status"] != "success" and not cplx:
                    _try_afdb_fallback(name, safe, od, r)
                if r["status"] != "success" and cplx:
                    st.error(
                        f"Complex '{name}' failed "
                        f"({slen} residues, {seq.count(':') + 1} chains).\n\n"
                        f"Try: fewer models, 'Fast' quality, or A100 GPU.")

            ss.results[name] = r
            ss.job_dirs[name] = str(od)
            ss.fasta_paths[name] = fa_path
            _report(name, r, run_log, job_dir=od)
            prog.progress((i + 1) / len(valid))
        st.success("Done — see Results / 3D Viewer")


def try_afdb(name, seq, safe, run_root, fa_path, ss) -> bool:
    try:
        af = fetch_afdb(name)
        if not af: return False
        url = af.get("pdb_url") or af.get("cif_url") or af.get("bcif_url")
        if not url: return False
        od = run_root / f"{safe}_{hashlib.sha1(seq.encode()).hexdigest()[:8]}"
        od.mkdir(parents=True, exist_ok=True)
        loc = od / (safe + "_AFDB" + Path(url).suffix)
        if not loc.exists(): urllib.request.urlretrieve(url, loc)
        ss.setdefault("_afdb_used", {})[name] = af["acc"]
        if loc.suffix.lower() in (".cif", ".bcif"):
            try: cif_to_pdb(loc, loc.with_name(loc.stem + "_c.pdb"))
            except Exception: pass
        from alphafold_fusion.runner import analyze
        r = analyze(od)
        if r["status"] == "success":
            ss.job_dirs[name] = str(od)
            ss.fasta_paths[name] = fa_path
            ss.results[name] = r
            st.success(f"AFDB structure used for {name}"); return True
    except Exception as e:
        st.info(f"AFDB: {e}")
    return False


def _try_afdb_fallback(name, safe, od, r):
    try:
        af2 = fetch_afdb(name)
        if af2:
            u2 = af2.get("pdb_url") or af2.get("cif_url")
            if u2:
                l2 = od / (safe + "_AFDB2" + Path(u2).suffix)
                if not l2.exists(): urllib.request.urlretrieve(u2, l2)
                if l2.suffix.lower() in (".cif", ".bcif"):
                    try: cif_to_pdb(l2, l2.with_name(l2.stem + "_c.pdb"))
                    except Exception: pass
                from alphafold_fusion.runner import analyze
                r.update(analyze(od))
    except Exception: pass


def _report(name, r, run_log, job_dir=None):
    if r["status"] == "success":
        best = sorted(
            r["models"],
            key=lambda m: (m["rank"] if m["rank"] is not None else 9999,
                           -(m["avg_plddt"] or 0)),
        )
        if best:
            b = best[0]
            st.success(f"{name}: {b['model_id']} | "
                       f"pLDDT~{b['avg_plddt'] or 0:.1f}")
            for w in quality_warnings(b): st.warning(w)
            # ══════ MSA empty check ══════
            if job_dir is not None:
                from alphafold_fusion.runner import (
                    count_msa_sequences, msa_warning)
                n_msa = count_msa_sequences(Path(job_dir))
                mw = msa_warning(n_msa, b.get("avg_plddt"))
                if mw:
                    st.error(mw)
                elif n_msa is not None:
                    st.caption(f"MSA depth: {n_msa} sequences")
        else:
            st.success(f"{name}: structure detected")
    else:
        tail = "\n".join((run_log or "").splitlines()[-20:])
        st.error(f"{name}: {r['message']}\n\n{tail}")

In [ ]:
%%writefile alphafold_fusion/pages/results.py
"""Results page — model comparison table, inspection, alignment comparator."""

from __future__ import annotations
from pathlib import Path
import numpy as np, pandas as pd
import plotly.graph_objects as go
import streamlit as st
from alphafold_fusion.alignment import build_a3m_data, find_a3m, parse_text_block
from alphafold_fusion.api import enrich_df_urls
from alphafold_fusion.pae import pae_from_pkl, pae_heatmap
from alphafold_fusion.plddt import plddt_profile, show_plddt_panels
from alphafold_fusion.render import patch_cdn, render_3d, show_local_png
from alphafold_fusion.structure import first_model_only, polymer_chains


def render() -> None:
    ss = st.session_state
    st.markdown('<div class="sub-header">Results</div>',
                unsafe_allow_html=True)
    vmode = ss.get("view_mode", "pLDDT (structures)")
    if vmode.startswith("pLDDT"): render_plddt_mode(ss)
    else: render_identity_mode(ss)


def render_plddt_mode(ss) -> None:
    if not ss.get("results"):
        st.info("No results yet. Run predictions first."); return
    st.markdown("### 🏆 Model Comparator")
    rows = []
    for n, r in ss.results.items():
        if r.get("status") != "success": continue
        for m in r["models"]:
            rows.append({
                "Sequence": n, "Model": m["model_id"],
                "Rank": m["rank"], "Avg_pLDDT": m["avg_plddt"],
                "PTM": m["ptm"], "ipTM": m["iptm"],
                "Fmt": m["fmt"].upper(), "File": m["file"],
            })
    if not rows: st.warning("No models found."); return

    # ══════ Best model summary cards ══════
    from alphafold_fusion.render import metric_card, confidence_style
    best_row = min(rows, key=lambda r: (
        r["Rank"] if r["Rank"] is not None else 9999,
        -(r["Avg_pLDDT"] or 0)))
    _model_label = (f" · {best_row['Model']}"
                    if best_row['Model'] not in ("-", None, "")
                    else " · AFDB")
    st.markdown(f"#### 🏆 Best: {best_row['Sequence']}{_model_label}")
    bc = st.columns(4)
    _p, _q, _c = confidence_style(best_row["Avg_pLDDT"], "plddt")
    bc[0].markdown(metric_card("Avg pLDDT",
        f"{best_row['Avg_pLDDT'] or 0:.1f}", pct=_p,
        quality=_q, color=_c), unsafe_allow_html=True)
    if best_row["PTM"] is not None:
        _p, _q, _c = confidence_style(best_row["PTM"], "score")
        bc[1].markdown(metric_card("pTM",
            f"{best_row['PTM']:.2f}", pct=_p,
            quality=_q, color=_c), unsafe_allow_html=True)
    else:
        bc[1].markdown(metric_card("pTM", "—", color="#94A3B8"),
                       unsafe_allow_html=True)
    if best_row["ipTM"] is not None:
        _p, _q, _c = confidence_style(best_row["ipTM"], "score")
        bc[2].markdown(metric_card("ipTM",
            f"{best_row['ipTM']:.2f}", pct=_p,
            quality=_q, color=_c), unsafe_allow_html=True)
    else:
        bc[2].markdown(metric_card("ipTM", "— (monomer)",
            color="#94A3B8"), unsafe_allow_html=True)
    bc[3].markdown(metric_card("Format",
        f"{best_row['Fmt']}", color="#0EA5E9"),
        unsafe_allow_html=True)
    st.write("")
    # ═══════════════════════════════════════════
    df = pd.DataFrame(rows).sort_values(
        ["Sequence", "Rank", "Avg_pLDDT"],
        ascending=[True, True, False], na_position="last")
    st.dataframe(df, use_container_width=True, height=300)

    st.markdown("### 👁️ Inspect Model")
    idx = st.selectbox(
        "Select:", range(len(df)),
        format_func=lambda i: (
            f"{df.iloc[i]['Sequence']} • {df.iloc[i]['Model']} • "
            f"pLDDT~{df.iloc[i]['Avg_pLDDT'] or 0:.1f}"),
        key="r_sel")
    rec = df.iloc[idx]; p = rec["File"]
    fm = "cif" if p.lower().endswith((".cif", ".bcif")) else "pdb"
    try: txt = open(p).read()
    except Exception as e: st.error(f"Error: {e}"); return

    cA, cB, cC, cD = st.columns(4)
    with cA:
        sty = st.selectbox("Style",
            ["Cartoon", "Stick", "Sphere", "Line", "Surface"], key="rs")
    with cB:
        sch = st.selectbox("Color",
            ["AlphaFold (4-color)", "Special (blue/orange)",
             "pLDDT (B-factor)", "Spectrum", "Chain"], key="rc")
    with cC: mono = st.checkbox("Monomer", True, key="rm")
    with cD: f1 = st.checkbox("1st model", True, key="rf")

    chs = polymer_chains(p)
    sc = st.selectbox("Chain", chs or ["A"], key="rch") if mono else None
    if f1: txt = first_model_only(fm, txt)
    html = patch_cdn(render_3d(txt, fm, sty, sch, mono, sc))
    st.components.v1.html(html, height=680)
    st.download_button("📥 Download", txt, Path(p).name,
        "chemical/x-mmcif" if fm == "cif" else "chemical/x-pdb",
        use_container_width=True)
    show_plddt_panels(txt, fm, p, sc if mono else None, "rp")

    # ── MSA coverage plot (ColabFold) ──
    _cov = (ss.results.get(rec["Sequence"]) or {}).get("coverage_png")
    if _cov and Path(_cov).exists():
        with st.expander("📊 MSA Coverage (ColabFold)", expanded=False):
            if not show_local_png(_cov, "Sequence coverage of the MSA "
                                  "(depth per residue position)."):
                st.info("Coverage image unavailable.")

    if st.button("Show top-3 pLDDT profiles & PAE", key="t3"):
        show_top3(rec["Sequence"], ss)


def show_top3(seq_name: str, ss) -> None:
    rs = ss.results.get(seq_name) or {}
    ms = sorted(rs.get("models") or [],
                key=lambda x: (x["rank"] if x["rank"] is not None else 9999,
                               -(x["avg_plddt"] or 0)))[:3]
    if not ms: st.info("No models to compare."); return
    fig = go.Figure()
    for mr in ms:
        mf = "cif" if mr["file"].lower().endswith((".cif", ".bcif")) else "pdb"
        _, prof = plddt_profile(mr["file"], mf)
        if prof:
            fig.add_trace(go.Scatter(
                x=list(range(1, len(prof) + 1)), y=prof, mode="lines",
                name=f"{mr['model_id']} (r={mr['rank'] or '?'})"))
    if fig.data:
        fig.update_layout(title=f"Per-residue pLDDT — {seq_name}",
                          xaxis_title="Residue", yaxis_title="pLDDT",
                          yaxis_range=[0, 100],
                          margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)
    else: st.info("pLDDT profiles unavailable.")

    jd = Path(ss.job_dirs.get(seq_name, ""))
    if not jd.exists(): return
    tabs = st.tabs([f"{j+1}. {mr['model_id']}" for j, mr in enumerate(ms)])
    for tab, mr in zip(tabs, ms):
        with tab:
            pm = pae_from_pkl(jd, mr["model_id"])
            if pm is not None:
                f_pae = pae_heatmap(pm, f"PAE — {mr['model_id']}")
                if f_pae: st.plotly_chart(f_pae, use_container_width=True)
            else:
                pp = rs.get("pae_png")
                if not (pp and show_local_png(
                        pp, f"PAE — {mr['model_id']} (ColabFold)")):
                    st.info("No PAE available.")


def render_identity_mode(ss) -> None:
    st.markdown("### 🧬 Alignment Comparator")
    with st.expander("📥 Manual import", expanded=not ss.get("aln_import")):
        ti = st.text_area("Paste block:", height=140, key="ti")
        if st.button("Analyze", key="ab", use_container_width=True):
            d = parse_text_block(ti)
            if d and d.get("hits"):
                ss.setdefault("aln_import", {})[d["name"]] = d
                st.success(f"✅ {d['name']}: {len(d['hits'])} hits")
            else: st.error("Unrecognized format.")

    with st.expander("⚙️ Auto import (.a3m)"):
        jds = list(ss.get("job_dirs", {}).keys())
        if jds:
            sel = st.selectbox("Sequence:", jds, key="as_sel")
            if st.button("Build from .a3m", key="ba",
                         use_container_width=True):
                jd = Path(ss.job_dirs.get(sel, ""))
                if jd.exists():
                    is_afdb = sel in ss.get("_afdb_used", {})
                    if is_afdb:
                        st.info(
                            "ℹ️ **This structure was fetched from AlphaFold DB** "
                            "— no MSA was generated.\n\n"
                            "To get alignment data, re-run this protein "
                            "with **AFDB-first unchecked** so ColabFold "
                            "runs a full MSA search."
                        )
                    else:
                        with st.spinner("Scanning for .a3m files..."):
                            ds = build_a3m_data(sel, jd)
                        if ds and ds.get("hits"):
                            ss.setdefault("aln_import", {})[ds["name"]] = ds
                            from alphafold_fusion.alignment import find_a3m
                            a3m_files = find_a3m(jd)
                            if a3m_files:
                                try:
                                    from alphafold_fusion.alignment import _read_a3m
                                    raw_a3m = _read_a3m(a3m_files[0])
                                    ss.setdefault("_a3m_raw", {})[sel] = raw_a3m
                                except Exception:
                                    pass
                            st.success(
                                f"✅ {ds['name']}: {len(ds['hits'])} hits")
                            try: st.rerun()
                            except Exception: pass
                        else:
                            a3m_files = find_a3m(jd)
                            if a3m_files:
                                st.warning(
                                    f"⚠️ Found {len(a3m_files)} .a3m file(s) "
                                    f"but could not extract hits.\n\n"
                                    f"Files: {', '.join(f.name for f in a3m_files[:5])}\n\n"
                                    f"The .a3m may contain only the query "
                                    f"sequence (no homologs found by MMseqs2)."
                                )
                            else:
                                try:
                                    all_f = [f for f in jd.rglob("*")
                                             if f.is_file()]
                                    exts = sorted(set(
                                        f.suffix for f in all_f)) or ["(none)"]
                                except Exception:
                                    all_f = []; exts = ["?"]
                                st.warning(
                                    f"❌ No .a3m files in `{jd.name}/`\n\n"
                                    f"**{len(all_f)} file(s)** found — "
                                    f"extensions: {', '.join(exts)}\n\n"
                                    f"This can happen if:\n"
                                    f"- The MSA search was skipped "
                                    f"(single_sequence mode)\n"
                                    f"- ColabFold stored MSA data in .pkl "
                                    f"format only\n"
                                    f"- The prediction failed before MSA "
                                    f"generation"
                                )
                else:
                    st.error(f"Directory not found: `{jd}`")
        else:
            st.info("No jobs available.")

    rows = []
    for n, d in (ss.get("aln_import") or {}).items():
        for h in d.get("hits", []):
            rows.append({
                "Sequence": n, "Accession": h.get("acc"),
                "Identity%": h.get("Identity_pct"),
                "Score": h.get("Score"), "Evalue": h.get("Evalue"),
                "Cov%": h.get("Coverage_pct"), "Core": h.get("Core"),
            })
    dfh = (pd.DataFrame(rows) if rows
           else pd.DataFrame(columns=["Sequence", "Accession", "Identity%",
                                      "Score", "Evalue", "Cov%", "Core"]))
    if dfh.empty: st.info("No alignments loaded yet."); return

    c1, c2 = st.columns(2)
    with c1: nrows = st.slider("Top N", 50, 2000, min(300, len(dfh)), 50)
    with c2: sort = st.selectbox("Sort",
        ["Identity%", "Score", "Evalue", "Cov%"], 0)
    asc = (sort == "Evalue")
    dfv = (dfh.sort_values([sort, "Identity%"],
                           ascending=[asc, False], na_position="last")
           .head(nrows).reset_index(drop=True))

    # Download raw .a3m alignment
    a3m_cache = ss.get("_a3m_raw", {})
    if a3m_cache:
        for name, raw in a3m_cache.items():
            st.download_button(
                f"📥 Download MSA (.a3m) — {name}",
                raw, f"{name}_msa.a3m", "text/plain",
                key=f"dl_a3m_{name}")
    if st.checkbox("Enrich UniProt/AFDB URLs", False):
        dfv = enrich_df_urls(dfv)
    try:
        st.dataframe(dfv, use_container_width=True, height=340,
                     column_config={
                         "UniProt_URL": st.column_config.LinkColumn("UniProt"),
                         "AFDB_URL": st.column_config.LinkColumn("AFDB"),
                     })
    except Exception:
        st.dataframe(dfv, use_container_width=True, height=340)

In [ ]:
%%writefile alphafold_fusion/pages/viewer.py
"""3-D Viewer page — interactive exploration with domain overlays.

Domain sources: UniProt features and InterPro entries (database
annotations mapped onto the predicted structure).

Disorder overlay: Akdel et al. (2022) Nat Struct Mol Biol 29:1056.
"""

from __future__ import annotations
from pathlib import Path
import numpy as np
import streamlit as st
from alphafold_fusion.api import (
    fetch_interpro, fetch_uniprot, guess_acc,
    interpro_domains, uniprot_domains,
)
from alphafold_fusion.config import log
from alphafold_fusion.pae import pae_from_pkl, pae_heatmap
from alphafold_fusion.plddt import plddt_by_chain, show_plddt_panels
from alphafold_fusion.render import (
    domain_legend, patch_cdn, render_3d,
    render_domains_3d, viewer_key,
)
from alphafold_fusion.runner import model_name_for_path
from alphafold_fusion.structure import first_model_only, polymer_chains


def render() -> None:
    ss = st.session_state
    st.markdown('<div class="sub-header">3D Viewer</div>',
                unsafe_allow_html=True)
    if not ss.get("results"):
        st.info("No structures. Run predictions first."); return

    choices: list[tuple[str, dict]] = []
    for n, r in ss.results.items():
        if r.get("status") != "success": continue
        for m in r["models"]:
            label = (f"{n} | {m['model_id']} | "
                     f"pLDDT {m['avg_plddt'] or 0:.1f}")
            choices.append((label, m))
    if not choices: st.warning("No PDB/CIF found."); return

    idx = st.selectbox("Model:", range(len(choices)),
                       format_func=lambda i: choices[i][0], key="v_sel")
    m = choices[idx][1]; p = m["file"]
    fm = "cif" if p.lower().endswith((".cif", ".bcif")) else "pdb"
    try:
        with open(p) as f:
            txt = f.read()
    except Exception as e:
        st.error(f"Cannot read structure file: {e}")
        return

    c1, c2, c3, c4 = st.columns(4)
    with c1:
        sty = st.selectbox("Style",
            ["Cartoon", "Stick", "Sphere", "Line", "Surface"], key="vs")
    with c2:
        sch = st.selectbox("Color",
            ["AlphaFold (4-color)", "Special (blue/orange)",
             "pLDDT (B-factor)", "Spectrum", "Chain"], key="vc")
    with c3: mono = st.checkbox("Monomer", True, key="vm")
    with c4: f1 = st.checkbox("1st model", True, key="vf")

    chs = polymer_chains(p)
    sc = st.selectbox("Chain", chs or ["A"], key="vch") if mono else None
    if f1: txt = first_model_only(fm, txt)

    html = patch_cdn(render_3d(txt, fm, sty, sch, mono, sc))
    stamp = viewer_key("v", p=p, s=sty, c=sch, m=mono, ch=sc)
    st.components.v1.html(f"<!-- {stamp} -->\n{html}", height=680)
    st.download_button("Download structure", txt, Path(p).name,
        "chemical/x-mmcif" if fm == "cif" else "chemical/x-pdb",
        use_container_width=True)
    show_plddt_panels(txt, fm, p, sc if mono else None, "vp")

    _render_domains(txt, fm, sty, mono, sc, p, ss)
    _render_disorder_overlay(txt, fm, mono, sc, ss)
    _render_pae(p, ss)


def _render_domains(txt, fm, sty, mono, sc, path, ss) -> None:
    st.subheader("Domain Annotation")
    all_predicted = [n for n, r in (ss.results or {}).items()
                     if (r or {}).get("status") == "success"]
    sg = model_name_for_path(path, ss.results) or ""
    if all_predicted and len(all_predicted) > 1:
        dp = st.selectbox("Protein:", all_predicted,
                          index=(all_predicted.index(sg)
                                 if sg in all_predicted else 0),
                          key="vdp")
    else:
        dp = sg or (all_predicted[0] if all_predicted else "")

    afdb_used = ss.get("_afdb_used", {})
    aa = guess_acc(dp, afdb_used) if dp else None

    cS, cA = st.columns([1, 3])
    with cS:
        src = st.radio("Source", ["UniProt", "InterPro"],
                       horizontal=True, key="vds")
    with cA:
        if src != "PAE domains":
            ua = st.text_input("UniProt Accession", value=aa or "", key="vua")
            auto = st.checkbox("Auto-generate", True, key="vag")
        else:
            ua = ""; auto = False

    do_gen = st.button("Generate", key="vgd")
    doit = do_gen or (auto and (ua or aa))

    if doit:
        acc = ua or aa
        if not acc:
            st.info("Enter UniProt Accession manually."); return
        if src == "UniProt":
            with st.spinner(f"Fetching UniProt domains for {acc}..."):
                j = fetch_uniprot(acc)
                segs = uniprot_domains(j) if j else []
        else:
            with st.spinner(f"Fetching InterPro domains for {acc}..."):
                j = fetch_interpro(acc)
                segs = interpro_domains(j) if j else []
        if segs:
            hd = patch_cdn(render_domains_3d(
                txt, fm, segs, sty, sc if mono else None))
            st.components.v1.html(hd, height=680)
            st.markdown(domain_legend(segs), unsafe_allow_html=True)
        else:
            st.info(f"No domain found via {src} ({acc}).")

def _render_disorder_overlay(txt, fm, mono, sc, ss) -> None:
    """Disorder overlay: pLDDT < 50 (AF2 very low confidence band,
    Jumper et al. 2021; Tunyasuvunakool et al. 2021). Disorder
    principle: Akdel et al. (2022) NSMB 29:1056."""
    with st.expander("Disorder Overlay (pLDDT < 50 = AF2 very low "
                     "confidence; disorder principle: Akdel et al. 2022)"):
        from alphafold_fusion.disorder import disorder_from_structure
        import py3Dmol

        dis = disorder_from_structure(txt, fm, sc)
        if not any(d.get("regions") for d in dis.values()):
            st.info("No disordered regions detected (all pLDDT >= 50).")
            return

        for ch_id, d in dis.items():
            if d.get("regions"):
                st.markdown(f"**Chain {ch_id}**: "
                            f"{d['fraction_disordered']:.1%} disordered")

        v = py3Dmol.view(width=1000, height=650)
        v.addModel(txt, "cif" if fm == "cif" else "pdb")
        v.setStyle({}, {"cartoon": {"color": "#BBBBBB"}})
        pc = plddt_by_chain(txt, fm, sc)
        for ch_id, vals in pc.items():
            ordered = [i + 1 for i, val in enumerate(vals) if val >= 50]
            if ordered:
                sel = {"resi": ordered}
                if mono and sc: sel["chain"] = sc
                v.setStyle(sel, {"cartoon": {"color": "#4361ee"}})
        for ch_id, d in dis.items():
            for r in d.get("regions", []):
                rng = list(range(r["start"], r["end"] + 1))
                sel = {"resi": rng}
                if mono and sc: sel["chain"] = sc
                v.setStyle(sel, {"cartoon": {"color": "#FF8C42",
                                             "opacity": 0.85}})
        v.setBackgroundColor("white"); v.zoomTo()
        st.components.v1.html(patch_cdn(v._make_html()), height=680)
def _render_pae(path, ss) -> None:
    sn = model_name_for_path(path, ss.results)
    if not sn: return
    rs = ss.results.get(sn) or {}
    pj = rs.get("pae_json"); pp = rs.get("pae_png")
    if not (pj or pp): return
    with st.expander("PAE Heatmap"):
        if pj:
            f_pae = pae_heatmap(pj, f"PAE — {sn}")
            if f_pae: st.plotly_chart(f_pae, use_container_width=True)
            else: st.info("PAE JSON unreadable.")
        elif pp:
            st.image(pp, use_container_width=True)


def _render_coverage(path, ss) -> None:
    from alphafold_fusion.runner import model_name_for_path
    sn = model_name_for_path(path, ss.results)
    if not sn:
        return
    cov = (ss.results.get(sn) or {}).get("coverage_png")
    if cov and Path(cov).exists():
        with st.expander("MSA Coverage"):
            st.image(cov, use_container_width=True,
                     caption="MSA coverage (ColabFold).")

In [ ]:
%%writefile alphafold_fusion/pages/analysis.py
"""Analysis page — integrated post-prediction structural analysis.

Four analysis modules grounded in published methods:

1. Disorder: Akdel et al. (2022) Nat Struct Mol Biol 29:1056
2. Inter-chain contacts: Duarte et al. (2012) BMC Bioinformatics 13:334
3. Ensemble variance: Kabsch (1976) + Wallner (2023)
4. Conservation: Shannon (1948) + Valdar (2002)

Each module reports continuous distributions where applicable.
"""

from __future__ import annotations
import json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import streamlit as st

from alphafold_fusion.config import log
from alphafold_fusion.plddt import plddt_by_chain
from alphafold_fusion.pae import pae_from_pkl
from alphafold_fusion.alignment import build_a3m_data
from alphafold_fusion.structure import polymer_chains


def render() -> None:
    ss = st.session_state
    st.markdown('<div class="sub-header">Post-Prediction Analysis</div>',
                unsafe_allow_html=True)

    if not ss.get("results"):
        st.info("No predictions available. Run predictions first.")
        return

    successful = {n: r for n, r in ss.results.items()
                  if r.get("status") == "success"}
    if not successful:
        st.warning("No successful predictions to analyse.")
        return

    seq_name = st.selectbox("Select prediction:", list(successful.keys()),
                            key="analysis_seq")
    result = successful[seq_name]
    models = result.get("models", [])
    if not models:
        st.warning("No models found.")
        return

    best = sorted(models, key=lambda m: (
        m["rank"] if m["rank"] is not None else 9999,
        -(m["avg_plddt"] or 0)))[0]
    try:
        txt = open(best["file"]).read()
    except Exception as e:
        st.error(f"Cannot read structure: {e}")
        return

    fm = "cif" if best["file"].lower().endswith((".cif", ".bcif")) else "pdb"
    chs = polymer_chains(best["file"])
    sel_chain = chs[0] if chs else "A"
    jd = Path(ss.get("job_dirs", {}).get(seq_name, ""))
    pc = plddt_by_chain(txt, fm, sel_chain)
    plddt_vals = [v for arr in pc.values() for v in arr]
    if plddt_vals:
        ss[f"_plddt_profile_{seq_name}"] = plddt_vals

    # ═══════════════════════════════════════════════════════════════
    # 1. INTRINSIC DISORDER
    # Akdel et al. (2022) Nat Struct Mol Biol 29:1056-1067
    # Piovesan et al. (2022) NAR 50:D471-D477 (DisProt guidelines)
    # ═══════════════════════════════════════════════════════════════
    st.markdown("### 🌊 Intrinsic Disorder Regions")
    st.caption("pLDDT < 50 = AF2 very low confidence band (Jumper 2021; "
               "Tunyasuvunakool 2021); disorder principle Akdel et al. "
               "(2022) *NSMB* 29:1056 | DisProt: Piovesan et al. (2022) "
               "*NAR* 50:D471")

    if plddt_vals:
        from alphafold_fusion.disorder import (
            disorder_from_structure, to_gff3)

        dis_result = disorder_from_structure(txt, fm, sel_chain)
        for ch_id, dis in dis_result.items():
            st.markdown(f"**Chain {ch_id}**: "
                        f"{dis['fraction_disordered']:.1%} disordered, "
                        f"{dis['n_idr_regions']} region(s)")
            if dis["regions"]:
                st.dataframe(pd.DataFrame(dis["regions"]),
                             use_container_width=True, height=180)

                fig_dis = go.Figure()
                fig_dis.add_trace(go.Scatter(
                    x=list(range(1, dis["n_residues"] + 1)),
                    y=plddt_vals[:dis["n_residues"]],
                    mode="lines", name="pLDDT",
                    line=dict(color="#457b9d")))
                fig_dis.add_hline(
                    y=dis["threshold_used"], line_dash="dash",
                    line_color="red",
                    annotation_text=f"Threshold {dis['threshold_used']} "
                                    f"(Akdel et al. 2022)")
                for r in dis["regions"]:
                    fig_dis.add_vrect(
                        x0=r["start"], x1=r["end"],
                        fillcolor="rgba(255,127,14,0.2)", line_width=0)
                fig_dis.update_layout(
                    title=f"Disorder — Chain {ch_id}",
                    xaxis_title="Residue", yaxis_title="pLDDT",
                    yaxis_range=[0, 100],
                    margin=dict(l=10, r=10, t=40, b=10))
                st.plotly_chart(fig_dis, use_container_width=True)

                gff3 = to_gff3(dis, seqid=seq_name, chain=ch_id)
                st.download_button(
                    f"Download GFF3 (Chain {ch_id})", gff3,
                    f"{seq_name}_chain{ch_id}_disorder.gff3",
                    "text/plain", key=f"gff3_{ch_id}")

        ss[f"_disorder_{seq_name}"] = dis_result

    # ═══════════════════════════════════════════════════════════════
    # 2. INTER-CHAIN CONTACT MAP (multimers only)
    # Cα-Cα 8 Å contact map (Duarte et al. 2012 BMC Bioinformatics 13:334)
    # NOTE: contact map, NOT a SASA-based interface definition.
    # ═══════════════════════════════════════════════════════════════
    if len(chs) >= 2:
        st.markdown("🔗 Inter-Chain Contact Map")
        st.caption(
            "Cα-Cα contact map, cutoff 8 Å (Duarte et al. 2012 "
            "*BMC Bioinformatics* 13:334). "
            "This is a contact map, NOT a SASA-based interface.")

        # Load PAE matrix for inter-chain decomposition (optional)
        pae_mat = None
        if jd.exists():
            pae_mat = pae_from_pkl(jd, best["model_id"])

        from alphafold_fusion.interface_analysis import (
            compute_contacts, interchain_pae)

        ci, cj = st.columns(2)
        with ci: ch_a = st.selectbox("Chain A", chs, 0, key="iface_a")
        with cj: ch_b = st.selectbox("Chain B", chs,
                                     min(1, len(chs) - 1), key="iface_b")
        cutoff = st.slider(
            "Contact cutoff (Å) — Duarte et al. 2012 use 8.0",
            4.0, 12.0, 8.0, 0.5, key="iface_cut")

        with st.spinner("Computing contacts..."):
            iface = compute_contacts(txt, fm, ch_a, ch_b, cutoff)

        if "error" not in iface:
            from alphafold_fusion.render import metric_card, confidence_style
            mc1, mc2, mc3 = st.columns(3)
            mc1.markdown(metric_card("Contacts",
                f"{iface['n_contacts']}", color="#0EA5E9"),
                unsafe_allow_html=True)
            _ip = iface['mean_plddt_interface'] or 0
            _p, _q, _c = confidence_style(_ip, "plddt")
            mc2.markdown(metric_card("Interface mean pLDDT",
                f"{_ip:.1f}", pct=_p, quality=_q, color=_c),
                unsafe_allow_html=True)
            mc3.markdown(metric_card("Contact residues",
                f"{iface['n_interface_a']}+{iface['n_interface_b']}",
                color="#7C3AED"),
                unsafe_allow_html=True)

            if iface["contacts"]:
                st.dataframe(pd.DataFrame(iface["contacts"][:200]),
                             use_container_width=True, height=250)

            if pae_mat is not None:
                chain_lens = []
                for ch_name in chs:
                    pc_ch = plddt_by_chain(txt, fm, ch_name)
                    if ch_name in pc_ch:
                        chain_lens.append(len(pc_ch[ch_name]))
                if chain_lens and sum(chain_lens) == pae_mat.shape[0]:
                    ic_pae = interchain_pae(pae_mat, chain_lens)
                    rows_ic = []
                    for k, v in ic_pae.items():
                        if v["is_interchain"]:
                            rows_ic.append({
                                "Pair": f"{v['chain_i']}→{v['chain_j']}",
                                "Mean PAE (Å)": v["mean_pae"],
                                "TM-kernel score": v["tm_kernel_score"],
                                "d₀ (Å)": v["d0_used"],
                            })
                    if rows_ic:
                        st.markdown("**Inter-chain PAE decomposition** "
                                    "(TM-kernel score = mean TM kernel over "
                                    "block; NOT AlphaFold ipTM):")
                        st.dataframe(pd.DataFrame(rows_ic),
                                     use_container_width=True)

            ss[f"_interface_{seq_name}"] = iface
        else:
            st.warning(iface["error"])

    # ═══════════════════════════════════════════════════════════════
    # 3. ENSEMBLE VARIANCE
    # Kabsch (1976) Acta Cryst A 32:922-923
    # Wallner (2023) Bioinformatics 39:btad573
    # ═══════════════════════════════════════════════════════════════
    if len(models) >= 2:
        st.markdown("### 🎯 Multi-Model Ensemble Variance")
        st.caption("Kabsch (1976) *Acta Cryst A* 32:922 | "
                   "Wallner (2023) *Bioinformatics* 39:btad573")

        from alphafold_fusion.ensemble_variance import compute_ensemble_rmsf

        with st.spinner("Superposing models and computing RMSF..."):
            ens = compute_ensemble_rmsf(models, sel_chain)

        if "error" not in ens:
            from alphafold_fusion.render import metric_card
            ec1, ec2, ec3, ec4 = st.columns(4)
            ec1.markdown(metric_card("Mean RMSF",
                f"{ens['mean_rmsf']:.2f}", " Å", color="#7C3AED"),
                unsafe_allow_html=True)
            ec2.markdown(metric_card("Median RMSF",
                f"{ens['median_rmsf']:.2f}", " Å", color="#7C3AED"),
                unsafe_allow_html=True)
            ec3.markdown(metric_card("Std RMSF",
                f"{ens['std_rmsf']:.2f}", " Å", color="#8B5CF6"),
                unsafe_allow_html=True)
            ec4.markdown(metric_card("Models Used",
                f"{ens['n_models']}", color="#0EA5E9"),
                unsafe_allow_html=True)

            rmsf_vals = ens["rmsf"]
            fig_rmsf = go.Figure()
            fig_rmsf.add_trace(go.Scatter(
                x=list(range(1, len(rmsf_vals) + 1)), y=rmsf_vals,
                mode="lines", name="RMSF",
                line=dict(color="#2d6a4f")))
            fig_rmsf.update_layout(
                title="Per-Residue RMSF Across Models",
                xaxis_title="Residue", yaxis_title="RMSF (Å)",
                margin=dict(l=10, r=10, t=40, b=10))
            st.plotly_chart(fig_rmsf, use_container_width=True)

            if ens.get("pairwise_rmsd"):
                st.markdown("**Pairwise RMSD:**")
                st.dataframe(pd.DataFrame(ens["pairwise_rmsd"]),
                             use_container_width=True, height=150)

            if plddt_vals and len(rmsf_vals) == len(plddt_vals):
                fig_ov = go.Figure()
                fig_ov.add_trace(go.Scatter(
                    x=plddt_vals, y=rmsf_vals,
                    mode="markers",
                    marker=dict(size=3, opacity=0.4),
                    name="Residues"))
                fig_ov.update_layout(
                    title="pLDDT vs RMSF (complementary metrics, "
                          "Wallner 2023)",
                    xaxis_title="pLDDT", yaxis_title="RMSF (Å)",
                    margin=dict(l=10, r=10, t=40, b=10))
                st.plotly_chart(fig_ov, use_container_width=True)

            ss[f"_ensemble_{seq_name}"] = ens
        else:
            st.info(ens["error"])

    # ═══════════════════════════════════════════════════════════════
    # 4. EVOLUTIONARY CONSERVATION (Henikoff-weighted)
    # Shannon (1948) Bell Syst Tech J 27:379
    # Valdar (2002) Proteins 48:227
    # Sequence weighting: Henikoff & Henikoff (1994) J Mol Biol 243:574
    # ═══════════════════════════════════════════════════════════════
    st.markdown("### 🧬 Evolutionary Conservation")
    st.caption("Shannon (1948) *Bell Syst Tech J* 27:379 | "
               "Valdar (2002) *Proteins* 48:227 | "
               "Weighting: Henikoff & Henikoff (1994) *J Mol Biol* 243:574")

    aln_data = (ss.get("aln_import") or {}).get(f"{seq_name} (AUTO)")
    if aln_data is None and jd.exists():
        with st.spinner("Scanning for MSA..."):
            aln_data = build_a3m_data(seq_name, jd)
        if aln_data and aln_data.get("hits"):
            ss.setdefault("aln_import", {})[aln_data["name"]] = aln_data

    if aln_data and aln_data.get("hits") and aln_data.get("qseq"):
        from alphafold_fusion.conservation import conservation_profile

        aligned_seqs = [h.get("aln", "") for h in aln_data["hits"]
                        if h.get("aln")]
        if aligned_seqs:
            with st.spinner("Computing conservation..."):
                cons = conservation_profile(aln_data["qseq"], aligned_seqs)

            from alphafold_fusion.render import metric_card, confidence_style
            cc1, cc2, cc3 = st.columns(3)
            cc1.markdown(metric_card("MSA Depth",
                f"{cons['n_sequences']}", color="#0EA5E9"),
                unsafe_allow_html=True)
            cc2.markdown(metric_card("Effective Depth (Neff)",
                f"{cons.get('n_effective', 0):.1f}", color="#06B6D4"),
                unsafe_allow_html=True)
            _p, _q, _c = confidence_style(cons["mean_conservation"], "fraction")
            cc3.markdown(metric_card("Mean Conservation",
                f"{cons['mean_conservation']:.3f}", pct=_p, color="#7C3AED"),
                unsafe_allow_html=True)

            fig_cons = go.Figure()
            fig_cons.add_trace(go.Scatter(
                x=list(range(1, cons["n_positions"] + 1)),
                y=cons["conservation"],
                mode="lines", name="Conservation",
                line=dict(color="#6a0dad")))
            fig_cons.update_layout(
                title=f"Per-Residue Conservation "
                      f"(MSA depth = {cons['n_sequences']}, "
                      f"Neff = {cons.get('n_effective', 0):.1f})",
                xaxis_title="Residue",
                yaxis_title="Conservation C(i) = 1 - H(i)/H_max",
                yaxis_range=[0, 1.05],
                margin=dict(l=10, r=10, t=40, b=10))
            st.plotly_chart(fig_cons, use_container_width=True)

            ss[f"_conservation_{seq_name}"] = cons
    else:
        st.info("No MSA available. Use MMseqs2 MSA mode (not single_sequence) "
                "to enable conservation analysis.")

    # ═══════════════════════════════════════════════════════════════
    # REPORT EXPORT
    # FAIR: Wilkinson et al. (2016) Scientific Data 3:160018
    # ═══════════════════════════════════════════════════════════════
    st.markdown("### Export Report")
    st.caption("FAIR principles: Wilkinson et al. (2016) *Sci Data* 3:160018")

    if st.button("Generate JSON + TSV Report", key="gen_report",
                 use_container_width=True):
        from alphafold_fusion.report import build_report, to_tsv

        fasta_path = ss.get("fasta_paths", {}).get(seq_name, "")
        seq_str = ""
        if fasta_path:
            try:
                lines = open(fasta_path).readlines()
                seq_str = "".join(
                    l.strip() for l in lines if not l.startswith(">"))
            except Exception:
                pass

        report = build_report(
            sequence_name=seq_name, sequence=seq_str,
            model_metrics=[{
                "model_id": m["model_id"], "rank": m["rank"],
                "avg_plddt": m["avg_plddt"], "ptm": m["ptm"],
                "iptm": m["iptm"], "fmt": m["fmt"],
            } for m in models],
            plddt_profile=ss.get(f"_plddt_profile_{seq_name}"),
            disorder=ss.get(f"_disorder_{seq_name}"),
            domains=ss.get(f"_domains_{seq_name}"),
            interface=ss.get(f"_interface_{seq_name}"),
            conservation=ss.get(f"_conservation_{seq_name}"),
            ensemble=ss.get(f"_ensemble_{seq_name}"),
        )

        report_json = json.dumps(report, indent=2, default=str)
        report_tsv = to_tsv(report)

        c_j, c_t = st.columns(2)
        with c_j:
            st.download_button("JSON Report", report_json,
                               f"{seq_name}_report.json",
                               "application/json",
                               use_container_width=True)
        with c_t:
            st.download_button("TSV (per-residue)", report_tsv,
                               f"{seq_name}_per_residue.tsv",
                               "text/tab-separated-values",
                               use_container_width=True)
        st.success("Report generated.")

In [ ]:
%%writefile alphafold_fusion/pages/validation.py
"""Validation page — compare AFF disorder (pLDDT<50) with DisProt.

DisProt: Piovesan et al. (2022) NAR 50:D471.
pLDDT<50 baseline: Akdel et al. (2022) NSMB 29:1056.
"""

from __future__ import annotations
import numpy as np
import plotly.graph_objects as go
import streamlit as st
from alphafold_fusion.api import guess_acc
from alphafold_fusion.plddt import plddt_by_chain
from alphafold_fusion.structure import polymer_chains
from alphafold_fusion.disprot import (
    fetch_disprot, disprot_regions, compare_plddt_vs_disprot)


def render() -> None:
    ss = st.session_state
    st.markdown('<div class="sub-header">DisProt Validation</div>',
                unsafe_allow_html=True)
    st.caption("Compare AFF disorder (pLDDT < 50 = AF2 very low confidence "
               "band; disorder principle Akdel et al. 2022 *NSMB* 29:1056) "
               "against experimental DisProt annotations "
               "(Piovesan et al. 2022 *NAR* 50:D471).")

    if not ss.get("results"):
        st.info("No structures loaded. Load a protein first.")
        return

    successful = {n: r for n, r in ss.results.items()
                  if r.get("status") == "success"}
    if not successful:
        st.warning("No successful predictions.")
        return

    seq_name = st.selectbox("Protein:", list(successful.keys()),
                            key="val_seq")
    result = successful[seq_name]
    models = result.get("models", [])
    if not models:
        st.warning("No models.")
        return

    best = sorted(models, key=lambda m: (
        m["rank"] if m["rank"] is not None else 9999,
        -(m["avg_plddt"] or 0)))[0]
    fm = "cif" if best["file"].lower().endswith((".cif", ".bcif")) else "pdb"
    try:
        txt = open(best["file"]).read()
    except Exception as e:
        st.error(f"Cannot read: {e}")
        return

    chs = polymer_chains(best["file"])
    sel_chain = chs[0] if chs else "A"
    pc = plddt_by_chain(txt, fm, sel_chain)
    plddt_vals = [v for arr in pc.values() for v in arr]
    if not plddt_vals:
        st.warning("No pLDDT extracted.")
        return

    afdb_used = ss.get("_afdb_used", {})
    guess = guess_acc(seq_name, afdb_used) or ""
    acc = st.text_input("UniProt accession (for DisProt lookup):",
                        value=guess, key="val_acc").strip().upper()
    thr = st.slider("pLDDT threshold", 30.0, 70.0, 50.0, 1.0,
                    key="val_thr")

    if st.button("Compare with DisProt", key="val_go",
                 use_container_width=True):
        if not acc:
            st.error("Enter a UniProt accession.")
            return
        with st.spinner(f"Querying DisProt for {acc}..."):
            entry = fetch_disprot(acc)
        if not entry:
            st.warning(
                f"'{acc}' not found in DisProt. This protein has no "
                f"experimental disorder annotation. Note: absence from "
                f"DisProt does NOT mean the protein is ordered — it means "
                f"it has not been experimentally characterised.")
            return

        regs = disprot_regions(entry)
        if not regs:
            st.warning(
                f"DisProt entry {entry.get('disprot_id', '?')} has **0 "
                f"consensus 'pure disorder' (type D) regions**. "
                f"The protein may still carry other annotations "
                f"(e.g. disorder-to-order transitions, binding regions) "
                f"that are not counted here. This is why MCC = 0 for "
                f"this entry: there is no pure-disorder ground truth to "
                f"compare against.")
        else:
            st.success(f"DisProt entry {entry.get('disprot_id', '?')} — "
                       f"{len(regs)} consensus disordered region(s), "
                       f"length {entry.get('length', '?')}.")

        comp = compare_plddt_vs_disprot(plddt_vals, regs, thr)
        if "error" in comp:
            st.error(comp["error"]); return

        from alphafold_fusion.render import metric_card, confidence_style
        c1, c2, c3, c4 = st.columns(4)
        for col, name, val, kind in [
            (c1, "MCC", comp["MCC"], "mcc"),
            (c2, "F1", comp["F1"], "score"),
            (c3, "Precision", comp["precision"], "score"),
            (c4, "Recall", comp["recall"], "score"),
        ]:
            pct, q, color = confidence_style(val, kind)
            col.markdown(
                metric_card(name, f"{val:.3f}", pct=pct,
                            quality=q, color=color),
                unsafe_allow_html=True)

        cc1, cc2 = st.columns(2)
        cc1.markdown(metric_card(
            "DisProt disordered residues",
            f"{comp['disprot_disordered']}", color="#0EA5E9"),
            unsafe_allow_html=True)
        cc2.markdown(metric_card(
            f"AFF disordered (pLDDT<{thr:.0f})",
            f"{comp['aff_disordered']}", color="#7C3AED"),
            unsafe_allow_html=True)

        # Overlay figure
        truth = comp["truth_per_residue"]
        n = comp["n_residues"]
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=list(range(1, n + 1)), y=plddt_vals,
            mode="lines", name="pLDDT",
            line=dict(color="#457b9d")))
        fig.add_hline(y=thr, line_dash="dash", line_color="red",
                      annotation_text=f"Threshold {thr:.0f}")
        # DisProt regions shaded
        in_reg = False
        start = 0
        for i in range(n + 1):
            d = truth[i] if i < n else 0
            if d and not in_reg:
                in_reg = True; start = i + 1
            elif not d and in_reg:
                in_reg = False
                fig.add_vrect(x0=start, x1=i,
                              fillcolor="rgba(230,57,70,0.18)",
                              line_width=0,
                              annotation_text="DisProt",
                              annotation_position="top left")
        fig.update_layout(
            title=f"AFF pLDDT vs DisProt disorder — {acc}",
            xaxis_title="Residue", yaxis_title="pLDDT",
            yaxis_range=[0, 100],
            margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)

        st.info(
            "Red shading = experimentally confirmed disorder (DisProt). "
            "Regions below the red line = AFF-predicted disorder. "
            "Only residues present in the structure are compared. "
            "DisProt annotates confirmed disorder only; unshaded regions "
            "are not necessarily ordered — they may be uncharacterised.")

        ss[f"_disprot_comp_{seq_name}"] = {
            k: v for k, v in comp.items()
            if k not in ("truth_per_residue", "pred_per_residue")}

In [ ]:
%%writefile alphafold_fusion/pages/settings.py
"""Settings page — system information and runtime diagnostics."""

from __future__ import annotations
import subprocess
import psutil
import streamlit as st
from alphafold_fusion import __version__


def render() -> None:
    st.markdown('<div class="sub-header">⚙️ System Info</div>',
                unsafe_allow_html=True)
    c1, c2 = st.columns(2)
    with c1:
        st.metric("CPU Cores", psutil.cpu_count())
        st.metric("RAM", f"{psutil.virtual_memory().total / 2**30:.1f} GB")
        try:
            r = subprocess.run(
                ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                capture_output=True, text=True, check=False)
            gpu_name = "None"
            if r.returncode == 0 and r.stdout.strip():
                gpu_name = r.stdout.strip().split("\n")[0]
            st.metric("GPU", gpu_name)
        except Exception:
            st.metric("GPU", "None")
    with c2:
        st.metric("Version", __version__)
        try:
            import jax
            st.caption(f"JAX: {jax.default_backend()} | {jax.devices()}")
        except Exception:
            st.caption("JAX: not available")

In [ ]:
%%writefile app.py
"""AlphaFold Fusion — Streamlit entry point."""

from __future__ import annotations
import streamlit as st
from alphafold_fusion import __version__
from alphafold_fusion.config import CSS, RESULTS_DIR, CACHE_DIR
from alphafold_fusion.pages import PAGES, VIEW_MODES
from alphafold_fusion.pages import (
    home, predictions, results, viewer, settings, analysis, validation)

st.set_page_config(page_title="AlphaFold Fusion", page_icon="🧬",
                   layout="wide", initial_sidebar_state="expanded")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

st.markdown(CSS, unsafe_allow_html=True)
st.markdown('<div class="main-header">AlphaFold Fusion</div>',
            unsafe_allow_html=True)
st.markdown(
    f'<p style="text-align:center;color:#666;margin-bottom:1rem">'
    f'v{__version__} — Multi-protein structure prediction and analysis</p>',
    unsafe_allow_html=True)

for _k, _v in {"results": {}, "job_dirs": {}, "fasta_paths": {},
                "fasta_text": "", "aln_import": {}, "_afdb_used": {},
                "_blast_acc": {}}.items():
    st.session_state.setdefault(_k, _v)
with st.sidebar:
    # ══════ System status badge ══════
    import subprocess as _sp
    try:
        _b = _sp.run([__import__("sys").executable, "-c",
            "import jax; print(jax.default_backend())"],
            capture_output=True, text=True, timeout=15).stdout.strip()
    except Exception:
        _b = "unknown"
    _dot = "#059669" if _b == "gpu" else "#DC2626"
    _txt = "GPU active" if _b == "gpu" else f"CPU ({_b})"
    _n = len([n for n, r in st.session_state.get("results", {}).items()
              if (r or {}).get("status") == "success"])
    st.markdown(
        f'<div style="background:#F8FAFC;border:1px solid #E2E8F0;'
        f'border-radius:10px;padding:10px 12px;margin-bottom:12px">'
        f'<div style="font-size:.7rem;color:#94A3B8;font-weight:600;'
        f'text-transform:uppercase;letter-spacing:.5px">System</div>'
        f'<div style="margin-top:6px;font-size:.85rem;color:#334155">'
        f'<span style="display:inline-block;width:8px;height:8px;'
        f'border-radius:50%;background:{_dot};margin-right:6px"></span>'
        f'{_txt}</div>'
        f'<div style="font-size:.85rem;color:#334155;margin-top:4px">'
        f'<span style="display:inline-block;width:8px;height:8px;'
        f'border-radius:50%;background:#2563EB;margin-right:6px"></span>'
        f'{_n} prediction(s)</div></div>',
        unsafe_allow_html=True)
    # ═══════════════════════════════════
with st.sidebar:
    st.markdown("## Navigation")
    st.radio("Page", PAGES,
             index=PAGES.index(st.session_state.get("nav_page", PAGES[0])),
             key="nav_page")
    st.markdown("## View mode")
    st.radio("Mode", VIEW_MODES,
             index=(0 if st.session_state.get(
                 "view_mode", VIEW_MODES[0]).startswith("pLDDT") else 1),
             key="view_mode")

_DISPATCH = {
    PAGES[0]: home.render,
    PAGES[1]: predictions.render,
    PAGES[2]: results.render,
    PAGES[3]: viewer.render,
    PAGES[4]: analysis.render,
    PAGES[5]: validation.render,
    PAGES[6]: settings.render,
}
page = st.session_state.get("nav_page", PAGES[0])
_DISPATCH.get(page, home.render)()

st.markdown("---")
st.markdown(
    f'<div style="text-align:center;color:#888;padding:.5rem">'
    f'AlphaFold Fusion v{__version__}</div>',
    unsafe_allow_html=True)

## ⚙️ 2. Installation
JAX (GPU), ColabFold, and dependencies. NumPy pinned to 1.26.4.
*(~10 min on first run)*

In [ ]:
import os, sys, subprocess

def run(cmd, **kw):
    """Run subprocess without ever crashing the cell."""
    try:
        return subprocess.run(
            cmd, capture_output=True, text=True,
            check=False, timeout=600, **kw
        )
    except FileNotFoundError:
        return type("R", (), {"returncode": 127, "stdout": "", "stderr": "not found"})()
    except Exception as e:
        return type("R", (), {"returncode": 1, "stdout": "", "stderr": str(e)})()

# ── 0. GPU Detection ─────────────────────────────────────────
print("🔍 Step 0/5: GPU detection...")
HAS_GPU = False

gpu_check = run(["nvidia-smi", "--query-gpu=name,driver_version",
                  "--format=csv,noheader"])
if gpu_check.returncode == 0 and gpu_check.stdout.strip():
    HAS_GPU = True
    print(f"  🎮 GPU: {gpu_check.stdout.strip()}")
else:
    print("  ⚠️ Pas de GPU détecté !")
    print("  ╔═══════════════════════════════════════════════════╗")
    print("  ║ Runtime → Change runtime type → GPU (T4) → Save ║")
    print("  ║ Puis relancez cette cellule                      ║")
    print("  ╚═══════════════════════════════════════════════════╝")
    print("  → Continuation en mode CPU (visualisation only)...\n")

# ── 1. Cleaning ─────────────────────────────────────────────
print("🔄 Step 1/5: Cleaning...")
for pkg in ["jax", "jaxlib",
            "jax-cuda12-plugin", "jax-cuda12-pjrt",
            "jax-cuda11-plugin", "jax-cuda11-pjrt",
            "streamlit", "protobuf", "numpy",
            "importlib-metadata", "packaging"]:
    run([sys.executable, "-m", "pip", "uninstall", "-y", pkg])
print("  ✅ Cleaned")

# ── 2. JAX (GPU ou CPU) ─────────────────────────────────────
print("\n🔄 Step 2/5: Installation JAX...")
JAX_VERSION = "0.4.38"
if HAS_GPU:
    print(f"  → JAX {JAX_VERSION} + CUDA 12 (NumPy 1.26 compatible)...")
    r = run([sys.executable, "-m", "pip", "install", "-q",
             f"jax[cuda12]=={JAX_VERSION}"])
    if r.returncode != 0:
        print("  → Fallback: jax 0.4.33...")
        run([sys.executable, "-m", "pip", "install", "-q",
             "jax[cuda12]==0.4.33"])
else:
    print(f"  → JAX {JAX_VERSION} CPU only...")
    run([sys.executable, "-m", "pip", "install", "-q",
         f"jax[cpu]=={JAX_VERSION}"])


# Vérification JAX
jax_test = run([sys.executable, "-c",
                "import jax; print(jax.default_backend()); print(jax.__version__)"])
if jax_test.returncode == 0:
    backend, version = jax_test.stdout.strip().split("\n")[:2]
    icon = "✅" if (backend == "gpu") == HAS_GPU else "⚠️"
    print(f"  {icon} JAX {version} — backend: {backend}")
else:
    print(f"  ❌ JAX import failed: {jax_test.stderr.strip()[-80:]}")

# ── 3. Dependencies app ──────────────────────────────────────
print("\n🔄 Step 3/5: Dependencies app...")
app_packages = [
    "protobuf==4.25.5",
    "numpy==1.26.4",
    "importlib-metadata==6.11.0",
    "packaging==23.2",
    "streamlit==1.28.0",
    "plotly==5.17.0",
    "py3Dmol==2.1.0",
    "biopython>=1.83,<2",
    "Pillow==10.1.0",
    "psutil==5.9.8",
    "gemmi==0.6.6",
    "pandas>=2,<3",
    "tenacity==8.5.0",
]
fail_count = 0
for pkg in app_packages:
    r = run([sys.executable, "-m", "pip", "install", "-q", pkg])
    if r.returncode != 0:
        fail_count += 1
        print(f"  ⚠️ {pkg}: {r.stderr.strip()[-60:]}")

if fail_count == 0:
    print("  ✅ All installed packages")
else:
    print(f"  ⚠️ {fail_count} package(s) en erreur")

# ── 4. ColabFold ─────────────────────────────────────────────
print("\n🔄 Step 4/5: ColabFold...")

if HAS_GPU:
    print("  → ColabFold Installation (2-5 min)...")

    # 4a. ColabFold dependencies BEFORE (--no-deps later)
    cf_deps = [
        "appdirs",
        "requests",
        "tqdm",
        "absl-py",
        "dm-tree",
        "scipy",
        "matplotlib",
        "dm-haiku",
        "ml-collections",
        "immutabledict",
        "alphafold-colabfold",
    ]
    print("  → Installing ColabFold dependencies...")
    dep_fail = 0
    for dep in cf_deps:
        r = run([sys.executable, "-m", "pip", "install", "-q", dep])
        if r.returncode != 0:
            dep_fail += 1
            print(f"    ⚠️ {dep}: {r.stderr.strip()[-60:]}")
    if dep_fail == 0:
        print(f"  ✅ {len(cf_deps)} dependencies OK")
    else:
        print(f"  ⚠️ {dep_fail}/{len(cf_deps)} dep(s) failed")

    # 4b. ColabFold (--no-deps to protect JAX)
    cf = run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
              "colabfold @ https://codeload.github.com/sokrypton/ColabFold/zip/refs/heads/main"])

    # 4c. Reinstall JAX just in case
    run([sys.executable, "-m", "pip", "install", "-q",
         "jax[cuda12]==0.4.38"])

    # 4d. ColabFold import verification
    cf_check = run([sys.executable, "-c",
                    "from colabfold.batch import main; print('OK')"])
    if cf_check.returncode == 0:
        print("  ✅ ColabFold installed and importable")
    else:
        err = cf_check.stderr.strip().split("\n")[-1][:80]
        print(f"  ❌ ColabFold import error: {err}")
        # Diagnostic test
        missing = run([sys.executable, "-c",
            "import colabfold; "
            "from colabfold import download; "
            "print('download OK')"])
        if missing.returncode != 0:
            m_err = missing.stderr.strip().split("\n")[-1]
            if "No module named" in m_err:
                mod = m_err.split("'")[-2] if "'" in m_err else "?"
                print(f"    → Missing: {mod}")
                print(f"    → Fix: pip install {mod}")
                run([sys.executable, "-m", "pip", "install", "-q", mod])

    # ══════════════════════════════════════════════════════════════
    # 4e. Enforce NumPy < 2 for ColabFold/AlphaFold compatibility
    # ColabFold's bundled AlphaFold is not NumPy-2 compatible.
    # We pin NumPy 1.26.x explicitly instead of patching source code.
    # ══════════════════════════════════════════════════════════════
    print("  → Enforcing NumPy 1.26.x for ColabFold compatibility...")
    run([sys.executable, "-m", "pip", "install", "-q",
         "numpy==1.26.4"])
    np_check = run([sys.executable, "-c",
                    "import numpy; print(numpy.__version__)"])
    np_ver = np_check.stdout.strip()
    if np_ver.startswith("1.26"):
        print(f"  ✅ NumPy pinned at {np_ver}")
    else:
        print(f"  ⚠️ NumPy is {np_ver} (expected 1.26.x). "
              f"Restart runtime (Runtime > Restart session) and "
              f"re-run this cell BEFORE launching the app.")
    # ══════════════════════════════════════════════════════════════

else:
    print("  ⏭️ Skipped (pas de GPU)")
    print("  → Mode visualisation uniquement")

# ── 5. Final check ───────────────────────────────────
print("\n🔄 Step 5/5: Final check...")

checks = [
    ("Streamlit",  "import streamlit; print(streamlit.__version__)"),
    ("Plotly",     "import plotly; print(plotly.__version__)"),
    ("py3Dmol",    "import py3Dmol; print('OK')"),
    ("Gemmi",      "import gemmi; print(gemmi.__version__)"),
    ("BioPython",  "from Bio.PDB import PDBIO; print('OK')"),
    ("Psutil",     "import psutil; print('OK')"),
    ("NumPy",      "import numpy; print(numpy.__version__)"),
    ("JAX",        "import jax; print(jax.default_backend())"),
]
if HAS_GPU:
    checks.append(
        ("ColabFold", "from colabfold.batch import main; print('OK')"))

errors = []
for name, cmd in checks:
    r = run([sys.executable, "-c", cmd])
    if r.returncode == 0:
        info = r.stdout.strip().split("\n")[-1]
        print(f"  ✅ {name:12s} → {info}")
    else:
        err = r.stderr.strip().split("\n")[-1][:60]
        print(f"  ❌ {name:12s} → {err}")
        errors.append(name)

# ── Résumé ────────────────────────────────────────────────────
print("\n" + "=" * 60)
if not errors:
    if HAS_GPU:
        print("🎉 EVERYTHING IS OK — GPU + COLABFOLD READY")
    else:
        print("🎉 APP READY (viewing mode / CPU)")
        print("  For GPU folding: Runtime > Change runtime > GPU")
    print("=" * 60)
    print("\n▶️ Launch the following cell to start the app")
else:
    print(f"⚠️ Erreurs: {', '.join(errors)}")
    print("=" * 60)
    if "ColabFold" in errors:
        print("💡 ColabFold is not importing correctly.")
        print("   → Runtime > Restart the session, then relaunch")
    else:
        print("💡 Runtime > Restart the session, then relaunch this cell")

## ✅ 3. DisProt Validation
Benchmark pLDDT<50 disorder against DisProt consensus
(60 eukaryotic proteins). Reproduces AUC-ROC = 0.80.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# VALIDATION vs DisProt — disorder classifier (pLDDT < 50)
#
# Ground truth: DisProt consensus "Structural state" type "D" (pure disorder).
# Metrics: MCC + F1 (at pLDDT<50 threshold) AND AUC-ROC (threshold-free).
#
# Note: Akdel et al. (2022, NSMB 29:1056) benchmarked pLDDT against the
# IUPred2 dataset using AUC-ROC (not MCC, not DisProt). This validation is
# COMPLEMENTARY: it uses DisProt as experimental ground truth. We report AUC
# to align with Akdel's threshold-free methodology.
# ══════════════════════════════════════════════════════════════════════════════
import json, time, os, tempfile, urllib.request, random
import numpy as np
import gemmi

N_PROTEINS = 60
PLDDT_THRESHOLD = 50.0
MAX_LEN = 1400


def _get_json(url, timeout=30):
    try:
        req = urllib.request.Request(url, headers={"Accept": "application/json"})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return json.loads(r.read().decode("utf-8", "ignore"))
    except Exception:
        return None


def _get_cif_url(acc, timeout=20):
    j = _get_json(f"https://alphafold.ebi.ac.uk/api/prediction/{acc}", timeout)
    if isinstance(j, list) and j:
        return j[0].get("cifUrl")
    return None


def _plddt(acc, timeout=30):
    url = _get_cif_url(acc, timeout)
    if not url:
        return None, "no_cif_url"
    try:
        req = urllib.request.Request(url)
        with urllib.request.urlopen(req, timeout=timeout) as r:
            raw = r.read()
    except Exception as e:
        return None, f"download_{str(e)[:20]}"
    if not raw or len(raw) < 100:
        return None, "empty_download"
    tmp = tempfile.NamedTemporaryFile("wb", suffix=".cif", delete=False)
    tmp.write(raw); tmp.close()
    try:
        st = gemmi.read_structure(tmp.name)
        if len(st) == 0:
            return None, "no_model"
        vals = {}
        for ch in st[0]:
            for res in ch:
                for at in res:
                    if at.name == "CA":
                        try:
                            vals[int(res.seqid.num)] = float(at.b_iso)
                        except Exception:
                            pass
                        break
        return (vals, "ok") if vals else (None, "no_ca")
    except Exception as e:
        return None, f"parse_{str(e)[:20]}"
    finally:
        os.remove(tmp.name)


def _disorder_regs(entry):
    """Pure disorder regions: type 'D' under 'Structural state'.

    IMPORTANT: DisProt puts pure disorder under 'Structural state'
    (type 'D'), NOT under 'full' (which may report transitions 'T').
    """
    cons = entry.get("disprot_consensus") or {}
    regs = []
    for r in cons.get("Structural state", []):
        if r.get("type") == "D":
            try:
                regs.append((int(r["start"]), int(r["end"])))
            except Exception:
                pass
    return regs


def _metrics(yt, yp):
    tp = int(np.sum((yt == 1) & (yp == 1)))
    tn = int(np.sum((yt == 0) & (yp == 0)))
    fp = int(np.sum((yt == 0) & (yp == 1)))
    fn = int(np.sum((yt == 1) & (yp == 0)))
    den = np.sqrt(float((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)))
    mcc = (tp*tn - fp*fn)/den if den > 0 else 0.0
    prec = tp/(tp+fp) if (tp+fp) > 0 else 0.0
    rec = tp/(tp+fn) if (tp+fn) > 0 else 0.0
    f1 = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0
    return mcc, f1, prec, rec, tp, fp, fn, tn


def _auc_roc(y_true, scores):
    """AUC-ROC computed manually (no sklearn dependency).

    scores: higher = more likely disordered (we use -pLDDT).
    Akdel et al. (2022) use AUC-ROC as their primary disorder metric.
    """
    y = np.asarray(y_true)
    s = np.asarray(scores, dtype=float)
    n_pos = int(y.sum())
    n_neg = int(len(y) - n_pos)
    if n_pos == 0 or n_neg == 0:
        return None
    # Rank-based AUC (Mann-Whitney U statistic)
    order = np.argsort(s)
    ranks = np.empty(len(s), dtype=float)
    ranks[order] = np.arange(1, len(s) + 1)
    # Handle ties: average ranks
    _, inv, counts = np.unique(s, return_inverse=True, return_counts=True)
    cum = np.cumsum(counts)
    start = cum - counts
    avg_rank = (start + cum + 1) / 2.0
    ranks = avg_rank[inv]
    sum_pos = ranks[y == 1].sum()
    auc = (sum_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    return float(auc)


print("Downloading DisProt...")
disprot = _get_json("https://disprot.org/api/search?release=current"
                    "&show_ambiguous=false&format=json")
if not disprot or "data" not in disprot:
    print("DisProt failed.")
else:
    entries = disprot["data"]
    print(f"  {len(entries)} entries")

    usable = []
    for e in entries:
        acc, length = e.get("acc"), e.get("length")
        regs = _disorder_regs(e)
        tax = e.get("taxonomy") or []
        is_euk = len(tax) > 0 and tax[0] == "Eukaryota"
        if acc and length and regs and length <= MAX_LEN and is_euk:
            usable.append((e, acc, length, regs))

    random.seed(42)
    random.shuffle(usable)
    print(f"  Eukaryotic usable: {len(usable)}")

    all_true, all_pred, all_plddt = [], [], []
    n_done = 0
    reasons = {}

    for e, acc, length, regs in usable[:N_PROTEINS * 6]:
        if n_done >= N_PROTEINS:
            break
        plddt, why = _plddt(acc)
        if not plddt:
            reasons[why] = reasons.get(why, 0) + 1
            continue

        truth = np.zeros(length, dtype=int)
        for s, en in regs:
            truth[max(0, s-1):min(length, en)] = 1
        pred = np.zeros(length, dtype=int)
        plddt_arr = np.full(length, np.nan)
        for pos, v in plddt.items():
            if 1 <= pos <= length:
                pred[pos-1] = 1 if v < PLDDT_THRESHOLD else 0
                plddt_arr[pos-1] = v
        mask = np.array([(i+1) in plddt for i in range(length)])
        if mask.sum() < 10:
            reasons["low_cover"] = reasons.get("low_cover", 0) + 1
            continue

        all_true.extend(truth[mask].tolist())
        all_pred.extend(pred[mask].tolist())
        all_plddt.extend(plddt_arr[mask].tolist())
        n_done += 1
        if n_done % 10 == 0:
            print(f"  ... {n_done} done")
        time.sleep(0.1)

    print(f"\n  {n_done} evaluated | skip reasons: {reasons}")

    if all_true:
        yt = np.array(all_true)
        yp = np.array(all_pred)
        plddt_scores = np.array(all_plddt)
        mcc, f1, prec, rec, tp, fp, fn, tn = _metrics(yt, yp)
        # AUC uses -pLDDT (low pLDDT => high disorder score)
        auc = _auc_roc(yt, -plddt_scores)

        print("\n" + "="*55)
        print("  DisProt validation (pLDDT-based disorder)")
        print("="*55)
        print(f"  Proteins   : {n_done}")
        print(f"  Residues   : {len(yt):,}")
        print(f"  Disordered : {int(yt.sum()):,} ({100*yt.mean():.1f}%)")
        print("  " + "-"*51)
        print("  Threshold-free metric (Akdel-style):")
        print(f"  AUC-ROC    : {auc:.3f}" if auc else "  AUC-ROC    : N/A")
        print("  " + "-"*51)
        print(f"  At pLDDT<{PLDDT_THRESHOLD:.0f} threshold:")
        print(f"  MCC        : {mcc:.3f}")
        print(f"  F1         : {f1:.3f}")
        print(f"  Precision  : {prec:.3f}")
        print(f"  Recall     : {rec:.3f}")
        print(f"  Confusion  : TP={tp} FP={fp} FN={fn} TN={tn}")
        print("="*55)

        with open("disprot_validation.json", "w") as f:
            json.dump({
                "n_proteins": n_done,
                "n_residues": int(len(yt)),
                "disordered_fraction": round(float(yt.mean()), 3),
                "AUC_ROC": round(float(auc), 3) if auc else None,
                "MCC": round(float(mcc), 3),
                "F1": round(float(f1), 3),
                "precision": round(float(prec), 3),
                "recall": round(float(rec), 3),
                "threshold": PLDDT_THRESHOLD,
                "ground_truth": "DisProt Structural state type D",
                "note": ("AUC aligns with Akdel et al. 2022 methodology "
                         "(threshold-free); they used IUPred2 dataset, "
                         "here DisProt is used as experimental ground truth."),
            }, f, indent=2)
        print("Saved disprot_validation.json")
    else:
        print("No residues. Skip reasons:", reasons)

## 🧪 4. Unit Tests
22 tests covering pure functions (sequence, disorder,
conservation, DisProt). Expected: 22/22 passed.

In [ ]:
%%writefile tests/test_core.py
"""Unit tests for AlphaFold Fusion core functions.

Run with:  pytest tests/ -v
Covers pure functions (no network, no Streamlit).
"""

import sys
sys.path.insert(0, "/content")

# Compatibility shim for NumPy legacy aliases (scipy via xarray).
import numpy as _np
for _a, _t in [("long", int), ("uint", int), ("ulong", int),
               ("longlong", int), ("ulonglong", int), ("unicode", str)]:
    if not hasattr(_np, _a):
        setattr(_np, _a, _t)

import numpy as np
import pytest


def test_clean_sequence():
    from alphafold_fusion.sequence import clean_sequence
    assert clean_sequence("acdef") == "ACDEF"
    assert clean_sequence("ACD-EF*1") == "ACDEF"
    assert clean_sequence("ACD:EFG") == "ACD:EFG"
    assert clean_sequence("") == ""


def test_is_complex():
    from alphafold_fusion.sequence import is_complex
    assert is_complex("ACDE:FGHI") is True
    assert is_complex("ACDEFG") is False


def test_total_length():
    from alphafold_fusion.sequence import total_length
    assert total_length("ACDE:FGH") == 7
    assert total_length("ACDEFG") == 6


def test_parse_fasta_single():
    from alphafold_fusion.sequence import parse_fasta
    result = parse_fasta(">prot1\nACDEF\nGHIKL")
    assert len(result) == 1
    assert result[0][0] == "prot1"
    assert result[0][1] == "ACDEFGHIKL"


def test_parse_fasta_multiple():
    from alphafold_fusion.sequence import parse_fasta
    result = parse_fasta(">p1\nACDE\n>p2\nFGHI")
    assert len(result) == 2
    assert result[1][1] == "FGHI"


def test_safe_basename():
    from alphafold_fusion.sequence import safe_basename
    name = safe_basename("my/prot name!", "ACDEF")
    assert "/" not in name and " " not in name and len(name) > 0


def test_is_uniprot_acc():
    from alphafold_fusion.api import is_uniprot_acc
    assert is_uniprot_acc("Q9UKV8") is True
    assert is_uniprot_acc("P04637") is True
    assert is_uniprot_acc("AGO2") is False
    assert is_uniprot_acc("") is False


def test_extract_uniprot_acc():
    from alphafold_fusion.api import extract_uniprot_acc
    assert extract_uniprot_acc("sp|Q9UKV8|AGO2_HUMAN") == "Q9UKV8"
    assert extract_uniprot_acc("tr|A5D9G9|A5D9G9_BOVIN") == "A5D9G9"
    assert extract_uniprot_acc("just a name") is None
    assert extract_uniprot_acc("UniRef90_Q9UKV8") == "Q9UKV8"


def test_identify_idrs_empty():
    from alphafold_fusion.disorder import identify_idrs
    result = identify_idrs([])
    assert result["n_residues"] == 0
    assert result["regions"] == []


def test_identify_idrs_all_ordered():
    from alphafold_fusion.disorder import identify_idrs
    result = identify_idrs([90.0] * 50)
    assert result["fraction_disordered"] == 0.0
    assert len(result["regions"]) == 0


def test_identify_idrs_all_disordered():
    from alphafold_fusion.disorder import identify_idrs
    result = identify_idrs([30.0] * 50)
    assert result["fraction_disordered"] == 1.0
    assert len(result["regions"]) == 1
    assert result["regions"][0]["start"] == 1
    assert result["regions"][0]["end"] == 50


def test_identify_idrs_min_length():
    from alphafold_fusion.disorder import identify_idrs
    vals = [90.0] * 20 + [30.0] * 3 + [90.0] * 20
    result = identify_idrs(vals, min_length=5)
    assert len(result["regions"]) == 0


def test_identify_idrs_region_detected():
    from alphafold_fusion.disorder import identify_idrs
    vals = [90.0] * 20 + [30.0] * 10 + [90.0] * 20
    result = identify_idrs(vals, min_length=5)
    assert len(result["regions"]) == 1
    assert result["regions"][0]["start"] == 21
    assert result["regions"][0]["end"] == 30
    assert result["regions"][0]["length"] == 10


def test_conservation_identical_column():
    from alphafold_fusion.conservation import conservation_profile
    result = conservation_profile("ACDEF", ["ACDEF"] * 10)
    assert result["n_positions"] == 5
    var = conservation_profile("ACDEF", ["KLMNP", "QRSTV", "WYACD"])
    assert result["mean_conservation"] > var["mean_conservation"]


def test_conservation_variable_column():
    from alphafold_fusion.conservation import conservation_profile
    result = conservation_profile("ACDEF", ["KLMNP", "QRSTV", "WYACD"])
    assert result["n_positions"] == 5
    assert 0.0 <= result["mean_conservation"] <= 1.0


def test_conservation_has_neff():
    from alphafold_fusion.conservation import conservation_profile
    result = conservation_profile("ACDEF", ["ACDEF", "ACDEG"])
    assert "n_effective" in result
    assert result["n_effective"] > 0


def test_disprot_regions_parsing():
    from alphafold_fusion.disprot import disprot_regions
    entry = {"disprot_consensus": {"Structural state": [
        {"start": 10, "end": 20, "type": "D"},
        {"start": 30, "end": 40, "type": "S"},
        {"start": 50, "end": 60, "type": "D"},
    ]}}
    regs = disprot_regions(entry)
    assert (10, 20) in regs
    assert (50, 60) in regs
    assert len(regs) == 2


def test_compare_perfect_match():
    from alphafold_fusion.disprot import compare_plddt_vs_disprot
    plddt = [90.0] * 10 + [30.0] * 10 + [90.0] * 10
    comp = compare_plddt_vs_disprot(plddt, [(11, 20)], threshold=50.0)
    assert comp["MCC"] > 0.9
    assert comp["TP"] == 10
    assert comp["FP"] == 0


def test_compare_no_overlap():
    from alphafold_fusion.disprot import compare_plddt_vs_disprot
    comp = compare_plddt_vs_disprot([90.0] * 30, [(11, 20)], threshold=50.0)
    assert comp["TP"] == 0
    assert comp["FN"] == 10


def test_compare_empty_plddt():
    from alphafold_fusion.disprot import compare_plddt_vs_disprot
    comp = compare_plddt_vs_disprot([], [(1, 5)])
    assert "error" in comp


def test_detect_fmt():
    from alphafold_fusion.structure import detect_fmt
    assert detect_fmt("data_AF-P04637\n_atom_site.x") == "cif"
    assert detect_fmt("ATOM      1  N   MET") == "pdb"


def test_is_amino_acid():
    from alphafold_fusion.structure import is_amino_acid
    class FakeRes:
        def __init__(self, name): self.name = name
    assert is_amino_acid(FakeRes("ALA")) is True
    assert is_amino_acid(FakeRes("HOH")) is False


if __name__ == "__main__":
    pytest.main([__file__, "-v"])

In [ ]:
import subprocess, sys

# Fix scipy for NumPy 1.26 (avoid np.long crash via xarray→scipy)
print("Fixing scipy for NumPy 1.26...")
# Install scipy + numpy TOGETHER to keep binaries aligned
# (avoids "numpy.dtype size changed" ABI mismatch)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scipy==1.11.4", "numpy==1.26.4"],
               capture_output=True)

# Install pytest
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"],
               capture_output=True)

# Run tests
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=line",
     "-p", "no:cacheprovider"],
    capture_output=True, text=True, cwd="/content")

print(result.stdout[-3000:])
if result.stderr:
    print("STDERR:", result.stderr[-300:])

In [ ]:
import numpy as np

def _mcc_formule(yt, yp):
    tp = np.sum((yt==1)&(yp==1)); tn = np.sum((yt==0)&(yp==0))
    fp = np.sum((yt==0)&(yp==1)); fn = np.sum((yt==1)&(yp==0))
    den = np.sqrt(float((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)))
    return (tp*tn - fp*fn)/den if den>0 else 0.0

def _mcc_pearson(yt, yp):
    return np.corrcoef(yt, yp)[0, 1]

yp = (plddt_scores < 50).astype(int)
print(f"MCC formule : {_mcc_formule(yt, yp):.6f}")
print(f"MCC Pearson : {_mcc_pearson(yt, yp):.6f}")
diff = abs(_mcc_formule(yt,yp) - _mcc_pearson(yt,yp))
print(f"Différence  : {diff:.2e}")
print("✅ IDENTIQUE" if diff < 1e-4 else "❌")

## 📊 5. Sensitivity Analysis
Robustness of metrics to the minimum IDR length
(min_length ∈ {1,3,5,8,10}). AUC-ROC is invariant.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SENSITIVITY ANALYSIS — min_length ∈ {1, 3, 5, 8, 10}
#
# Goal: show that our conclusions are robust to the heuristic minimum
# IDR length. min_length filters short predicted-disorder runs: a
# pLDDT<50 residue is kept as "disordered" only if it belongs to a
# consecutive segment of length >= min_length.
#
# KEY POINT:
#   - AUC-ROC is per-residue on continuous pLDDT  => INDEPENDENT of min_length
#   - MCC/F1 use the (length-filtered) binary prediction => may vary slightly
#
# Requires the validation cell (60 proteins) to have run first, so that
# `usable`, `_plddt`, `_disorder_regs`, `_metrics`, `_auc_roc` exist.
# ══════════════════════════════════════════════════════════════════════════════
import json, time
import numpy as np

MIN_LENGTHS = [1, 3, 5, 8, 10]
PLDDT_THRESHOLD = 50.0
N_PROTEINS = 60
MAX_LEN = 1400


def _filter_min_length(pred_binary, min_length):
    """Keep only disordered runs of length >= min_length (else set to 0).

    Reproduces identify_idrs region filtering on a per-residue binary
    prediction vector (operating on covered residues, in order).
    """
    p = np.asarray(pred_binary, dtype=int).copy()
    n = len(p)
    out = np.zeros(n, dtype=int)
    start = None
    for i in range(n + 1):
        if i < n and p[i] == 1:
            if start is None:
                start = i
        else:
            if start is not None:
                if (i - start) >= min_length:
                    out[start:i] = 1
                start = None
    return out


# ── Collect per-protein (truth, plddt) on covered residues only ──────────────
try:
    _ = usable          # noqa: F821  (from validation cell)
    _ = _plddt          # noqa: F821
    _ = _disorder_regs  # noqa: F821
    _ = _metrics        # noqa: F821
    _ = _auc_roc        # noqa: F821
    _have_deps = True
except NameError:
    _have_deps = False
    print("❌ Run the DisProt validation cell first "
          "(need usable, _plddt, _disorder_regs, _metrics, _auc_roc).")

if _have_deps:
    print("Collecting per-protein data (reusing AFDB, ~same set as validation)...")
    per_protein = []   # list of (truth_cov, plddt_cov) arrays
    n_done = 0

    for e, acc, length, regs in usable[:N_PROTEINS * 6]:
        if n_done >= N_PROTEINS:
            break
        plddt, why = _plddt(acc)
        if not plddt:
            continue

        truth = np.zeros(length, dtype=int)
        for s, en in regs:
            truth[max(0, s - 1):min(length, en)] = 1
        plddt_arr = np.full(length, np.nan)
        for pos, v in plddt.items():
            if 1 <= pos <= length:
                plddt_arr[pos - 1] = v

        mask = np.array([(i + 1) in plddt for i in range(length)])
        if mask.sum() < 10:
            continue

        # covered residues, in sequence order (matches app behaviour)
        per_protein.append((truth[mask], plddt_arr[mask]))
        n_done += 1
        if n_done % 10 == 0:
            print(f"  ... {n_done} done")
        time.sleep(0.05)

    print(f"  {n_done} proteins collected\n")

    # ── AUC-ROC (computed once — length-free) ────────────────────────────────
    all_truth = np.concatenate([t for t, _ in per_protein])
    all_plddt = np.concatenate([p for _, p in per_protein])
    auc_global = _auc_roc(all_truth, -all_plddt)  # low pLDDT => high disorder

    # ── Loop over min_length ─────────────────────────────────────────────────
    rows = []
    for ml in MIN_LENGTHS:
        yt_all, yp_all = [], []
        for truth_cov, plddt_cov in per_protein:
            pred_raw = (plddt_cov < PLDDT_THRESHOLD).astype(int)
            pred_filt = _filter_min_length(pred_raw, ml)
            yt_all.append(truth_cov)
            yp_all.append(pred_filt)
        yt = np.concatenate(yt_all)
        yp = np.concatenate(yp_all)
        mcc, f1, prec, rec, tp, fp, fn, tn = _metrics(yt, yp)
        rows.append({
            "min_length": ml,
            "AUC_ROC": round(float(auc_global), 3),  # constant
            "MCC": round(float(mcc), 3),
            "F1": round(float(f1), 3),
            "precision": round(float(prec), 3),
            "recall": round(float(rec), 3),
            "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        })

    # ── Report ───────────────────────────────────────────────────────────────
    print("=" * 66)
    print("  SENSITIVITY ANALYSIS — min_length (DisProt, n=%d proteins)" % n_done)
    print("=" * 66)
    print(f"  AUC-ROC (threshold-free, length-free) = {auc_global:.3f}  [CONSTANT]")
    print("  " + "-" * 62)
    print(f"  {'min_len':>7} | {'MCC':>6} | {'F1':>6} | {'Prec':>6} | {'Rec':>6}")
    print("  " + "-" * 62)
    for r in rows:
        print(f"  {r['min_length']:>7} | {r['MCC']:>6.3f} | {r['F1']:>6.3f} | "
              f"{r['precision']:>6.3f} | {r['recall']:>6.3f}")
    print("  " + "-" * 62)

    mccs = [r["MCC"] for r in rows]
    f1s = [r["F1"] for r in rows]
    print(f"  MCC range: {max(mccs) - min(mccs):.3f}   "
          f"F1 range: {max(f1s) - min(f1s):.3f}")
    print("=" * 66)
    verdict = ("ROBUST ✅" if (max(mccs) - min(mccs) < 0.05
                               and max(f1s) - min(f1s) < 0.05)
               else "check variation")
    print(f"  Verdict: metrics vary < 0.05 across min_length → {verdict}")
    print("=" * 66)

    with open("sensitivity_min_length.json", "w") as f:
        json.dump({
            "n_proteins": n_done,
            "threshold": PLDDT_THRESHOLD,
            "AUC_ROC_constant": round(float(auc_global), 3),
            "min_lengths": MIN_LENGTHS,
            "results": rows,
            "MCC_range": round(max(mccs) - min(mccs), 3),
            "F1_range": round(max(f1s) - min(f1s), 3),
            "note": ("AUC-ROC is per-residue on continuous pLDDT and thus "
                     "independent of min_length. MCC/F1 use the "
                     "length-filtered binary prediction. Robustness is "
                     "demonstrated by the small metric range across "
                     "min_length in {1,3,5,8,10}."),
        }, f, indent=2)
    print("Saved sensitivity_min_length.json")

## 🚀 6. Launch Application
Starts the Streamlit app and exposes a **public link** (Cloudflare). Keep this cell running while testing.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Launch Streamlit + Expose URL
#
# Generates BOTH a Colab proxy link (local, this session only) AND a
# Cloudflare tunnel link (PUBLIC, shareable with reviewers).
# Waits for DNS propagation so the public link is ready when displayed.
# ══════════════════════════════════════════════════════════════════════════════

import os
import sys
import subprocess
import time
import re
import urllib.request
from pathlib import Path

# ── Configuration ────────────────────────────────────────────────────────────
PORT = 8501
APP_FILE = "app.py"
STARTUP_WAIT = 12
TUNNEL_TIMEOUT = 90
CLOUDFLARED_BIN = "/usr/local/bin/cloudflared"
CLOUDFLARED_VERSION = "2024.8.3"
CLOUDFLARED_URL = (
    f"https://github.com/cloudflare/cloudflared/releases/download/"
    f"{CLOUDFLARED_VERSION}/cloudflared-linux-amd64"
)
THEME = {
    "primaryColor": "#4361ee",
    "backgroundColor": "#ffffff",
    "secondaryBackgroundColor": "#f8f9fa",
    "textColor": "#212529",
    "font": "sans serif",
}

# ── 1. Cleanup ───────────────────────────────────────────────────────────────
print("🧹 Stopping previous instances...")
os.system("pkill -f streamlit 2>/dev/null || true")
os.system("pkill -f cloudflared 2>/dev/null || true")
time.sleep(1)

# ── 2. Streamlit config ──────────────────────────────────────────────────────
os.makedirs(".streamlit", exist_ok=True)
Path(".streamlit/config.toml").write_text(f"""
[server]
headless = true
address = "0.0.0.0"
port = {PORT}
enableCORS = false
enableXsrfProtection = false

[runner]
magicEnabled = false

[browser]
gatherUsageStats = false

[theme]
primaryColor = "{THEME['primaryColor']}"
backgroundColor = "{THEME['backgroundColor']}"
secondaryBackgroundColor = "{THEME['secondaryBackgroundColor']}"
textColor = "{THEME['textColor']}"
font = "{THEME['font']}"
""")

# ── 3. Start Streamlit ──────────────────────────────────────────────────────
print(f"🚀 Starting Streamlit on port {PORT}...")
subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", APP_FILE,
     "--server.port", str(PORT), "--server.address", "0.0.0.0"],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)
time.sleep(STARTUP_WAIT)

# ── 4. Expose URLs (BOTH proxy AND Cloudflare) ───────────────────────────────
_BANNER = """
<div style="padding:20px;border-radius:16px;text-align:center;color:#fff;
margin-top:12px;background:{bg}">
<h3 style="margin:0 0 6px">{title}</h3>
<p style="margin:0 0 12px;opacity:.9;font-size:.9rem">{subtitle}</p>
<a href="{url}" target="_blank" style="display:inline-block;padding:12px 24px;
background:{btn};border-radius:12px;color:white;text-decoration:none;
font-weight:bold">{label}</a>
<p style="margin-top:10px;font-family:monospace;font-size:.8rem;
word-break:break-all">{url}</p></div>
"""

colab_url = None
cloudflare_url = None

# ── 4a. Colab proxy (LOCAL — this session only) ──────────────────────────────
try:
    from google.colab import output
    from IPython.display import HTML, display
    colab_url = output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
    print(f"✅ Colab proxy ready: {colab_url}")
except Exception as e:
    print(f"⚠ Colab proxy unavailable: {e}")

# ── 4b. Cloudflare tunnel (PUBLIC — shareable) — ALWAYS launch ──────────────
print("🌐 Starting Cloudflare tunnel (public, shareable link)...")
try:
    if not Path(CLOUDFLARED_BIN).exists():
        subprocess.run(
            ["wget", "-q", "-O", CLOUDFLARED_BIN, CLOUDFLARED_URL],
            check=False,
        )
        os.chmod(CLOUDFLARED_BIN, 0o755)
    proc = subprocess.Popen(
        [CLOUDFLARED_BIN, "tunnel", "--url",
         f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    t0 = time.time()
    while time.time() - t0 < TUNNEL_TIMEOUT:
        line = proc.stdout.readline()
        if not line:
            time.sleep(0.2)
            continue
        m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
        if m:
            cloudflare_url = m.group(0)
            print(f"✅ Cloudflare tunnel created: {cloudflare_url}")
            break
    if not cloudflare_url:
        print("⚠ Cloudflare tunnel timed out.")
except Exception as e:
    print(f"⚠ Cloudflare error: {e}")

# ── 4c. Wait for DNS propagation (trycloudflare needs ~20-30s) ──────────────
if cloudflare_url:
    print("⏳ Waiting for public tunnel DNS to propagate...")
    tunnel_live = False
    for _ in range(8):  # up to 80s
        time.sleep(10)
        try:
            req = urllib.request.Request(
                cloudflare_url, headers={"User-Agent": "Mozilla/5.0"})
            urllib.request.urlopen(req, timeout=10)
            tunnel_live = True
            print("✅ Public tunnel is LIVE and responding!")
            break
        except Exception:
            print("  ⏳ still propagating...")
    if not tunnel_live:
        print("  ⚠ Tunnel created but slow; it should work in a browser "
              "within a minute (refresh the page).")

# ── 5. Display BOTH links with clear labels ─────────────────────────────────
try:
    from IPython.display import HTML, display

    if cloudflare_url:
        display(HTML(_BANNER.format(
            bg="linear-gradient(135deg,#0ea5e9,#0369a1)",
            title="🌐 PUBLIC LINK (share with reviewers)",
            subtitle="Works from any browser, any computer. "
                     "Use THIS link to share.",
            url=cloudflare_url, btn="#111827",
            label="OPEN (Public / Cloudflare)")))

    if colab_url:
        display(HTML(_BANNER.format(
            bg="linear-gradient(135deg,#4361ee,#3a0ca3)",
            title="💻 LOCAL LINK (this session only)",
            subtitle="Works ONLY in YOUR current browser session. "
                     "Do NOT share this one.",
            url=colab_url, btn="#ff6b6b",
            label="OPEN (Local / Colab)")))

    if not (cloudflare_url or colab_url):
        print(f"\n💻 Local access: http://localhost:{PORT}")
except Exception:
    if cloudflare_url:
        print(f"\n🌐 PUBLIC (share this): {cloudflare_url}")
    if colab_url:
        print(f"💻 LOCAL (session only): {colab_url}")

# ── 6. Reviewer guidance + security notice ──────────────────────────────────
print("\n" + "=" * 62)
print("  📋 FOR REVIEWERS")
print("=" * 62)
if cloudflare_url:
    print("  ✅ Use the PUBLIC (Cloudflare) link above.")
    print("     If the page does not load immediately, wait ~20s")
    print("     and refresh (the tunnel is initialising).")
else:
    print("  ⚠ Cloudflare link unavailable — re-run this cell,")
    print("     or use the Local link if testing in this session.")
print("  • Keep this cell running while testing.")
print("  • Colab may disconnect after ~90 min of inactivity.")
print("=" * 62)

print("\n" + "=" * 62)
print("  ⚠️  SECURITY NOTICE")
print("=" * 62)
print("  Public URL, NO authentication. Anyone with the link can")
print("  access it while this session runs.")
print("  • Sequences are sent to third-party services (ColabFold")
print("    MMseqs2, EBI). Do NOT submit confidential data.")
print("  • Stop the app (interrupt this cell) when finished.")
print("=" * 62)

In [ ]:
# ══════════════════════════════════════════════════════════════
# DIAGNOSTIC GPU — vérifie hardware + JAX backend
# ══════════════════════════════════════════════════════════════
import subprocess, sys, os

print("="*60)
print("  🔍 DIAGNOSTIC GPU")
print("="*60)

# ── 1. GPU physique (nvidia-smi) ──
print("\n[1/4] GPU physique :")
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,driver_version",
                    "--format=csv,noheader"],
                   capture_output=True, text=True)
if r.returncode == 0 and r.stdout.strip():
    print(f"  ✅ {r.stdout.strip()}")
    HAS_GPU = True
else:
    print("  ❌ Aucun GPU détecté !")
    print("  → Runtime > Change runtime type > GPU (T4) > Save")
    HAS_GPU = False

# ── 2. Mémoire GPU détaillée ──
print("\n[2/4] Mémoire GPU :")
r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.free,memory.total",
                    "--format=csv,noheader,nounits"],
                   capture_output=True, text=True)
if r.returncode == 0 and r.stdout.strip():
    used, free, total = r.stdout.strip().split(", ")
    print(f"  Utilisée : {used} MB | Libre : {free} MB | Total : {total} MB")
    pct = 100 * int(used) / int(total)
    if pct > 50:
        print(f"  ⚠️ {pct:.0f}% occupée — réinitialisation recommandée")
    else:
        print(f"  ✅ {pct:.0f}% occupée — OK")

# ── 3. Backend JAX (le vrai test pour ColabFold) ──
print("\n[3/4] Backend JAX :")
r = subprocess.run([sys.executable, "-c",
                    "import jax; print(jax.default_backend()); "
                    "print(jax.devices())"],
                   capture_output=True, text=True, timeout=60)
if r.returncode == 0:
    lines = r.stdout.strip().split("\n")
    backend = lines[0] if lines else "?"
    devices = lines[1] if len(lines) > 1 else "?"
    if backend == "gpu":
        print(f"  ✅ JAX backend : {backend}")
        print(f"  Devices : {devices}")
    else:
        print(f"  ❌ JAX backend : {backend} (ATTENDU: gpu)")
        print("  → ColabFold tournera sur CPU = pLDDT catastrophique !")
        print("  → Réinitialise (Script 2 ci-dessous)")
else:
    print(f"  ❌ JAX import échoué : {r.stderr.strip()[-100:]}")

# ── 4. Processus qui occupent le GPU ──
print("\n[4/4] Processus GPU actifs :")
r = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory",
                    "--format=csv,noheader"],
                   capture_output=True, text=True)
if r.returncode == 0 and r.stdout.strip():
    print(f"  {r.stdout.strip()}")
else:
    print("  ✅ Aucun processus actif (GPU libre)")

print("\n" + "="*60)

In [ ]:
import shutil
from pathlib import Path
for d in ["/content/alphafold_cache", "/content/alphafold_results"]:
    p = Path(d)
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)
print("✅ Cache + résultats purgés")

# Nettoie aussi les a3m/pdb orphelins dans /tmp et /content
import glob, os
for f in glob.glob("/content/**/*.a3m", recursive=True):
    try: os.remove(f)
    except: pass
print("✅ a3m orphelins supprimés")

In [ ]:
import os, gc
os.system("pkill -f colabfold 2>/dev/null || true")
gc.collect()
try:
    import jax; jax.clear_caches()
except: pass
print("✅ GPU reset")